# Power BI Performance Analyzer Diagnostics

Turn a Power BI Performance Analyzer JSON export into a prioritized, readable root-cause report and remediation plan, backed by a lossless event model, data-quality rules, and reproducible provenance.

## How to use

1. In Power BI Desktop or the service, open **Performance Analyzer**, start recording, and reproduce the slow interaction.
2. Prefer a second **Refresh visuals** capture after the model/report is warm; cold-start allocation can inflate timings. Record the scenario, cache state, and storage mode in `CAPTURE_METADATA` (for Direct Lake, `"Direct Lake on SQL"` or `"Direct Lake on OneLake"` switches DirectQuery guidance to fallback guidance).
3. Set `INPUT_PATH` to any Performance Analyzer JSON file or a folder containing exports. A folder selects its newest JSON; a blank value searches beside the notebook.
4. Choose `VALIDATION_MODE` (`compatible` or `strict`) and `QUERY_TEXT_MODE` (`full`, `redact`, or `omit`) before sharing outputs.
5. Run all cells. Unsupported or malformed files stop with a schema-focused error instead of silently producing a report; structurally valid files with issues get a Pass / Warning / Fail data-quality banner.

**Command line:** cell 1 is tagged `parameters` and also reads environment overrides (`PBI_PA_INPUT_PATH`, `PBI_PA_OUTPUT_DIR`, `PBI_PA_VALIDATION_MODE`, `PBI_PA_QUERY_TEXT_MODE`, `PBI_PA_METADATA_PATH`, `PBI_PA_REGRESSION_DIR`, `PBI_PA_STORAGE_MODE`, `PBI_PA_CACHE_STATE`), for example:

```powershell
$env:PBI_PA_INPUT_PATH = "C:\captures\PowerBIPerformanceData.json"; $env:PBI_PA_QUERY_TEXT_MODE = "redact"
jupyter execute "Power BI Performance Analyzer Diagnostics.ipynb"
```

> Performance Analyzer reports elapsed durations on a shared UI thread. **Other** commonly represents preparation, queueing, or synchronization with sibling visuals. This notebook excludes Other from its actionable ranking and labels Other-only diagnoses as low confidence.

In [ ]:
# Input and output configuration (this cell is tagged "parameters" for papermill-style runs)
INPUT_PATH = r""  # JSON file or folder; blank = newest JSON beside the notebook
OUTPUT_DIR = "powerbi_performance_report"
USE_SAMPLE_IF_NO_FILE = False

# Millisecond thresholds; tune these for the report's audience and SLA.
REVIEW_VISUAL_MS = 1000
SLOW_VISUAL_MS = 2000
CRITICAL_VISUAL_MS = 5000
SLOW_DAX_MS = 1000
SLOW_DIRECT_QUERY_MS = 1000
DIRECT_QUERY_REVIEW_MS = 5000
SLOW_RENDER_MS = 500
MAX_VISUALS_PER_PAGE = 7
TOP_N = 20

# Validation, privacy, and provenance
VALIDATION_MODE = "compatible"      # "compatible" (tolerate schema drift, emit warnings) or "strict" (published draft-06 schema; stop on mismatch)
QUERY_TEXT_MODE = "full"            # "full", "redact" (replace literal values), or "omit" (remove DAX/source query text from every output)
INCLUDE_FULL_SOURCE_PATH = False    # False writes only the file name into reports
MAX_INPUT_MB = 200                  # reject larger inputs before parsing
CAPTURE_METADATA = {                # optional context recorded in every output; leave blank when unknown
    "report_name": "",
    "page": "",
    "scenario": "",                 # e.g. "initial page load", "refresh visuals", "slicer change"
    "storage_mode": "",             # e.g. "Import", "DirectQuery", "Composite", "Direct Lake on SQL", "Direct Lake on OneLake" (drives Direct Lake fallback guidance)
    "power_bi_version": "",
    "cache_state": "",              # e.g. "cold", "warm" - never inferred from file names
    "test_machine": "",
    "analyst_notes": "",
}
CAPTURE_METADATA_PATH = ""          # optional JSON file whose keys override CAPTURE_METADATA

# Trace interpretation
CLOCK_TOLERANCE_MS = 5              # presentation tolerance for CHILD_OUTSIDE_PARENT; raw timestamps are never changed
INTERACTION_WINDOW_MS = 120_000     # max gap between a User Action and a root visual update for derived interaction attribution
VISUAL_STATUS_CODE_MAP = {}         # e.g. {"3": "abandoned"}; numeric status codes are undocumented, so map them only after confirming for your Power BI version
HIGH_RESULT_ROWS = 100_000          # DAX RowCount / DirectQuery RowsRead review lead (diagnostic lead, not causality)
SUMMARY_TOP_N = 5                   # top visuals / DAX queries in the Run Summary
MAX_ISSUES_PER_RULE = 200           # per-rule cap for detailed quality-issue rows; totals are always counted
TIMELINE_MAX_EVENTS = 4000          # longest events kept in the HTML timeline when a trace is larger
REGRESSION_CAPTURES_DIR = ""        # optional folder of real captures for the regression matrix cell

# Command-line overrides, e.g.:
#   $env:PBI_PA_INPUT_PATH="C:\captures\run.json"; $env:PBI_PA_QUERY_TEXT_MODE="redact"
#   jupyter execute "Power BI Performance Analyzer Diagnostics.ipynb"
import os as _os

INPUT_PATH = _os.environ.get("PBI_PA_INPUT_PATH", INPUT_PATH)
OUTPUT_DIR = _os.environ.get("PBI_PA_OUTPUT_DIR", OUTPUT_DIR)
VALIDATION_MODE = _os.environ.get("PBI_PA_VALIDATION_MODE", VALIDATION_MODE).strip().lower()
QUERY_TEXT_MODE = _os.environ.get("PBI_PA_QUERY_TEXT_MODE", QUERY_TEXT_MODE).strip().lower()
CAPTURE_METADATA_PATH = _os.environ.get("PBI_PA_METADATA_PATH", CAPTURE_METADATA_PATH)
REGRESSION_CAPTURES_DIR = _os.environ.get("PBI_PA_REGRESSION_DIR", REGRESSION_CAPTURES_DIR)
CAPTURE_METADATA["storage_mode"] = _os.environ.get("PBI_PA_STORAGE_MODE", CAPTURE_METADATA["storage_mode"])
CAPTURE_METADATA["cache_state"] = _os.environ.get("PBI_PA_CACHE_STATE", CAPTURE_METADATA["cache_state"])
if VALIDATION_MODE not in {"compatible", "strict"}:
    raise ValueError(f"VALIDATION_MODE must be 'compatible' or 'strict', not {VALIDATION_MODE!r}")
if QUERY_TEXT_MODE not in {"full", "redact", "omit"}:
    raise ValueError(f"QUERY_TEXT_MODE must be 'full', 'redact', or 'omit', not {QUERY_TEXT_MODE!r}")

print(f"Configuration loaded. Input: {INPUT_PATH or '<auto-discover>'} | Validation: {VALIDATION_MODE} | Query text: {QUERY_TEXT_MODE}")

## 1. Install and import dependencies

The analyzer requires `pandas` and `numpy`. Plotly is optional for interactive charts; Matplotlib is used as a fallback. `pyarrow` is optional and adds `events.parquet` next to `events.csv`. For a missing package, uncomment and run:

```python
# %pip install pandas numpy plotly matplotlib pyarrow --quiet
```

In [ ]:
import base64
import bisect
import hashlib
import html
import importlib.util
import json
import os
import platform
import re
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
if PLOTLY_AVAILABLE:
    import plotly
    import plotly.graph_objects as go
    charting_version = plotly.__version__
else:
    import matplotlib
    import matplotlib.pyplot as plt
    charting_version = matplotlib.__version__

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
PYARROW_AVAILABLE = importlib.util.find_spec("pyarrow") is not None

versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "charting": charting_version,
    "parquet": "pyarrow" if PYARROW_AVAILABLE else "unavailable (events.csv only)",
}
print(versions)

## 2. Load, validate, inspect, and normalize Performance Analyzer JSON

The official export is an event **forest** (`version` plus `events`) linked by `id` / `parentId`. The parser follows the blueprint processing order:

1. Validate the root envelope and each event's required fields (`id`, `name`, `component`, `start`).
2. Parse timestamps as UTC without modifying them; index events by `id`; detect duplicate IDs, unresolved parents, and parent cycles.
3. Derive ancestors, depth, root, visual, semantic-query, and DAX-query ancestors; compute exact `end - start` durations only when `end` exists.
4. Extract documented metrics into typed columns while preserving every raw event and its complete `metrics` bag (`events.csv`).
5. Attribute events to visual updates and, separately, attribute root visual updates to the nearest preceding User Action as a **derived** interaction.
6. Calculate covered time with interval unions (never by summing overlapping intervals) and run the data-quality rules.

`VALIDATION_MODE="compatible"` tolerates schema drift (for example, Power BI 1.1.0 exports that omit `component` on canvas events, add `sessionId`, or use numeric visual status codes) and reports it. `VALIDATION_MODE="strict"` mirrors Microsoft's published draft-06 schema and stops on any mismatch. Records with missing identifiers, invalid timestamps, duplicate IDs, or parent cycles are quarantined from analysis but retained in `events.csv`. Flattened or transformed exports remain supported through alias matching and are labeled `NON_OFFICIAL_SHAPE`.

In [ ]:
NORMALIZED_COLUMNS = [
    "page", "visual", "visual_id", "visual_type", "start_time", "total_ms",
    "dax_ms", "direct_query_ms", "render_ms", "parameter_ms", "other_ms",
    "query_ms", "actionable_ms", "dax_query", "native_query_text",
    "direct_query_executions", "source_event_count", "raw_properties",
    "visual_event_id", "status", "total_ms_source", "dax_query_count", "direct_query_count",
    "dax_error_count", "dax_canceled_count", "has_dax_query", "has_direct_query",
    "possible_canvas_cache_hit", "is_truncated", "data_quality_flags", "data_quality_status",
    "derived_interaction_id", "interaction_label",
]

# Defaults for records that cannot carry event-level evidence (flattened exports, hand-built frames).
VISUAL_COLUMN_DEFAULTS = {
    "native_query_text": "",
    "direct_query_executions": None,
    "source_event_count": 0,
    "raw_properties": "{}",
    "visual_event_id": "",
    "status": "",
    "total_ms_source": "export field",
    "dax_query_count": 0,
    "direct_query_count": 0,
    "dax_error_count": 0,
    "dax_canceled_count": 0,
    "has_dax_query": False,
    "has_direct_query": False,
    "possible_canvas_cache_hit": False,
    "is_truncated": False,
    "data_quality_flags": "",
    "data_quality_status": "Not assessed (no event tree)",
    "derived_interaction_id": "",
    "interaction_label": "",
}


def apply_visual_defaults(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in NORMALIZED_COLUMNS:
        if column not in result.columns:
            result[column] = None
    for column, default in VISUAL_COLUMN_DEFAULTS.items():
        if column == "direct_query_executions":
            result[column] = result[column].map(lambda value: value if isinstance(value, list) else [])
            continue
        result[column] = result[column].map(
            lambda value, fallback=default: fallback if value is None or (isinstance(value, float) and not np.isfinite(value)) else value
        )
    for column in ("dax_query", "native_query_text", "data_quality_flags", "status", "visual_event_id"):
        result[column] = result[column].fillna("").astype(str)
    for column in ("has_dax_query", "has_direct_query", "possible_canvas_cache_hit", "is_truncated"):
        result[column] = result[column].astype(bool)
    for column in ("dax_query_count", "direct_query_count", "dax_error_count", "dax_canceled_count", "source_event_count"):
        result[column] = pd.to_numeric(result[column], errors="coerce").fillna(0).astype(int)
    return result[NORMALIZED_COLUMNS]

ALIASES = {
    "page": {"page", "pagename", "section", "sectionname", "reportpage"},
    "visual": {"visual", "visualname", "visualtitle", "title", "displayname"},
    "visual_id": {"visualid", "visualguid", "containerid", "objectid"},
    "visual_type": {"visualtype", "type", "visualkind"},
    "query": {"query", "daxquery", "querytext", "commandtext", "expression"},
    "native_query": {
        "nativequerytext", "nativequery", "sourcequerytext", "sourcequery",
        "sqlquery", "sqltext", "kqlquery", "kqltext",
        "directquerytext", "directquerysql",
    },
    "source_label": {"sourcelabel", "actionsource", "eventsource"},
    "total": {"duration", "durationms", "totalduration", "totaldurationms", "elapsed", "elapsedms"},
    "dax": {"dax", "daxms", "daxquery", "daxqueryms", "daxqueryduration", "daxquerydurationms"},
    "direct": {"directquery", "directqueryms", "directqueryduration", "directquerydurationms"},
    "render": {"render", "renderms", "renderduration", "renderdurationms", "visualdisplay", "visualdisplayms"},
    "parameter": {"evaluatedparameters", "evaluatedparametersms", "fieldparameters", "parameterdurationms"},
    "other": {"other", "otherms", "otherduration", "otherdurationms"},
    "start": {"start", "starttime", "timestamp", "time"},
}

EVENT_KEYS = {
    "events": {"events", "eventlist", "performanceevents"},
    "id": {"id", "eventid"},
    "parent_id": {"parentid", "parenteventid"},
    "name": {"name", "eventname"},
    "component": {"component", "category"},
    "start": {"start", "starttime", "timestamp"},
    "end": {"end", "endtime"},
    "metrics": {"metrics", "properties", "details", "timings"},
    "version": {"version", "schemaversion"},
}


def normalize_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def direct_value(mapping: Any, key_group: str) -> Any:
    if not isinstance(mapping, dict):
        return None
    aliases = EVENT_KEYS[key_group]
    return next((value for key, value in mapping.items() if normalize_key(key) in aliases), None)


def to_number(value: Any, default: float = 0.0) -> float:
    if value is None or isinstance(value, bool):
        return default
    text = str(value).strip().replace(",", "")
    match = re.fullmatch(r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*(?:ms|milliseconds?)?", text, re.IGNORECASE)
    if not match:
        return default
    try:
        number = float(match.group(1))
    except (TypeError, ValueError):
        return default
    return number if np.isfinite(number) and number >= 0 else default


def parse_timestamp(value: Any) -> datetime | None:
    if value in (None, "") or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        numeric_value = float(value)
        if not np.isfinite(numeric_value):
            return None
        seconds = numeric_value / 1000 if abs(numeric_value) >= 100_000_000_000 else numeric_value
        try:
            return datetime.fromtimestamp(seconds, tz=timezone.utc)
        except (OSError, OverflowError, ValueError):
            return None
    text = str(value).strip()
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    try:
        parsed = datetime.fromisoformat(text)
        return parsed.replace(tzinfo=parsed.tzinfo or timezone.utc)
    except ValueError:
        return None


def iter_key_values(value: Any) -> Iterable[tuple[str, Any]]:
    if isinstance(value, dict):
        for key, child in value.items():
            yield str(key), child
            yield from iter_key_values(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_key_values(child)


def find_value(value: Any, alias_group: str) -> Any:
    aliases = ALIASES[alias_group]
    for key, child in iter_key_values(value):
        if normalize_key(key) in aliases and not isinstance(child, (dict, list)):
            if child not in (None, ""):
                return child
    return None


def event_duration_ms(event: dict[str, Any]) -> float:
    start = parse_timestamp(direct_value(event, "start"))
    end = parse_timestamp(direct_value(event, "end"))
    if start and end:
        try:
            return max((end - start).total_seconds() * 1000, 0.0)
        except TypeError:
            return 0.0
    for container in (event, direct_value(event, "metrics") or {}):
        duration = find_value(container, "total")
        if duration is not None:
            return to_number(duration)
    return 0.0


def classify_event(event: dict[str, Any]) -> str | None:
    text = normalize_key(" ".join(str(direct_value(event, key) or "") for key in ("name", "component")))
    if "directquery" in text or ("direct" in text and "query" in text):
        return "direct_query_ms"
    if "evaluatedparameter" in text or "fieldparameter" in text:
        return "parameter_ms"
    if "daxquery" in text or ("dax" in text and ("query" in text or "execute" in text)):
        return "dax_ms"
    if "visualdisplay" in text or "render" in text or "drawvisual" in text:
        return "render_ms"
    if text == "other" or text.endswith("other") or "otherduration" in text:
        return "other_ms"
    return None


def is_execute_query_owner(event: dict[str, Any]) -> bool:
    text = normalize_key(
        f"{direct_value(event, 'name') or ''} {direct_value(event, 'component') or ''}"
    )
    return "executequery" in text and "directquery" not in text


def event_metrics_query_text(event: dict[str, Any]) -> str:
    metrics = direct_value(event, "metrics")
    if not isinstance(metrics, dict):
        return ""
    for key, value in iter_key_values(metrics):
        if normalize_key(key) in ALIASES["query"] and value not in (None, ""):
            return str(value)
    return ""


def event_metadata(event: dict[str, Any]) -> dict[str, Any]:
    return {
        "page": find_value(event, "page"),
        "visual": find_value(event, "visual"),
        "visual_id": find_value(event, "visual_id"),
        "visual_type": find_value(event, "visual_type"),
        "dax_query": None if is_execute_query_owner(event) else find_value(event, "query"),
    }


def direct_scalar_map(record: dict[str, Any]) -> dict[str, Any]:
    values = {normalize_key(key): value for key, value in record.items() if not isinstance(value, (dict, list))}
    timing_bags = {"metrics", "properties", "details", "timings"}
    for key, bag in record.items():
        if normalize_key(key) in timing_bags and isinstance(bag, dict):
            values.update({normalize_key(child_key): value for child_key, value in bag.items() if not isinstance(value, (dict, list))})
    return values


def first_alias(values: dict[str, Any], alias_group: str) -> Any:
    return next((values[key] for key in ALIASES[alias_group] if key in values and values[key] not in (None, "")), None)


def iter_dicts(value: Any) -> Iterable[dict[str, Any]]:
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_dicts(child)


def parse_flat_export(payload: Any) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen: set[tuple[Any, ...]] = set()
    for record in iter_dicts(payload):
        values = direct_scalar_map(record)
        visual = first_alias(values, "visual")
        visual_id = first_alias(values, "visual_id")
        duration_values = {
            "total_ms": to_number(first_alias(values, "total")),
            "dax_ms": to_number(first_alias(values, "dax")),
            "direct_query_ms": to_number(first_alias(values, "direct")),
            "render_ms": to_number(first_alias(values, "render")),
            "parameter_ms": to_number(first_alias(values, "parameter")),
            "other_ms": to_number(first_alias(values, "other")),
        }
        has_component_key = any(any(alias in values for alias in ALIASES[group]) for group in ("dax", "direct", "render", "other"))
        if not (visual or visual_id) or not any(duration_values.values()) or not has_component_key:
            continue
        query_ms = max(duration_values["dax_ms"], duration_values["direct_query_ms"])
        actionable_ms = query_ms + duration_values["render_ms"] + duration_values["parameter_ms"]
        total_ms = duration_values["total_ms"] or actionable_ms + duration_values["other_ms"]
        if duration_values["other_ms"] == 0 and total_ms > actionable_ms:
            duration_values["other_ms"] = total_ms - actionable_ms
        identity = (
            first_alias(values, "page"), visual, visual_id, first_alias(values, "start"),
            total_ms, duration_values["dax_ms"], duration_values["render_ms"],
        )
        if identity in seen:
            continue
        seen.add(identity)
        rows.append({
            "page": str(first_alias(values, "page") or "Page unavailable in export"),
            "visual": str(visual or f"Visual {visual_id}"),
            "visual_id": str(visual_id or ""),
            "visual_type": str(first_alias(values, "visual_type") or "Unknown"),
            "start_time": str(first_alias(values, "start") or ""),
            "total_ms": round(total_ms, 3),
            **{key: round(value, 3) for key, value in duration_values.items() if key != "total_ms"},
            "query_ms": round(query_ms, 3),
            "actionable_ms": round(actionable_ms, 3),
            "dax_query": str(first_alias(values, "query") or ""),
            "native_query_text": str(first_alias(values, "native_query") or ""),
            "direct_query_executions": [],
            "source_event_count": 1,
            "raw_properties": json.dumps(record, ensure_ascii=True, default=str),
            "total_ms_source": "export field" if duration_values["total_ms"] else "derived from component fields",
            "has_dax_query": duration_values["dax_ms"] > 0 or bool(first_alias(values, "query")),
            "has_direct_query": duration_values["direct_query_ms"] > 0,
            "dax_query_count": 1 if first_alias(values, "query") else 0,
        })
    return apply_visual_defaults(pd.DataFrame(rows, columns=NORMALIZED_COLUMNS))

In [ ]:
# Event catalog, versions, quality-rule catalog, and interval helpers (blueprint sections 4, 5, 10, 12, 16)
PARSER_VERSION = "2.0.0"
MAPPING_VERSION = "1.0.0"
FINDING_RULES_VERSION = "1.0.0"
RULESET_VERSIONS = {
    "parser_version": PARSER_VERSION,
    "mapping_version": MAPPING_VERSION,
    "finding_rules_version": FINDING_RULES_VERSION,
}

REPORT_CANVAS, DSE_COMPONENT, AS_COMPONENT, CHANGE_DETECTION = "Report Canvas", "DSE", "AS", "Change Detection"


def make_type_key(component: Any, name: Any) -> str:
    return f"{component} / {name}"


# (component, name) -> friendly category, whether an end timestamp is documented, documented metric keys.
KNOWN_EVENT_CATALOG: dict[tuple[str, str], dict[str, Any]] = {
    (REPORT_CANVAS, "User Action"): {"category": "User action", "has_end": False, "metrics": {"sourceLabel"}},
    (REPORT_CANVAS, "Visual Container Lifecycle"): {"category": "Visual total", "has_end": True, "metrics": {"status", "visualTitle", "visualId", "visualType"}},
    (REPORT_CANVAS, "Query"): {"category": "Canvas query", "has_end": True, "metrics": set()},
    (REPORT_CANVAS, "Query Generation"): {"category": "Query generation", "has_end": True, "metrics": set()},
    (REPORT_CANVAS, "Parse Query Result"): {"category": "Result parsing", "has_end": True, "metrics": set()},
    (REPORT_CANVAS, "Render"): {"category": "Visual display", "has_end": True, "metrics": set()},
    (REPORT_CANVAS, "Data View Transform"): {"category": "Data transform", "has_end": True, "metrics": set()},
    (REPORT_CANVAS, "Geocoding"): {"category": "Geocoding", "has_end": True, "metrics": set()},
    (DSE_COMPONENT, "Execute Semantic Query"): {"category": "Semantic query", "has_end": True, "metrics": set()},
    (DSE_COMPONENT, "Open Connection"): {"category": "Open connection", "has_end": True, "metrics": set()},
    (DSE_COMPONENT, "Execute DAX Query"): {"category": "DAX query", "has_end": True, "metrics": {"QueryText", "RowCount", "Error", "Canceled"}},
    (DSE_COMPONENT, "Metrics Truncated"): {"category": "Truncation marker", "has_end": False, "metrics": set()},
    (AS_COMPONENT, "Execute Query"): {"category": "Model query", "has_end": True, "metrics": set()},
    (AS_COMPONENT, "Serialize Rowset"): {"category": "Serialize rowset", "has_end": True, "metrics": set()},
    (AS_COMPONENT, "Get Source Connection"): {"category": "Source connection", "has_end": True, "metrics": set()},
    (AS_COMPONENT, "Execute Direct Query"): {"category": "Direct source query", "has_end": True, "metrics": {"QueryText", "ActualQueryDuration", "RowsRead", "DataReadDuration", "IsGetSourceCapabilitiesQuery"}},
    (AS_COMPONENT, "Metrics Truncated"): {"category": "Truncation marker", "has_end": False, "metrics": {"Count"}},
    (CHANGE_DETECTION, "Execute Change Detection"): {"category": "Change detection", "has_end": True, "metrics": {"changeDetectionMeasure"}},
}
KNOWN_TYPE_KEYS = {make_type_key(*key): spec for key, spec in KNOWN_EVENT_CATALOG.items()}
VISUAL_KEY = make_type_key(REPORT_CANVAS, "Visual Container Lifecycle")
USER_ACTION_KEY = make_type_key(REPORT_CANVAS, "User Action")
CANVAS_QUERY_KEY = make_type_key(REPORT_CANVAS, "Query")
SEMANTIC_QUERY_KEY = make_type_key(DSE_COMPONENT, "Execute Semantic Query")
DAX_KEY = make_type_key(DSE_COMPONENT, "Execute DAX Query")
DIRECT_QUERY_KEY = make_type_key(AS_COMPONENT, "Execute Direct Query")

_names_to_components: dict[str, set[str]] = defaultdict(set)
for _component, _name in KNOWN_EVENT_CATALOG:
    _names_to_components[_name].add(_component)
UNIQUE_NAME_COMPONENT = {name: next(iter(components)) for name, components in _names_to_components.items() if len(components) == 1}
# Undocumented canvas event names observed in 1.1.0 exports; used only to infer an omitted component.
OBSERVED_CANVAS_NAMES = {"Query Pending", "Query Executing", "Visual Update", "Visual Update Async", "Visual Container Resource Load"}

# Friendly-category rules (blueprint 11.2). Covered time is always an interval union, never a sum.
ACTIONABLE_CATEGORY_BY_KEY = {DAX_KEY: "dax_ms", DIRECT_QUERY_KEY: "direct_query_ms", make_type_key(REPORT_CANVAS, "Render"): "render_ms"}
EXPLAINED_CANVAS_CATEGORIES = {"Canvas query", "Query generation", "Result parsing", "Visual display", "Data transform", "Geocoding", "Evaluated parameters"}
VALID_VISUAL_STATUSES = {"started", "finished", "abandoned"}

SEVERITY_ORDER = {"Info": 0, "Warning": 1, "Error": 2}
QUALITY_RULES: dict[str, dict[str, str]] = {
    "ROOT_NOT_OBJECT": {"severity": "Error", "class": "Invalid input", "condition": "Root JSON value is not an object."},
    "ROOT_REQUIRED_FIELD_MISSING": {"severity": "Error", "class": "Invalid input", "condition": "`version` or `events` missing (a recognized alias downgrades this to a warning in compatible mode)."},
    "EVENTS_EMPTY": {"severity": "Warning", "class": "Incomplete trace", "condition": "The events array is empty."},
    "EVENT_NOT_OBJECT": {"severity": "Error", "class": "Invalid input", "condition": "An events[] item is not an object; it is quarantined."},
    "EVENT_REQUIRED_FIELD_MISSING": {"severity": "Error", "class": "Invalid input", "condition": "`id`, `name`, `component`, or `start` missing. Missing id/name/start quarantines the event; an omitted component is inferred and reported as a warning in compatible mode."},
    "EVENT_ID_DUPLICATE": {"severity": "Error", "class": "Invalid input", "condition": "Duplicate event ID; later occurrences are retained in events.csv but quarantined from analysis."},
    "PARENT_NOT_FOUND": {"severity": "Warning", "class": "Incomplete trace", "condition": "`parentId` does not resolve; the subtree is not attributed to a visual."},
    "PARENT_CYCLE": {"severity": "Error", "class": "Invalid input", "condition": "Cycle detected in the parent graph; members and descendants are quarantined."},
    "TIMESTAMP_INVALID": {"severity": "Error", "class": "Invalid input", "condition": "Timestamp cannot be parsed; the event is quarantined."},
    "TIMESTAMP_NONSTANDARD": {"severity": "Warning", "class": "Schema drift", "condition": "Timestamp was parsed but is not an RFC 3339 date-time string."},
    "NEGATIVE_DURATION": {"severity": "Error", "class": "Invalid input", "condition": "`end < start`; raw values are retained and excluded from covered-time calculations."},
    "EVENT_END_MISSING": {"severity": "Warning", "class": "Incomplete trace", "condition": "A documented duration event has no `end`; its duration is unknown (not zero)."},
    "CHILD_OUTSIDE_PARENT": {"severity": "Warning", "class": "Clock alignment", "condition": "Child interval falls outside its parent interval beyond CLOCK_TOLERANCE_MS; timestamps are not modified."},
    "TRACE_TRUNCATED": {"severity": "Warning", "class": "Incomplete trace", "condition": "A `Metrics Truncated` event was observed; event detail is incomplete."},
    "VISUAL_ABANDONED": {"severity": "Warning", "class": "Trace content", "condition": "Visual lifecycle status is `abandoned`."},
    "VISUAL_STATUS_NONSTANDARD": {"severity": "Info", "class": "Schema drift", "condition": "Visual lifecycle status is not started/finished/abandoned and has no VISUAL_STATUS_CODE_MAP entry."},
    "DAX_ERROR": {"severity": "Error", "class": "Trace content", "condition": "DAX metric `Error` is true."},
    "DAX_CANCELED": {"severity": "Warning", "class": "Trace content", "condition": "DAX metric `Canceled` is true."},
    "UNKNOWN_EVENT_TYPE": {"severity": "Info", "class": "Schema drift", "condition": "New `(component, name)` pair observed; retained in events.csv."},
    "UNKNOWN_METRIC": {"severity": "Info", "class": "Schema drift", "condition": "New metric key observed on a documented event type; retained in metrics_json."},
    "STRICT_SCHEMA_MISMATCH": {"severity": "Warning", "class": "Schema drift", "condition": "Input differs from the published draft-06 schema (Error and stop in strict mode)."},
    "QUERY_TEXT_UNAVAILABLE": {"severity": "Info", "class": "Trace content", "condition": "A DAX or DirectQuery event exists but its query text is absent."},
    "NON_OFFICIAL_SHAPE": {"severity": "Warning", "class": "Schema drift", "condition": "Input was parsed as flattened records; event-level validation is unavailable."},
}
QUALITY_ISSUE_COLUMNS = ["rule_id", "severity", "rule_class", "scope_type", "scope_id", "message", "evidence_json"]
RFC3339_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}[Tt]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:[Zz]|[+-]\d{2}:\d{2})$")


class StrictSchemaError(ValueError):
    """Raised in strict validation mode when the input does not match the published schema."""


class QualityIssueCollector:
    def __init__(self, max_per_rule: int):
        self.max_per_rule = max(1, int(max_per_rule))
        self.rows: list[dict[str, Any]] = []
        self.counts: Counter = Counter()
        self.severity_by_rule: dict[str, str] = {}

    def add(self, rule_id: str, severity: str | None, scope_type: str, scope_id: Any, message: str, evidence: Any = None) -> None:
        severity = severity or QUALITY_RULES[rule_id]["severity"]
        self.counts[rule_id] += 1
        previous = self.severity_by_rule.get(rule_id)
        if previous is None or SEVERITY_ORDER[severity] > SEVERITY_ORDER[previous]:
            self.severity_by_rule[rule_id] = severity
        if self.counts[rule_id] <= self.max_per_rule:
            self.rows.append({
                "rule_id": rule_id, "severity": severity,
                "rule_class": QUALITY_RULES[rule_id]["class"],
                "scope_type": scope_type, "scope_id": "" if scope_id is None else str(scope_id),
                "message": message,
                "evidence_json": json.dumps(evidence or {}, sort_keys=True, default=str, ensure_ascii=False),
            })

    def frame(self) -> pd.DataFrame:
        rows = list(self.rows)
        for rule_id, count in sorted(self.counts.items()):
            if count > self.max_per_rule:
                rows.append({
                    "rule_id": rule_id, "severity": self.severity_by_rule[rule_id],
                    "rule_class": QUALITY_RULES[rule_id]["class"], "scope_type": "Run", "scope_id": "",
                    "message": f"{count - self.max_per_rule:,} additional {rule_id} occurrence(s) not listed (MAX_ISSUES_PER_RULE={self.max_per_rule}).",
                    "evidence_json": json.dumps({"total_occurrences": count}, sort_keys=True),
                })
        return pd.DataFrame(rows, columns=QUALITY_ISSUE_COLUMNS)

    def status(self) -> str:
        worst = max((SEVERITY_ORDER[value] for value in self.severity_by_rule.values()), default=0)
        return {2: "Fail", 1: "Warning"}.get(worst, "Pass")


def parse_event_timestamp(value: Any) -> tuple[datetime | None, str]:
    """Return (UTC datetime, status) where status is missing, rfc3339, nonstandard, or invalid."""
    if value is None or value == "":
        return None, "missing"
    if isinstance(value, str) and RFC3339_PATTERN.match(value.strip()):
        parsed = parse_timestamp(value)
        return (parsed.astimezone(timezone.utc), "rfc3339") if parsed else (None, "invalid")
    parsed = parse_timestamp(value)
    return (parsed.astimezone(timezone.utc), "nonstandard") if parsed else (None, "invalid")


def metric_lookup(metrics: Any, name: str) -> Any:
    if not isinstance(metrics, dict):
        return None
    if name in metrics:
        return metrics[name]
    target = normalize_key(name)
    return next((value for key, value in metrics.items() if normalize_key(key) == target), None)


def optional_number(value: Any) -> float | None:
    if value is None or isinstance(value, bool) or value == "":
        return None
    sentinel = -1.0
    number = to_number(value, default=sentinel)
    return None if number == sentinel else number


def optional_bool(value: Any) -> bool | None:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)
    if isinstance(value, str) and value.strip().lower() in {"true", "false", "1", "0", "yes", "no"}:
        return value.strip().lower() in {"true", "1", "yes"}
    return None


def milliseconds_between(start: datetime, end: datetime) -> float:
    return (end - start).total_seconds() * 1000.0


def covered_ms(intervals: Iterable[tuple[datetime, datetime]], clip: tuple[datetime, datetime] | None = None) -> tuple[float, int]:
    """Union of intervals (blueprint 10.2); optionally intersected with a parent interval. Returns (ms, clipped_count)."""
    clipped_count = 0
    prepared: list[tuple[datetime, datetime]] = []
    for start, end in intervals:
        if start is None or end is None or end < start:
            continue
        if clip is not None:
            clipped_start, clipped_end = max(start, clip[0]), min(end, clip[1])
            if (clipped_start, clipped_end) != (start, end):
                clipped_count += 1
            if clipped_end < clipped_start:
                continue
            start, end = clipped_start, clipped_end
        prepared.append((start, end))
    if not prepared:
        return 0.0, clipped_count
    prepared.sort()
    merged = [list(prepared[0])]
    for start, end in prepared[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return sum(milliseconds_between(start, end) for start, end in merged), clipped_count


def query_fingerprint(query: Any) -> str:
    """Same normalization as query_signature: whitespace-collapsed, lower-cased SHA-256 prefix."""
    normalized = re.sub(r"\s+", " ", str(query or "").strip()).lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:12] if normalized else ""


PUBLISHED_EVENT_PROPERTIES = {"id", "parentId", "name", "component", "start", "end", "metrics"}


def strict_schema_problems(payload: Any) -> list[tuple[str, str]]:
    """Mirror of Microsoft's published draft-06 export schema (blueprint Appendix A). Returns (group, path)."""
    problems: list[tuple[str, str]] = []
    if not isinstance(payload, dict):
        return [("root is not an object", "$")]
    for key in sorted(set(payload) - {"version", "events"}):
        problems.append((f"root additional property '{key}' is not allowed", f"$.{key}"))
    for key in ("version", "events"):
        if key not in payload:
            problems.append((f"root required property '{key}' is missing", "$"))
    if "version" in payload and payload["version"] != "1.0.0":
        problems.append((f"version {str(payload['version'])[:20]!r} is not in the published enum ['1.0.0']", "$.version"))
    events = payload.get("events")
    if "events" in payload and not isinstance(events, list):
        problems.append(("events is not an array", "$.events"))
        return problems
    for index, event in enumerate(events or []):
        path = f"$.events[{index}]"
        if not isinstance(event, dict):
            problems.append(("event is not an object", path))
            continue
        for key in sorted(set(event) - PUBLISHED_EVENT_PROPERTIES):
            problems.append((f"event additional property '{key}' is not allowed", f"{path}.{key}"))
        for key in ("id", "name", "component", "start"):
            if key not in event:
                problems.append((f"event required property '{key}' is missing", path))
        for key in ("id", "parentId", "name", "component"):
            if key in event and not isinstance(event[key], str):
                problems.append((f"event property '{key}' is not a string", f"{path}.{key}"))
        for key in ("start", "end"):
            if key in event and not (isinstance(event[key], str) and RFC3339_PATTERN.match(event[key]) and parse_timestamp(event[key]) is not None):
                problems.append((f"event property '{key}' is not an RFC 3339 date-time string", f"{path}.{key}"))
        if "metrics" in event and not isinstance(event["metrics"], dict):
            problems.append(("event property 'metrics' is not an object", f"{path}.metrics"))
    return problems


# Query-text privacy policy (blueprint 15). Fingerprints are computed from the original text before redaction.
_NUMBER = r"(?:0[xX][0-9A-Fa-f]+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|\.\d+(?:[eE][+-]?\d+)?)"
_DAX_TOKEN = re.compile(r"'(?:[^']|'')*'|\[(?:[^\]]|\]\])*\]|\"(?:[^\"]|\"\")*\"|(?<![\w.])" + _NUMBER + r"(?![\w\]])")
_SQL_TOKEN = re.compile(r"\"(?:[^\"]|\"\")*\"|\[(?:[^\]]|\]\])*\]|`[^`]*`|\$(?P<tag>[A-Za-z_]\w*|)\$[\s\S]*?\$(?P=tag)\$|(?:(?<!\w)[NnEeXxBb])?'(?:[^']|'')*'|(?<![\w.@#$])" + _NUMBER + r"(?![\w\]])")


def redact_query_text(text: str, dialect: str = "dax") -> str:
    """Replace literal values (strings and numbers) while keeping object names; dialect is 'dax' or 'sql'."""
    def replace(match: re.Match) -> str:
        token = match.group(0)
        if dialect == "dax":
            if token.startswith("\""):
                return "\"<redacted>\""
            return "<n>" if token[0].isdigit() or token[0] == "." else token
        if token.startswith("$") or "'" in token[:2]:
            return "'<redacted>'"
        return "<n>" if token[0].isdigit() or token[0] == "." else token

    return (_DAX_TOKEN if dialect == "dax" else _SQL_TOKEN).sub(replace, text)


def apply_query_text_policy(text: Any, dialect: str = "dax", mode: str | None = None) -> str:
    mode = mode or QUERY_TEXT_MODE
    text = "" if text is None or (isinstance(text, float) and not np.isfinite(text)) else str(text)
    if not text.strip() or mode == "full":
        return text
    if mode == "omit":
        return f"[{'DAX' if dialect == 'dax' else 'Source'} query text omitted: QUERY_TEXT_MODE=omit]"
    return redact_query_text(text, dialect)


QUERY_TEXT_KEYS = ALIASES["query"] | ALIASES["native_query"]


def apply_query_policy_to_json(value: Any, dialect: str = "dax", mode: str | None = None) -> Any:
    """Apply the query-text policy to every query-like key inside a JSON-compatible structure."""
    mode = mode or QUERY_TEXT_MODE
    if isinstance(value, dict):
        result = {}
        for key, child in value.items():
            if normalize_key(key) in QUERY_TEXT_KEYS and mode != "full":
                if isinstance(child, str):
                    result[key] = apply_query_text_policy(child, dialect, mode)
                elif child is None:
                    result[key] = None
                else:
                    result[key] = f"[non-text query value removed: QUERY_TEXT_MODE={mode}]"
            else:
                result[key] = apply_query_policy_to_json(child, dialect, mode)
        return result
    if isinstance(value, list):
        return [apply_query_policy_to_json(child, dialect, mode) for child in value]
    return value


print(f"Event catalog loaded: {len(KNOWN_EVENT_CATALOG)} documented event types | {RULESET_VERSIONS}")

In [ ]:
# Lossless parser, validator, and hierarchy builder (blueprint sections 7.1, 9.1, 12)
def _clean_id(value: Any) -> str | None:
    if value is None or isinstance(value, (dict, list, bool)) or str(value).strip() == "":
        return None
    return str(value)


def normalize_raw_events(raw_events: list[Any], issues: QualityIssueCollector) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    first_index_by_id: dict[str, int] = {}
    for index, raw in enumerate(raw_events):
        record: dict[str, Any] = {
            "event_index": index, "raw": raw, "id": None, "parent_id": None, "name": None,
            "component_raw": None, "start": None, "end": None, "start_raw": None, "end_raw": None,
            "metrics": {}, "is_quarantined": False, "quarantine_reason": "",
        }
        records.append(record)
        if not isinstance(raw, dict):
            issues.add("EVENT_NOT_OBJECT", None, "Event", f"events[{index}]", f"events[{index}] is a {type(raw).__name__}, not an object; quarantined.", {"event_index": index})
            record.update(is_quarantined=True, quarantine_reason="not an object")
            continue
        metrics = direct_value(raw, "metrics")
        record.update(
            id=_clean_id(direct_value(raw, "id")),
            parent_id=_clean_id(direct_value(raw, "parent_id")),
            name=None if direct_value(raw, "name") in (None, "") else str(direct_value(raw, "name")),
            component_raw=None if direct_value(raw, "component") in (None, "") else str(direct_value(raw, "component")),
            start_raw=direct_value(raw, "start"),
            end_raw=direct_value(raw, "end"),
            metrics=metrics if isinstance(metrics, dict) else {},
        )
        scope_id = record["id"] or f"events[{index}]"
        missing = [field for field in ("id", "name") if record[field] is None]
        if record["start_raw"] in (None, ""):
            missing.append("start")
        if missing:
            issues.add("EVENT_REQUIRED_FIELD_MISSING", "Error", "Event", scope_id,
                       f"Missing required field(s) {', '.join(missing)}; event quarantined.", {"event_index": index, "missing": missing})
            record.update(is_quarantined=True, quarantine_reason=f"missing {', '.join(missing)}")
        for field in ("start", "end"):
            parsed, status = parse_event_timestamp(record[f"{field}_raw"])
            record[field] = parsed
            if status == "invalid":
                issues.add("TIMESTAMP_INVALID", None, "Event", scope_id, f"`{field}` cannot be parsed as a date-time; event quarantined.",
                           {"event_index": index, "field": field, "value": str(record[f"{field}_raw"])[:40]})
                record.update(is_quarantined=True, quarantine_reason=(record["quarantine_reason"] + f"; invalid {field}").strip("; "))
            elif status == "nonstandard":
                issues.add("TIMESTAMP_NONSTANDARD", None, "Event", scope_id, f"`{field}` parsed but is not RFC 3339.",
                           {"event_index": index, "field": field, "value": str(record[f"{field}_raw"])[:40]})
        if record["id"] is not None:
            if record["id"] in first_index_by_id:
                issues.add("EVENT_ID_DUPLICATE", None, "Event", record["id"],
                           f"Duplicate id also used by events[{first_index_by_id[record['id']]}]; this occurrence is quarantined.",
                           {"event_index": index, "first_event_index": first_index_by_id[record["id"]]})
                record.update(is_quarantined=True, quarantine_reason=(record["quarantine_reason"] + "; duplicate id").strip("; "))
            else:
                first_index_by_id[record["id"]] = index
    return records


def detect_parent_cycles(by_id: dict[str, dict[str, Any]]) -> tuple[list[list[str]], set[str]]:
    """Return (cycles, nodes whose ancestor chain enters a cycle). Each event has at most one parent."""
    state: dict[str, str] = {}
    in_cycle: set[str] = set()
    reaches_cycle: set[str] = set()
    cycles: list[list[str]] = []
    for start in by_id:
        if start in state:
            continue
        walk: list[str] = []
        walk_index: dict[str, int] = {}
        node: str | None = start
        while node is not None and node in by_id and node not in state:
            walk_index[node] = len(walk)
            walk.append(node)
            state[node] = "walking"
            node = by_id[node]["parent_id"]
            if node in walk_index:
                cycle = walk[walk_index[node]:]
                cycles.append(cycle)
                in_cycle.update(cycle)
                break
        ends_in_cycle = node is not None and (node in in_cycle or node in reaches_cycle)
        for member in walk:
            state[member] = "done"
            if member not in in_cycle and ends_in_cycle:
                reaches_cycle.add(member)
    return cycles, reaches_cycle


def compute_ancestors(by_id: dict[str, dict[str, Any]]) -> dict[str, tuple[str, ...]]:
    ancestors: dict[str, tuple[str, ...]] = {}
    for event_id in by_id:
        if event_id in ancestors:
            continue
        pending: list[str] = []
        current = event_id
        base_node: str | None = None
        while True:
            if current in ancestors:
                base_node = current
                break
            pending.append(current)
            parent = by_id[current]["parent_id"]
            if parent is None or parent not in by_id:
                break
            current = parent
        if base_node is None:
            base_node = pending.pop()
            ancestors[base_node] = ()
        previous = base_node
        while pending:
            node = pending.pop()
            ancestors[node] = (previous,) + ancestors[previous]
            previous = node
    return ancestors


DOCUMENTED_COMPONENTS = {REPORT_CANVAS, DSE_COMPONENT, AS_COMPONENT, CHANGE_DETECTION}


def _name_only_match(record: dict[str, Any], normalized_name: str) -> bool:
    """Pane-style/variant exports only: a matching name counts when the component is not a documented one."""
    return normalize_key(record.get("name")) == normalized_name and record.get("component") not in DOCUMENTED_COMPONENTS


def is_visual_lifecycle_record(record: dict[str, Any]) -> bool:
    return record.get("event_type_key") == VISUAL_KEY or _name_only_match(record, "visualcontainerlifecycle")


def is_user_action_record(record: dict[str, Any]) -> bool:
    return record.get("event_type_key") == USER_ACTION_KEY or _name_only_match(record, "useraction")


def actionable_category_for(record: dict[str, Any]) -> str | None:
    if record.get("is_known_type"):
        return ACTIONABLE_CATEGORY_BY_KEY.get(record["event_type_key"])
    return classify_event(record["raw"]) if isinstance(record.get("raw"), dict) else None


def friendly_category_for(record: dict[str, Any]) -> str:
    if record.get("is_known_type"):
        return KNOWN_TYPE_KEYS[record["event_type_key"]]["category"]
    category = actionable_category_for(record)
    return {
        "dax_ms": "DAX query (name match)", "direct_query_ms": "Direct source query (name match)",
        "render_ms": "Visual display (name match)", "parameter_ms": "Evaluated parameters", "other_ms": "Other (exported)",
    }.get(category, f"Unmapped ({record.get('name')})")


def record_query_text(record: dict[str, Any]) -> str:
    text = metric_lookup(record["metrics"], "QueryText")
    if text in (None, "") and not record.get("is_known_type") and isinstance(record.get("raw"), dict):
        text = event_metrics_query_text(record["raw"])
    return "" if text in (None, "") else str(text)


def resolve_components(records: list[dict[str, Any]], by_id: dict[str, dict[str, Any]], ancestors: dict[str, tuple[str, ...]],
                       issues: QualityIssueCollector, strict: bool) -> None:
    ordered = sorted((record for record in records if record["id"] in by_id and by_id[record["id"]] is record),
                     key=lambda record: len(ancestors.get(record["id"], ())))
    for record in ordered:
        if record["component_raw"]:
            record.update(component=record["component_raw"], component_inferred=False)
            continue
        parent = by_id.get(record["parent_id"]) if record["parent_id"] else None
        if record["name"] in UNIQUE_NAME_COMPONENT:
            component, basis = UNIQUE_NAME_COMPONENT[record["name"]], "documented event name"
        elif parent is not None and parent.get("component") not in (None, "Unknown"):
            component, basis = parent["component"], "parent event component"
        elif record["name"] in OBSERVED_CANVAS_NAMES:
            component, basis = REPORT_CANVAS, "observed canvas event name"
        else:
            component, basis = "Unknown", "no inference available"
        record.update(component=component, component_inferred=True)
        issues.add("EVENT_REQUIRED_FIELD_MISSING", "Error" if strict else "Warning", "Event", record["id"],
                   f"`component` omitted; inferred as '{component}' from {basis}.", {"field": "component", "inferred": component, "basis": basis})
    for record in records:
        if "component" not in record:
            record.update(component=record["component_raw"], component_inferred=False)

In [ ]:
# Build the event model: validation -> hierarchy -> durations -> metrics -> quality rules (blueprint 9.1 order)
EVENT_COLUMNS = [
    "run_id", "event_index", "event_id", "parent_event_id", "component", "component_raw", "component_inferred",
    "event_name", "event_type_key", "is_known_type", "friendly_category", "start_utc", "end_utc", "duration_ms",
    "is_instantaneous", "timing_valid", "depth", "root_event_id", "visual_event_id", "semantic_query_event_id",
    "dax_query_event_id", "is_unattributed", "is_quarantined", "quarantine_reason", "metrics_json", "raw_event_json",
]


def iso_utc(value: datetime | None) -> str | None:
    return value.astimezone(timezone.utc).isoformat().replace("+00:00", "Z") if value else None


def is_dax_record(record: dict[str, Any]) -> bool:
    return record.get("event_type_key") == DAX_KEY or (not record.get("is_known_type") and record.get("actionable_category") == "dax_ms")


def is_direct_query_record(record: dict[str, Any]) -> bool:
    return record.get("event_type_key") == DIRECT_QUERY_KEY or (not record.get("is_known_type") and record.get("actionable_category") == "direct_query_ms")


def normalize_visual_status(raw_status: Any) -> str | None:
    if raw_status is None or raw_status == "":
        return None
    text = str(raw_status).strip()
    mapped = VISUAL_STATUS_CODE_MAP.get(text, VISUAL_STATUS_CODE_MAP.get(raw_status)) if isinstance(VISUAL_STATUS_CODE_MAP, dict) else None
    if mapped:
        return str(mapped).strip().lower()
    return text.lower() if text.lower() in VALID_VISUAL_STATUSES else None


def default_source_info(payload: Any) -> dict[str, Any]:
    canonical = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return {"source_file_name": "<in-memory payload>", "source_sha256": hashlib.sha256(canonical).hexdigest(), "source_size_bytes": None}


def build_event_model(payload: Any, *, validation_mode: str = "compatible", source_info: dict[str, Any] | None = None) -> dict[str, Any]:
    strict = validation_mode == "strict"
    source_info = {**default_source_info(payload), **(source_info or {})}
    issues = QualityIssueCollector(MAX_ISSUES_PER_RULE)
    if not isinstance(payload, dict):
        raise ValueError("ROOT_NOT_OBJECT: a Performance Analyzer event export must have a JSON object at the root.")
    raw_events = direct_value(payload, "events")
    if not isinstance(raw_events, list):
        raise ValueError("ROOT_REQUIRED_FIELD_MISSING: the official event format requires an 'events' array.")
    export_version = direct_value(payload, "version")
    for key, value in (("version", export_version), ("events", raw_events)):
        if key not in payload:
            alias_used = value is not None
            issues.add("ROOT_REQUIRED_FIELD_MISSING", "Warning" if alias_used and not strict else "Error", "Run", "",
                       f"Root property '{key}' is missing" + ("; a recognized alias was used." if alias_used else "."),
                       {"property": key, "alias_used": alias_used})

    schema_groups: dict[str, list[str]] = defaultdict(list)
    for group, path in strict_schema_problems(payload):
        schema_groups[group].append(path)
    for group, paths in sorted(schema_groups.items()):
        issues.add("STRICT_SCHEMA_MISMATCH", "Error" if strict else "Warning", "Run", "",
                   f"{group} ({len(paths):,} occurrence(s)).", {"occurrences": len(paths), "sample_paths": paths[:5]})
    if strict and schema_groups:
        summary = "; ".join(f"{group} x{len(paths):,}" for group, paths in sorted(schema_groups.items())[:8])
        raise StrictSchemaError(
            f"STRICT_SCHEMA_MISMATCH: the input does not match the published Performance Analyzer schema: {summary}. "
            "Set VALIDATION_MODE='compatible' to continue with schema-drift warnings."
        )
    if not raw_events:
        issues.add("EVENTS_EMPTY", None, "Run", "", "The events array is empty; nothing was captured.")

    records = normalize_raw_events(raw_events, issues)
    all_ids = {record["id"] for record in records if record["id"]}
    by_id = {record["id"]: record for record in records if not record["is_quarantined"] and record["id"] is not None}

    cycles, reaches_cycle = detect_parent_cycles(by_id)
    cycle_members = {member for cycle in cycles for member in cycle}
    for cycle in cycles:
        issues.add("PARENT_CYCLE", None, "Event", cycle[0], f"Parent cycle across {len(cycle)} event(s); members and descendants quarantined.",
                   {"cycle_event_ids": cycle[:20]})
    for event_id in sorted(cycle_members | reaches_cycle):
        record = by_id.pop(event_id)
        record.update(is_quarantined=True, quarantine_reason="parent cycle" if event_id in cycle_members else "descendant of parent cycle")

    ancestors = compute_ancestors(by_id)
    resolve_components(records, by_id, ancestors, issues, strict)
    for record in records:
        record["event_type_key"] = make_type_key(record.get("component"), record["name"]) if record["name"] else None
        record["is_known_type"] = record["event_type_key"] in KNOWN_TYPE_KEYS
        record["actionable_category"] = None if record["is_quarantined"] else actionable_category_for(record)
        record["friendly_category"] = friendly_category_for(record) if record["name"] else "Invalid event"
        start, end = record["start"], record["end"]
        record["duration_ms"] = milliseconds_between(start, end) if start and end else None
        record["is_instantaneous"] = record["end_raw"] in (None, "")
        record["timing_valid"] = start is not None and (end is None or end >= start)
        record.update(ancestors=(), depth=None, root_event_id=None, is_unattributed=None, visual_event_id=None,
                      visual_group_id=None, semantic_query_event_id=None, dax_query_event_id=None,
                      status_raw=None, status=None, outside_parent=False)

    tolerance = timedelta(milliseconds=float(CLOCK_TOLERANCE_MS))
    unknown_types: dict[str, dict[str, Any]] = defaultdict(lambda: {"count": 0, "metric_keys": set()})
    unknown_metrics: Counter = Counter()
    nonstandard_statuses: Counter = Counter()
    for record in (record for record in records if record["id"] in by_id and by_id[record["id"]] is record):
        event_id = record["id"]
        chain_ids = ancestors[event_id]
        chain = (record,) + tuple(by_id[ancestor] for ancestor in chain_ids)
        root_id = chain_ids[-1] if chain_ids else event_id
        record.update(ancestors=chain_ids, depth=len(chain_ids), root_event_id=root_id,
                      is_unattributed=by_id[root_id]["parent_id"] is not None)
        if record["parent_id"] and record["parent_id"] not in by_id:
            issues.add("PARENT_NOT_FOUND", None, "Event", event_id, f"parentId '{record['parent_id']}' does not resolve; subtree left unattributed.",
                       {"parent_id": record["parent_id"], "parent_quarantined": record["parent_id"] in all_ids})
        record["visual_event_id"] = next((item["id"] for item in chain if is_visual_lifecycle_record(item)), None)
        record["semantic_query_event_id"] = next((item["id"] for item in chain if item["event_type_key"] in (CANVAS_QUERY_KEY, SEMANTIC_QUERY_KEY)), None)
        record["dax_query_event_id"] = next((item["id"] for item in chain if is_dax_record(item)), None)
        if record["visual_event_id"]:
            record["visual_group_id"] = record["visual_event_id"]
        elif not record["is_unattributed"]:
            metadata_parent = next((item["id"] for item in chain[1:] if isinstance(item["raw"], dict)
                                    and (event_metadata(item["raw"])["visual_id"] or event_metadata(item["raw"])["visual"])), None)
            record["visual_group_id"] = metadata_parent or (chain_ids[0] if chain_ids else event_id)

        spec = KNOWN_TYPE_KEYS.get(record["event_type_key"])
        if spec is None:
            unknown_types[record["event_type_key"]]["count"] += 1
            unknown_types[record["event_type_key"]]["metric_keys"].update(map(str, record["metrics"].keys()))
        else:
            for key in record["metrics"]:
                if key not in spec["metrics"]:
                    unknown_metrics[(record["event_type_key"], str(key))] += 1
            if spec["has_end"] and record["end"] is None:
                issues.add("EVENT_END_MISSING", None, "Event", event_id, f"{record['event_type_key']} has no end; duration is unknown.", {"event_type_key": record["event_type_key"]})
        if record["start"] and record["end"] and record["end"] < record["start"]:
            issues.add("NEGATIVE_DURATION", None, "Event", event_id, f"end precedes start by {-record['duration_ms']:,.3f} ms; raw timestamps retained.",
                       {"start": record["start_raw"], "end": record["end_raw"], "duration_ms": record["duration_ms"]})
        parent = by_id.get(record["parent_id"]) if record["parent_id"] else None
        if parent is not None and parent["timing_valid"] and parent["end"] is not None and record["timing_valid"]:
            child_end = record["end"] or record["start"]
            if record["start"] < parent["start"] - tolerance or child_end > parent["end"] + tolerance:
                record["outside_parent"] = True
                issues.add("CHILD_OUTSIDE_PARENT", None, "Event", event_id, f"Interval falls outside parent {parent['id']} beyond {CLOCK_TOLERANCE_MS} ms tolerance.",
                           {"parent_id": parent["id"], "starts_before_parent_ms": round(max(milliseconds_between(record["start"], parent["start"]), 0), 3),
                            "ends_after_parent_ms": round(max(milliseconds_between(parent["end"], child_end), 0), 3)})
        if normalize_key(record["name"]) == "metricstruncated":
            omitted = optional_number(metric_lookup(record["metrics"], "Count"))
            issues.add("TRACE_TRUNCATED", None, "Event", event_id,
                       f"{record['component']} trace truncated; " + (f"{omitted:,.0f} event(s) omitted." if omitted is not None else "omitted count not supplied."),
                       {"component": record["component"], "omitted_count": omitted, "visual_event_id": record["visual_event_id"], "dax_query_event_id": record["dax_query_event_id"]})
        if is_visual_lifecycle_record(record):
            record["status_raw"] = metric_lookup(record["metrics"], "status")
            record["status"] = normalize_visual_status(record["status_raw"])
            if record["status"] == "abandoned":
                issues.add("VISUAL_ABANDONED", None, "Visual", event_id, "Visual update was abandoned before completion.",
                           {"visual_title": metric_lookup(record["metrics"], "visualTitle"), "status_raw": record["status_raw"]})
            elif record["status_raw"] not in (None, "") and record["status"] is None:
                nonstandard_statuses[str(record["status_raw"])] += 1
        if is_dax_record(record):
            if optional_bool(metric_lookup(record["metrics"], "Error")):
                issues.add("DAX_ERROR", None, "Query", event_id, "DAX query reported Error=true.", {"visual_event_id": record["visual_event_id"]})
            if optional_bool(metric_lookup(record["metrics"], "Canceled")):
                issues.add("DAX_CANCELED", None, "Query", event_id, "DAX query reported Canceled=true.", {"visual_event_id": record["visual_event_id"]})
        if (is_dax_record(record) or is_direct_query_record(record)) and not record_query_text(record).strip():
            kind = "DAX" if is_dax_record(record) else "DirectQuery source"
            reason = "" if kind == "DAX" else " Source text is documented only when the user owns the model and the source is SQL."
            issues.add("QUERY_TEXT_UNAVAILABLE", None, "Query", event_id, f"{kind} query text is not present in the export.{reason}", {"event_type_key": record["event_type_key"]})

    for raw_status, count in sorted(nonstandard_statuses.items()):
        issues.add("VISUAL_STATUS_NONSTANDARD", None, "Run", "", f"Visual lifecycle status {raw_status!r} is undocumented ({count:,} visual(s)); map it in VISUAL_STATUS_CODE_MAP only after confirming its meaning.",
                   {"status_raw": raw_status, "visual_count": count})
    for key, details in sorted(unknown_types.items(), key=lambda item: str(item[0])):
        issues.add("UNKNOWN_EVENT_TYPE", None, "Run", key, f"New event type '{key}' observed {details['count']:,} time(s); retained in events.csv.",
                   {"count": details["count"], "metric_keys": sorted(details["metric_keys"])})
    for (key, metric), count in sorted(unknown_metrics.items()):
        issues.add("UNKNOWN_METRIC", None, "Run", key, f"Undocumented metric '{metric}' on '{key}' observed {count:,} time(s); retained in metrics_json.",
                   {"event_type_key": key, "metric": metric, "count": count})

    run_id = source_info["source_sha256"]
    event_rows = []
    for record in records:
        raw = record["raw"]
        raw_metrics = direct_value(raw, "metrics") if isinstance(raw, dict) else None
        event_rows.append({
            "run_id": run_id, "event_index": record["event_index"], "event_id": record["id"], "parent_event_id": record["parent_id"],
            "component": record.get("component"), "component_raw": record["component_raw"], "component_inferred": bool(record.get("component_inferred")),
            "event_name": record["name"], "event_type_key": record["event_type_key"], "is_known_type": record["is_known_type"],
            "friendly_category": record["friendly_category"], "start_utc": iso_utc(record["start"]), "end_utc": iso_utc(record["end"]),
            "duration_ms": None if record["duration_ms"] is None else round(record["duration_ms"], 3),
            "is_instantaneous": record["is_instantaneous"], "timing_valid": record["timing_valid"], "depth": record["depth"],
            "root_event_id": record["root_event_id"], "visual_event_id": record["visual_event_id"],
            "semantic_query_event_id": record["semantic_query_event_id"], "dax_query_event_id": record["dax_query_event_id"],
            "is_unattributed": record["is_unattributed"], "is_quarantined": record["is_quarantined"], "quarantine_reason": record["quarantine_reason"],
            "metrics_json": None if raw_metrics is None else json.dumps(raw_metrics, ensure_ascii=False, sort_keys=True, default=str),
            "raw_event_json": json.dumps(raw, ensure_ascii=False, default=str),
        })
    model = {
        "payload_root": {key: value for key, value in payload.items() if not isinstance(value, list)},
        "export_version": export_version, "validation_mode": validation_mode, "source_info": source_info,
        "strict_schema_valid": not schema_groups, "records": records, "by_id": by_id, "issues": issues,
        "events": pd.DataFrame(event_rows, columns=EVENT_COLUMNS).astype({"depth": "Int64"}),
    }
    return build_derived_tables(model)

In [ ]:
# Derived tables: visual updates, DAX queries, DirectQuery queries, interactions, run summary (blueprint 8, 9.3, 10, 11)
VISUAL_UPDATE_COLUMNS = [
    "visual_event_id", "visual_id", "visual_title", "visual_type", "status", "status_raw", "start_utc", "end_utc",
    "total_elapsed_ms", "has_canvas_query", "has_dax_query", "has_direct_query", "possible_canvas_cache_hit", "backend_activity",
    "query_count", "direct_query_count", "canvas_query_ms", "dax_covered_ms", "dax_descendant_sum_ms", "direct_query_covered_ms",
    "query_generation_ms", "parse_ms", "render_ms", "transform_ms", "geocoding_ms", "parameter_ms", "display_covered_ms", "derived_other_ms",
    "dax_error_count", "dax_canceled_count", "max_dax_row_count", "total_rows_read", "is_truncated", "clipped_child_count",
    "children_outside_parent", "next_update_event_id", "derived_interaction_id", "interaction_label", "flags", "data_quality_status",
]
DAX_QUERY_COLUMNS = [
    "dax_event_id", "visual_event_id", "visual_id", "visual_title", "visual_type", "semantic_query_event_id", "start_utc", "end_utc",
    "duration_ms", "row_count", "error", "canceled", "query_fingerprint", "query_text_available", "as_event_count",
    "direct_query_count", "direct_query_covered_ms", "is_truncated", "derived_interaction_id", "query_text",
]
DIRECT_QUERY_COLUMNS = [
    "direct_query_event_id", "dax_event_id", "visual_event_id", "visual_id", "visual_title", "start_utc", "end_utc", "duration_ms",
    "actual_query_duration_ms", "data_read_duration_ms", "rows_read", "is_capabilities_query", "nested_in_direct_query",
    "query_fingerprint", "query_text_available", "query_text",
]
USER_ACTION_COLUMNS = ["derived_interaction_id", "user_action_event_id", "start_utc", "source_label"]


def _cover(events: list[dict[str, Any]], predicate, clip) -> tuple[float | None, int, int]:
    """Return (covered ms or None when not observed/unknown, clipped count, matching event count)."""
    matching = [event for event in events if predicate(event)]
    if not matching:
        return None, 0, 0
    intervals = [(event["start"], event["end"]) for event in matching if event["timing_valid"] and event["end"] is not None]
    if not intervals:
        return None, 0, len(matching)
    milliseconds, clipped = covered_ms(intervals, clip)
    return round(milliseconds, 3), clipped, len(matching)


def _is_render(event: dict[str, Any]) -> bool:
    return event["event_type_key"] == make_type_key(REPORT_CANVAS, "Render") or (not event["is_known_type"] and event["actionable_category"] == "render_ms")


def _is_parameter(event: dict[str, Any]) -> bool:
    return event["friendly_category"] == "Evaluated parameters"


def _is_explained_canvas(event: dict[str, Any]) -> bool:
    return event["friendly_category"] in EXPLAINED_CANVAS_CATEGORIES or _is_render(event) or _is_parameter(event)


def attribute_interactions(visuals: list[dict[str, Any]], user_actions: list[dict[str, Any]]) -> dict[str, tuple[str, str]]:
    """Temporal heuristic (blueprint 9.3): root visual -> nearest preceding User Action within INTERACTION_WINDOW_MS."""
    result: dict[str, tuple[str, str]] = {}
    starts = [action["start"] for action in user_actions]
    for visual in visuals:
        if visual["parent_id"] is not None or visual["start"] is None:
            continue
        position = bisect.bisect_right(starts, visual["start"])
        if position == 0:
            continue
        index, latest = position - 1, user_actions[position - 1]
        if index > 0 and starts[index - 1] == latest["start"]:
            continue
        if milliseconds_between(latest["start"], visual["start"]) > float(INTERACTION_WINDOW_MS):
            continue
        label = metric_lookup(latest["metrics"], "sourceLabel")
        result[visual["id"]] = (f"interaction-{index + 1:03d}", "" if label is None else str(label))
    return result


def build_derived_tables(model: dict[str, Any]) -> dict[str, Any]:
    records, by_id, issues = model["records"], model["by_id"], model["issues"]
    analyzable = [record for record in records if record["id"] in by_id and by_id[record["id"]] is record]
    events_by_visual: dict[str, list[dict[str, Any]]] = defaultdict(list)
    events_by_dax: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for record in analyzable:
        if record["visual_event_id"] and record["visual_event_id"] != record["id"]:
            events_by_visual[record["visual_event_id"]].append(record)
        if record["dax_query_event_id"] and record["dax_query_event_id"] != record["id"]:
            events_by_dax[record["dax_query_event_id"]].append(record)
    visuals = [record for record in analyzable if is_visual_lifecycle_record(record)]
    user_actions = sorted((record for record in analyzable if is_user_action_record(record)),
                          key=lambda record: (record["start"], record["event_index"]))
    interactions = attribute_interactions(visuals, user_actions)
    model["interactions"] = interactions

    def visual_context(visual_event_id: str | None) -> dict[str, Any]:
        visual = by_id.get(visual_event_id) if visual_event_id else None
        metrics = visual["metrics"] if visual else {}
        return {"visual_id": metric_lookup(metrics, "visualId"), "visual_title": metric_lookup(metrics, "visualTitle"),
                "visual_type": metric_lookup(metrics, "visualType")}

    dax_rows, direct_rows = [], []
    for record in analyzable:
        if is_dax_record(record):
            text = record_query_text(record)
            descendants = events_by_dax.get(record["id"], [])
            direct_children = [item for item in descendants if is_direct_query_record(item)]
            direct_cover, _, _ = _cover(direct_children, lambda item: True, None)
            dax_rows.append({
                "dax_event_id": record["id"], "visual_event_id": record["visual_event_id"], **visual_context(record["visual_event_id"]),
                "semantic_query_event_id": record["semantic_query_event_id"], "start_utc": iso_utc(record["start"]), "end_utc": iso_utc(record["end"]),
                "duration_ms": None if record["duration_ms"] is None else round(record["duration_ms"], 3),
                "row_count": optional_number(metric_lookup(record["metrics"], "RowCount")),
                "error": optional_bool(metric_lookup(record["metrics"], "Error")),
                "canceled": optional_bool(metric_lookup(record["metrics"], "Canceled")),
                "query_fingerprint": query_fingerprint(text), "query_text_available": bool(text.strip()),
                "as_event_count": sum(1 for item in descendants if item.get("component") == AS_COMPONENT),
                "direct_query_count": len(direct_children), "direct_query_covered_ms": direct_cover,
                "is_truncated": any(normalize_key(item["name"]) == "metricstruncated" for item in descendants),
                "derived_interaction_id": interactions.get(record["visual_event_id"], ("", ""))[0], "query_text": text,
            })
        if is_direct_query_record(record):
            text = record_query_text(record)
            nested = any(is_direct_query_record(by_id[ancestor]) for ancestor in record["ancestors"])
            direct_rows.append({
                "direct_query_event_id": record["id"], "dax_event_id": record["dax_query_event_id"], "visual_event_id": record["visual_event_id"],
                "visual_id": visual_context(record["visual_event_id"])["visual_id"], "visual_title": visual_context(record["visual_event_id"])["visual_title"],
                "start_utc": iso_utc(record["start"]), "end_utc": iso_utc(record["end"]),
                "duration_ms": None if record["duration_ms"] is None else round(record["duration_ms"], 3),
                "actual_query_duration_ms": optional_number(metric_lookup(record["metrics"], "ActualQueryDuration")),
                "data_read_duration_ms": optional_number(metric_lookup(record["metrics"], "DataReadDuration")),
                "rows_read": optional_number(metric_lookup(record["metrics"], "RowsRead")),
                "is_capabilities_query": optional_bool(metric_lookup(record["metrics"], "IsGetSourceCapabilitiesQuery")),
                "nested_in_direct_query": nested, "query_fingerprint": query_fingerprint(text), "query_text_available": bool(text.strip()), "query_text": text,
            })
    dax_frame = pd.DataFrame(dax_rows, columns=DAX_QUERY_COLUMNS)
    direct_frame = pd.DataFrame(direct_rows, columns=DIRECT_QUERY_COLUMNS)

    visual_rows = []
    for visual in visuals:
        descendants = events_by_visual.get(visual["id"], [])
        total = round(visual["duration_ms"], 3) if visual["timing_valid"] and visual["end"] is not None else None
        clip = (visual["start"], visual["end"]) if total is not None else None
        measures = {}
        for name, predicate in [
            ("canvas_query_ms", lambda item: item["event_type_key"] == CANVAS_QUERY_KEY),
            ("dax_covered_ms", is_dax_record), ("direct_query_covered_ms", is_direct_query_record),
            ("query_generation_ms", lambda item: item["friendly_category"] == "Query generation"),
            ("parse_ms", lambda item: item["friendly_category"] == "Result parsing"), ("render_ms", _is_render),
            ("transform_ms", lambda item: item["friendly_category"] == "Data transform"),
            ("geocoding_ms", lambda item: item["friendly_category"] == "Geocoding"), ("parameter_ms", _is_parameter),
            ("display_covered_ms", lambda item: _is_render(item) or item["friendly_category"] in {"Data transform", "Geocoding"}),
        ]:
            measures[name], _, _ = _cover(descendants, predicate, clip)
        tolerance = timedelta(milliseconds=float(CLOCK_TOLERANCE_MS))
        clipped_total = 0 if clip is None else sum(
            1 for item in descendants
            if item["timing_valid"] and item["end"] is not None and (item["start"] < clip[0] - tolerance or item["end"] > clip[1] + tolerance)
        )
        explained, _, explained_count = _cover(descendants, _is_explained_canvas, clip)
        derived_other = None if total is None else round(max(total - (explained or 0.0), 0.0), 3)
        dax_events = [item for item in descendants if is_dax_record(item)]
        direct_events = [item for item in descendants if is_direct_query_record(item)]
        has_semantic = any(item["event_type_key"] == SEMANTIC_QUERY_KEY for item in descendants)
        has_canvas_query = any(item["event_type_key"] == CANVAS_QUERY_KEY for item in descendants)
        truncated = any(normalize_key(item["name"]) == "metricstruncated" for item in descendants)
        has_as = any(item.get("component") == AS_COMPONENT for item in descendants)
        cache_hit = not dax_events and not has_semantic and not truncated
        if dax_events or has_semantic:
            backend = "Observed" if has_as else "DSE observed; AS detail not observed (possible remote model)"
        elif truncated:
            backend = "Not observed (trace truncated)"
        elif has_canvas_query:
            backend = "Not observed (possible canvas cache hit)"
        else:
            backend = "Not observed (possible canvas cache hit or non-query visual)"
        dax_errors = sum(1 for item in dax_events if optional_bool(metric_lookup(item["metrics"], "Error")))
        dax_canceled = sum(1 for item in dax_events if optional_bool(metric_lookup(item["metrics"], "Canceled")))
        outside = sum(1 for item in descendants if item["outside_parent"])
        negative_children = sum(1 for item in descendants if item["duration_ms"] is not None and item["duration_ms"] < 0)
        missing_end_children = sum(1 for item in descendants if item["is_known_type"] and KNOWN_TYPE_KEYS[item["event_type_key"]]["has_end"] and item["end"] is None)
        row_counts = [value for value in (optional_number(metric_lookup(item["metrics"], "RowCount")) for item in dax_events) if value is not None]
        rows_read = [value for value in (optional_number(metric_lookup(item["metrics"], "RowsRead")) for item in direct_events) if value is not None]
        next_update = None
        this_visual_id = metric_lookup(visual["metrics"], "visualId")
        if visual["status"] == "abandoned" and this_visual_id not in (None, ""):
            same_visual = [item for item in visuals if item is not visual and metric_lookup(item["metrics"], "visualId") == this_visual_id
                           and item["start"] and visual["start"] and item["start"] >= visual["start"]]
            next_update = min(same_visual, key=lambda item: item["start"])["id"] if same_visual else None
        flags = []
        if visual["status"] == "abandoned":
            flags.append("Abandoned")
        elif visual["status"] == "started":
            flags.append("Status started (not finished)")
        elif visual["status_raw"] not in (None, "") and visual["status"] is None:
            flags.append(f"Undocumented status {visual['status_raw']}")
        if total is None:
            flags.append("Lifecycle duration unknown" if visual["end"] is None else "Negative lifecycle duration")
        if truncated:
            flags.append("Trace truncated")
        if dax_errors:
            flags.append(f"DAX error ({dax_errors})")
        if dax_canceled:
            flags.append(f"DAX canceled ({dax_canceled})")
        if backend != "Observed":
            flags.append(backend)
        if outside or visual["outside_parent"]:
            flags.append(f"Children outside parent interval ({outside})")
        if clipped_total:
            flags.append(f"Child intervals clipped to lifecycle ({clipped_total})")
        if negative_children:
            flags.append(f"Child negative duration ({negative_children})")
        if missing_end_children:
            flags.append(f"Child end missing ({missing_end_children})")
        if any(not record_query_text(item).strip() for item in dax_events + direct_events):
            flags.append("Query text unavailable")
        if total is None or truncated or negative_children or missing_end_children:
            quality = "Incomplete"
        elif visual["status"] in {"abandoned", "started"} or dax_errors or dax_canceled or outside or clipped_total:
            quality = "Warning"
        else:
            quality = "Good"
        interaction = interactions.get(visual["id"], ("", ""))
        visual_rows.append({
            "visual_event_id": visual["id"], **visual_context(visual["id"]),
            "status": visual["status"] or ("not supplied" if visual["status_raw"] in (None, "") else f"undocumented ({visual['status_raw']})"),
            "status_raw": None if visual["status_raw"] is None else str(visual["status_raw"]),
            "start_utc": iso_utc(visual["start"]), "end_utc": iso_utc(visual["end"]), "total_elapsed_ms": total,
            "has_canvas_query": has_canvas_query, "has_dax_query": bool(dax_events), "has_direct_query": bool(direct_events),
            "possible_canvas_cache_hit": cache_hit, "backend_activity": backend, "query_count": len(dax_events),
            "direct_query_count": len(direct_events), **measures,
            "dax_descendant_sum_ms": round(sum(item["duration_ms"] for item in dax_events if item["duration_ms"] is not None and item["duration_ms"] >= 0), 3) if dax_events else None,
            "derived_other_ms": derived_other, "dax_error_count": dax_errors, "dax_canceled_count": dax_canceled,
            "max_dax_row_count": max(row_counts) if row_counts else None, "total_rows_read": sum(rows_read) if rows_read else None,
            "is_truncated": truncated, "clipped_child_count": clipped_total, "children_outside_parent": outside,
            "next_update_event_id": next_update, "derived_interaction_id": interaction[0], "interaction_label": interaction[1],
            "flags": "; ".join(flags), "data_quality_status": quality,
        })
    visual_frame = pd.DataFrame(visual_rows, columns=VISUAL_UPDATE_COLUMNS)
    user_action_frame = pd.DataFrame([
        {"derived_interaction_id": f"interaction-{index + 1:03d}", "user_action_event_id": action["id"],
         "start_utc": iso_utc(action["start"]), "source_label": metric_lookup(action["metrics"], "sourceLabel")}
        for index, action in enumerate(user_actions)
    ], columns=USER_ACTION_COLUMNS)

    starts = [record["start"] for record in analyzable if record["start"]]
    ends = [record["end"] for record in analyzable if record["end"]] or starts
    counts = issues.counts
    class_counts: Counter = Counter()
    for rule_id, count in counts.items():
        class_counts[QUALITY_RULES[rule_id]["class"]] += count
    status_counts = Counter(row["status"] for row in visual_rows)
    omitted = [optional_number(metric_lookup(record["metrics"], "Count")) for record in analyzable if normalize_key(record["name"]) == "metricstruncated"]
    assessment = []
    if any(issues.severity_by_rule.get(rule) == "Error" and QUALITY_RULES[rule]["class"] == "Invalid input" for rule in counts):
        assessment.append("Invalid input records quarantined")
    if class_counts.get("Incomplete trace"):
        assessment.append("Incomplete trace")
    if class_counts.get("Schema drift"):
        assessment.append("Valid but unfamiliar schema drift")
    if class_counts.get("Clock alignment"):
        assessment.append("Clock-alignment anomalies")
    source_info = model["source_info"]
    model.update(
        visual_updates=visual_frame, dax_queries=dax_frame, direct_queries=direct_frame, user_actions=user_action_frame,
        quality_issues=issues.frame(), quality_status=issues.status(),
    )
    model["run_summary"] = {
        "run_id": source_info["source_sha256"], "source_file_name": source_info["source_file_name"],
        "source_sha256": source_info["source_sha256"], "source_size_bytes": source_info.get("source_size_bytes"),
        "export_version": None if model["export_version"] is None else str(model["export_version"]),
        "session_id": model["payload_root"].get("sessionId"),
        "event_count": len(records), "analyzable_event_count": len(analyzable),
        "quarantined_event_count": sum(1 for record in records if record["is_quarantined"]),
        "unattributed_event_count": sum(1 for record in analyzable if record["is_unattributed"]),
        "min_start_utc": iso_utc(min(starts)) if starts else None, "max_end_utc": iso_utc(max(ends)) if ends else None,
        "capture_elapsed_ms": round(milliseconds_between(min(starts), max(ends)), 3) if starts else None,
        "strict_schema_valid": model["strict_schema_valid"], "validation_mode": model["validation_mode"],
        "quality_status": model["quality_status"], "input_assessment": "; ".join(assessment) or "Valid",
        "issue_counts_by_class": dict(sorted(class_counts.items())), "issue_counts_by_rule": dict(sorted(counts.items())),
        "visual_update_count": len(visual_rows), "visual_status_counts": dict(sorted(status_counts.items())),
        "dax_query_count": len(dax_rows), "direct_query_count": len(direct_rows),
        "dax_error_count": counts.get("DAX_ERROR", 0), "dax_canceled_count": counts.get("DAX_CANCELED", 0),
        "orphan_count": counts.get("PARENT_NOT_FOUND", 0), "cycle_count": counts.get("PARENT_CYCLE", 0),
        "duplicate_id_count": counts.get("EVENT_ID_DUPLICATE", 0), "truncation_count": counts.get("TRACE_TRUNCATED", 0),
        "truncated_omitted_events": sum(value for value in omitted if value is not None) if any(value is not None for value in omitted) else None,
        "unknown_event_type_count": counts.get("UNKNOWN_EVENT_TYPE", 0), "user_action_count": len(user_actions),
        "capture_metadata": source_info.get("capture_metadata", {}), "query_text_mode": QUERY_TEXT_MODE, **RULESET_VERSIONS,
    }
    return model

In [ ]:
# Visual-level timing frame from the event model, plus the parser dispatcher
def build_visual_frame(model: dict[str, Any]) -> pd.DataFrame:
    records, by_id = model["records"], model["by_id"]
    analyzable = [record for record in records if record["id"] in by_id and by_id[record["id"]] is record]
    visual_updates = model["visual_updates"].set_index("visual_event_id", drop=False) if not model["visual_updates"].empty else pd.DataFrame()

    transitions = sorted(record["start"] for record in analyzable
                         if record["start"] and normalize_key(metric_lookup(record["metrics"], "sourceLabel") or "") == "useractionchangepage")
    transitions = [(start, f"Page transition {index} (name unavailable)") for index, start in enumerate(transitions, start=1)]

    def inferred_page_label(start: datetime | None) -> str | None:
        labels = [label for transition_start, label in transitions if start and transition_start <= start]
        return labels[-1] if labels else None

    groups: dict[str, list[tuple[str, dict[str, Any]]]] = {}
    for record in analyzable:
        if is_visual_lifecycle_record(record):
            groups.setdefault(record["id"], [])
        if record["actionable_category"] and record["visual_group_id"]:
            groups.setdefault(record["visual_group_id"], []).append((record["actionable_category"], record))

    rows: list[dict[str, Any]] = []
    for group_id, categorized in groups.items():
        group = by_id[group_id]
        clip = (group["start"], group["end"]) if group["timing_valid"] and group["end"] is not None else None
        by_category: dict[str, list[dict[str, Any]]] = defaultdict(list)
        for category, record in categorized:
            by_category[category].append(record)
        durations = {}
        for category in ("dax_ms", "direct_query_ms", "render_ms", "parameter_ms", "other_ms"):
            covered, _, _ = _cover(by_category.get(category, []), lambda item: True, clip)
            durations[category] = covered or 0.0
        # Interval-safe aggregates: union of DAX and DirectQuery intervals, then union across all actionable intervals.
        query_union, _, _ = _cover(by_category.get("dax_ms", []) + by_category.get("direct_query_ms", []), lambda item: True, clip)
        actionable_union, _, _ = _cover(
            by_category.get("dax_ms", []) + by_category.get("direct_query_ms", []) + by_category.get("render_ms", []) + by_category.get("parameter_ms", []),
            lambda item: True, clip,
        )
        query_ms = max(query_union or 0.0, durations["dax_ms"], durations["direct_query_ms"])
        actionable_ms = max(actionable_union or 0.0, query_ms)
        measured = group["duration_ms"] if group["timing_valid"] and group["duration_ms"] is not None and group["duration_ms"] > 0 else None
        total_ms = measured if measured is not None else actionable_ms + durations["other_ms"]
        if measured is not None:
            total_source = "lifecycle"
        elif group["end"] is None:
            total_source = "derived from components (lifecycle end unavailable)"
        elif group["duration_ms"] == 0:
            total_source = "derived from components (lifecycle duration is zero)"
        else:
            total_source = "derived from components (lifecycle duration invalid)"
        if durations["other_ms"] == 0 and total_ms > actionable_ms:
            durations["other_ms"] = total_ms - actionable_ms

        context = [group["raw"], *(by_id[ancestor]["raw"] for ancestor in group["ancestors"]), *(record["raw"] for _, record in categorized)]
        metadata: dict[str, Any] = {}
        for field in ("page", "visual", "visual_id", "visual_type"):
            metadata[field] = next((event_metadata(item)[field] for item in context if isinstance(item, dict) and event_metadata(item)[field]), None)

        dax_events = sorted(by_category.get("dax_ms", []), key=lambda item: -(item["duration_ms"] or 0))
        dax_query = next((record_query_text(item) for item in dax_events if record_query_text(item).strip()), "")
        direct_events = sorted(by_category.get("direct_query_ms", []), key=lambda item: (item["start"], item["event_index"]))
        top_level_direct = [item for item in direct_events if not any(is_direct_query_record(by_id[ancestor]) for ancestor in item["ancestors"])]
        executions = []
        for item in top_level_direct:
            text = record_query_text(item)
            executions.append({
                "event_id": item["id"],
                "duration_ms": round(item["duration_ms"], 3) if item["duration_ms"] is not None and item["duration_ms"] >= 0 else None,
                "native_query_text": text,
                "capture_status": "Captured" if text.strip() else "DirectQuery text was not captured",
                "actual_query_duration_ms": optional_number(metric_lookup(item["metrics"], "ActualQueryDuration")),
                "rows_read": optional_number(metric_lookup(item["metrics"], "RowsRead")),
            })
        native_query_text = next((execution["native_query_text"] for execution in executions if execution["native_query_text"].strip()), "")

        fallback_name = str(group["name"] or "Visual")
        if normalize_key(fallback_name) in {"visualcontainerlifecycle", "updatevisual", "visual"}:
            fallback_name = f"Visual {metadata['visual_id'] or group_id}"
        update = visual_updates.loc[group_id].to_dict() if group_id in visual_updates.index else {}
        rows.append({
            "page": str(metadata["page"] or inferred_page_label(group["start"]) or "Page unavailable in export"),
            "visual": str(metadata["visual"] or fallback_name),
            "visual_id": str(metadata["visual_id"] or group_id),
            "visual_type": str(metadata["visual_type"] or "Unknown"),
            "start_time": str(group["start_raw"] or ""),
            "total_ms": round(total_ms, 3),
            **{key: round(value, 3) for key, value in durations.items()},
            "query_ms": round(query_ms, 3),
            "actionable_ms": round(actionable_ms, 3),
            "dax_query": dax_query,
            "native_query_text": native_query_text,
            "direct_query_executions": executions,
            "source_event_count": len(categorized),
            "raw_properties": json.dumps(group["raw"], ensure_ascii=True, default=str),
            "visual_event_id": group_id,
            "status": str(update.get("status") or ""),
            "total_ms_source": total_source,
            "dax_query_count": len(dax_events),
            "direct_query_count": len(direct_events),
            "dax_error_count": int(update.get("dax_error_count") or sum(1 for item in dax_events if optional_bool(metric_lookup(item["metrics"], "Error")))),
            "dax_canceled_count": int(update.get("dax_canceled_count") or sum(1 for item in dax_events if optional_bool(metric_lookup(item["metrics"], "Canceled")))),
            "has_dax_query": bool(dax_events),
            "has_direct_query": bool(direct_events),
            "possible_canvas_cache_hit": bool(update.get("possible_canvas_cache_hit", False)),
            "is_truncated": bool(update.get("is_truncated", False)),
            "data_quality_flags": str(update.get("flags") or ""),
            "data_quality_status": str(update.get("data_quality_status") or "Not assessed (no lifecycle event)"),
            "derived_interaction_id": str(update.get("derived_interaction_id") or ""),
            "interaction_label": str(update.get("interaction_label") or ""),
        })
    return apply_visual_defaults(pd.DataFrame(rows, columns=NORMALIZED_COLUMNS))


def build_flat_model(payload: Any, frame: pd.DataFrame, validation_mode: str, source_info: dict[str, Any] | None) -> dict[str, Any]:
    source_info = {**default_source_info(payload), **(source_info or {})}
    issues = QualityIssueCollector(MAX_ISSUES_PER_RULE)
    if not isinstance(payload, dict):
        issues.add("ROOT_NOT_OBJECT", "Warning", "Run", "", "Root is an array; parsed as flattened records instead of an official event export.")
    else:
        issues.add("ROOT_REQUIRED_FIELD_MISSING", "Warning", "Run", "", "No 'events' array; parsed as flattened records.", {"top_level_keys": sorted(map(str, payload))[:20]})
    issues.add("NON_OFFICIAL_SHAPE", None, "Run", "", f"{len(frame):,} flattened visual record(s) parsed; event hierarchy, duplicate, cycle, truncation, and timestamp checks are unavailable.")
    model = {
        "payload_root": payload if isinstance(payload, dict) and not any(isinstance(value, list) for value in payload.values()) else {},
        "export_version": direct_value(payload, "version") if isinstance(payload, dict) else None, "validation_mode": validation_mode,
        "source_info": source_info, "strict_schema_valid": False, "records": [], "by_id": {}, "issues": issues,
        "events": pd.DataFrame(columns=EVENT_COLUMNS),
    }
    return build_derived_tables(model)


def parse_performance_analyzer_with_model(payload: Any, validation_mode: str | None = None,
                                          source_info: dict[str, Any] | None = None) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    mode = validation_mode or VALIDATION_MODE
    if not isinstance(payload, (dict, list)):
        raise ValueError("Performance Analyzer JSON must contain an object or array at the top level.")
    errors: list[str] = []
    events = direct_value(payload, "events") if isinstance(payload, dict) else None
    if events is None and mode == "strict":
        raise StrictSchemaError("STRICT_SCHEMA_MISMATCH: strict mode requires an official export with a root 'events' array.")
    if events is not None:
        model = None
        try:
            model = build_event_model(payload, validation_mode=mode, source_info=source_info)
        except StrictSchemaError:
            raise
        except ValueError as exc:
            errors.append(str(exc))
        if model is not None:
            frame = build_visual_frame(model)
            if not frame.empty:
                return frame, {
                    "format": "official-event-tree",
                    "version": direct_value(payload, "version") or "unknown",
                    "event_count": len(events) if isinstance(events, list) else 0,
                }, model
            errors.append("the 'events' array is empty" if isinstance(events, list) and not events
                          else "official parser found no visual lifecycle or categorized timing events")
    frame = parse_flat_export(payload)
    if not frame.empty:
        return frame, {
            "format": "flattened-records",
            "version": (direct_value(payload, "version") or "unknown") if isinstance(payload, dict) else "unknown",
            "event_count": len(frame),
        }, build_flat_model(payload, frame, mode, source_info)
    top_keys = sorted(payload.keys()) if isinstance(payload, dict) else ["<top-level array>"]
    details = "; ".join(errors) if errors else "no records had both visual identity and component duration fields"
    raise ValueError(f"Unsupported or empty Performance Analyzer export ({details}). Observed top-level keys: {top_keys}")


def parse_performance_analyzer(payload: Any) -> tuple[pd.DataFrame, dict[str, Any]]:
    frame, metadata, _ = parse_performance_analyzer_with_model(payload)
    return frame, metadata

## 3. Synthetic validation payloads

Two embedded captures validate the parser before a real export is loaded:

- `build_official_sample_payload()` uses only the documented `(component, name)` event types from the export-format specification (User Action, Visual Container Lifecycle, Query, DSE, AS, and DirectQuery events). It includes a possible canvas cache hit and a DirectQuery source query, and it is the demo payload when `USE_SAMPLE_IF_NO_FILE=True`.
- `build_synthetic_payload()` is a **pane-category variant** that mimics the Performance Analyzer pane (`DAX Query`, `Visual display`, `Other`) rather than raw export events. It intentionally includes a high-Other visual so the validation can prove that synchronization time does not displace actionable DAX or rendering work. Its event types are reported as `UNKNOWN_EVENT_TYPE` schema drift, which is expected.

In [ ]:
def build_synthetic_payload() -> dict[str, Any]:
    return {
        "version": "1.0.0",
        "events": [
            {"id": "matrix", "name": "Visual Container Lifecycle", "component": "Visual", "start": "2026-01-01T12:00:00.000Z", "end": "2026-01-01T12:00:06.200Z", "metrics": {"pageName": "Executive", "visualName": "Sales Matrix", "visualId": "visual-matrix", "visualType": "matrix"}},
            {"id": "matrix-dax", "parentId": "matrix", "name": "DAX Query", "component": "DAX", "start": "2026-01-01T12:00:00.100Z", "end": "2026-01-01T12:00:05.500Z", "metrics": {"queryText": "EVALUATE SUMMARIZECOLUMNS('Product'[Category], \"Sales\", [Sales Amount])"}},
            {"id": "matrix-render", "parentId": "matrix", "name": "Visual display", "component": "Render", "start": "2026-01-01T12:00:05.500Z", "end": "2026-01-01T12:00:05.900Z", "metrics": {}},
            {"id": "matrix-other", "parentId": "matrix", "name": "Other", "component": "Other", "start": "2026-01-01T12:00:05.900Z", "end": "2026-01-01T12:00:06.200Z", "metrics": {}},
            {"id": "map", "name": "Visual Container Lifecycle", "component": "Visual", "start": "2026-01-01T12:00:10.000Z", "end": "2026-01-01T12:00:14.200Z", "metrics": {"pageName": "Executive", "visualName": "Customer Map", "visualId": "visual-map", "visualType": "customMap"}},
            {"id": "map-dax", "parentId": "map", "name": "DAX Query", "component": "DAX", "start": "2026-01-01T12:00:10.000Z", "end": "2026-01-01T12:00:10.400Z", "metrics": {"queryText": "EVALUATE TOPN(10000, SUMMARIZECOLUMNS('Customer'[City], \"Sales\", [Sales Amount]))"}},
            {"id": "map-render", "parentId": "map", "name": "Visual display", "component": "Render", "start": "2026-01-01T12:00:10.400Z", "end": "2026-01-01T12:00:13.600Z", "metrics": {}},
            {"id": "map-other", "parentId": "map", "name": "Other", "component": "Other", "start": "2026-01-01T12:00:13.600Z", "end": "2026-01-01T12:00:14.200Z", "metrics": {}},
            {"id": "card", "name": "Visual Container Lifecycle", "component": "Visual", "start": "2026-01-01T12:00:20.000Z", "end": "2026-01-01T12:00:25.500Z", "metrics": {"pageName": "Executive", "visualName": "Revenue Card", "visualId": "visual-card", "visualType": "card"}},
            {"id": "card-dax", "parentId": "card", "name": "DAX Query", "component": "DAX", "start": "2026-01-01T12:00:20.000Z", "end": "2026-01-01T12:00:20.150Z", "metrics": {"queryText": "EVALUATE ROW(\"Revenue\", [Revenue])"}},
            {"id": "card-render", "parentId": "card", "name": "Visual display", "component": "Render", "start": "2026-01-01T12:00:20.150Z", "end": "2026-01-01T12:00:20.250Z", "metrics": {}},
            {"id": "card-other", "parentId": "card", "name": "Other", "component": "Other", "start": "2026-01-01T12:00:20.250Z", "end": "2026-01-01T12:00:25.500Z", "metrics": {}},
        ],
    }


def build_schema_variant_payload() -> dict[str, Any]:
    payload = build_synthetic_payload()
    event_key_map = {
        "id": "EventId",
        "parentId": "ParentEventId",
        "name": "EventName",
        "component": "Category",
        "start": "StartTime",
        "end": "EndTime",
        "metrics": "Properties",
    }
    metric_key_map = {
        "pageName": "Page Name",
        "visualName": "Visual Title",
        "visualId": "Visual ID",
        "visualType": "Visual Type",
        "queryText": "Query Text",
    }
    events = []
    for source_event in payload["events"]:
        variant_event = {}
        for key, value in source_event.items():
            if key == "metrics":
                value = {metric_key_map.get(metric_key, metric_key): metric_value for metric_key, metric_value in value.items()}
            variant_event[event_key_map.get(key, key)] = value
        events.append(variant_event)
    return {"SchemaVersion": payload["version"], "Events": events}


def build_official_sample_payload() -> dict[str, Any]:
    """Synthetic capture that uses only documented (component, name) event types from the export-format specification."""
    def event(event_id, name, component, start, end=None, parent=None, metrics=None):
        record = {"id": event_id, "name": name, "component": component, "start": f"2026-01-01T12:00:{start}Z"}
        if end is not None:
            record["end"] = f"2026-01-01T12:00:{end}Z"
        if parent is not None:
            record["parentId"] = parent
        if metrics is not None:
            record["metrics"] = metrics
        return record

    rc, dse, as_ = "Report Canvas", "DSE", "AS"
    return {"version": "1.0.0", "events": [
        event("ua-1", "User Action", rc, "00.000", metrics={"sourceLabel": "UserAction_Refresh"}),
        event("v-matrix", "Visual Container Lifecycle", rc, "00.010", "03.400", metrics={"status": "finished", "visualTitle": "Sales Matrix", "visualId": "visual-matrix", "visualType": "pivotTable"}),
        event("q-matrix", "Query", rc, "00.020", "03.000", "v-matrix"),
        event("qg-matrix", "Query Generation", rc, "00.020", "00.060", "q-matrix"),
        event("sq-matrix", "Execute Semantic Query", dse, "00.070", "02.950", "q-matrix"),
        event("dax-matrix", "Execute DAX Query", dse, "00.080", "02.900", "sq-matrix", {"QueryText": "EVALUATE SUMMARIZECOLUMNS('Product'[Category], \"Sales\", [Sales Amount])", "RowCount": 12}),
        event("as-matrix", "Execute Query", as_, "00.090", "02.880", "dax-matrix"),
        event("pq-matrix", "Parse Query Result", rc, "02.950", "03.000", "q-matrix"),
        event("r-matrix", "Render", rc, "03.000", "03.380", "v-matrix"),
        event("t-matrix", "Data View Transform", rc, "03.010", "03.090", "r-matrix"),
        event("v-dq", "Visual Container Lifecycle", rc, "00.015", "07.200", metrics={"status": "finished", "visualTitle": "Orders by Region", "visualId": "visual-dq", "visualType": "clusteredBarChart"}),
        event("q-dq", "Query", rc, "00.030", "07.000", "v-dq"),
        event("sq-dq", "Execute Semantic Query", dse, "00.040", "06.950", "q-dq"),
        event("dax-dq", "Execute DAX Query", dse, "00.050", "06.900", "sq-dq", {"QueryText": "EVALUATE SUMMARIZECOLUMNS('Geo'[Region], \"Orders\", [Order Count])", "RowCount": 6}),
        event("as-dq", "Execute Query", as_, "00.060", "06.880", "dax-dq"),
        event("dq-1", "Execute Direct Query", as_, "00.100", "06.300", "as-dq", {"QueryText": "SELECT [Region], COUNT(*) FROM [dbo].[Orders] WHERE [Year] = 2026 GROUP BY [Region]", "ActualQueryDuration": 6050, "DataReadDuration": 90, "RowsRead": 6, "IsGetSourceCapabilitiesQuery": False}),
        event("r-dq", "Render", rc, "07.000", "07.180", "v-dq"),
        event("v-card", "Visual Container Lifecycle", rc, "00.012", "00.300", metrics={"status": "finished", "visualTitle": "Revenue Card", "visualId": "visual-card", "visualType": "card"}),
        event("q-card", "Query", rc, "00.020", "00.040", "v-card"),
        event("r-card", "Render", rc, "00.250", "00.290", "v-card"),
    ]}


synthetic_payload = build_synthetic_payload()
synthetic_frame, synthetic_metadata = parse_performance_analyzer(synthetic_payload)
synthetic_ranked = synthetic_frame.sort_values("actionable_ms", ascending=False).reset_index(drop=True)
variant_frame, variant_metadata = parse_performance_analyzer(build_schema_variant_payload())
assert synthetic_ranked["visual"].tolist() == ["Sales Matrix", "Customer Map", "Revenue Card"]
assert synthetic_ranked.loc[2, "other_ms"] > synthetic_ranked.loc[2, "actionable_ms"]
assert variant_metadata == {"format": "official-event-tree", "version": "1.0.0", "event_count": 12}
assert variant_frame["visual"].tolist() == synthetic_frame["visual"].tolist()
assert variant_frame["actionable_ms"].tolist() == synthetic_frame["actionable_ms"].tolist()
assert to_number("1,250.5 ms") == 1250.5
official_frame, official_metadata, official_model = parse_performance_analyzer_with_model(build_official_sample_payload())
assert official_model["quality_status"] == "Pass", official_model["quality_issues"]
assert official_frame.set_index("visual").loc["Revenue Card", "possible_canvas_cache_hit"]
assert official_frame.set_index("visual").loc["Orders by Region", "native_query_text"].startswith("SELECT")
print(synthetic_metadata)
print(f"Schema variant parsed: {len(variant_frame)} visuals | Official-catalog sample: {len(official_frame)} visuals, quality {official_model['quality_status']}")
display(synthetic_ranked[["page", "visual", "total_ms", "dax_ms", "render_ms", "other_ms", "actionable_ms"]])

## 4. Calculate metrics, detect bottlenecks, and build the recommendation catalog

Severity combines configurable absolute thresholds with a 90th-percentile outlier flag. Root cause is a **likely cause**, not proof: query text heuristics are evidence for the next diagnostic, while DAX Studio, source/gateway telemetry, and capacity monitoring establish the actual engine or infrastructure cause.

In [ ]:
SOURCE_URLS = {
    "performance_analyzer": "https://learn.microsoft.com/power-bi/create-reports/performance-analyzer",
    "optimization": "https://learn.microsoft.com/power-bi/guidance/power-bi-optimization",
    "monitoring": "https://learn.microsoft.com/power-bi/guidance/monitor-report-performance",
    "troubleshooting": "https://learn.microsoft.com/power-bi/guidance/report-performance-troubleshoot",
    "sqlbi": "https://www.sqlbi.com/articles/introducing-the-power-bi-performance-analyzer/",
    "bpa": "https://community.fabric.microsoft.com/blog/fbc_pbiupdatesblog/best-practice-rules-to-improve-your-models-performance/5175648",
    "report_analyzer": "https://github.com/m-kovalsky/ReportAnalyzer",
    "dax_decision_guide": "https://github.com/microsoft/skills-for-fabric/blob/main/skills/semantic-model-authoring/references/dax-perf-decision-guide.md",
    "dax_pattern_catalog": "https://github.com/microsoft/skills-for-fabric/blob/main/skills/semantic-model-authoring/references/dax-perf-patterns.md",
}

RULES_CATALOG = {
    "DQ": {
        "cause": "DirectQuery/data source",
        "recommendation": "Tune the source query and indexes; preserve query folding; review gateway/network locality; consider aggregations, Dual, or Import where requirements allow.",
        "verification": "Capture the translated SQL/KQL and source duration; compare source execution, gateway telemetry, and a controlled Import/aggregation test.",
        "source": SOURCE_URLS["optimization"],
    },
    "DAX": {
        "cause": "DAX/semantic model",
        "recommendation": "Resolve referenced measure/UDF definitions, then use the routed DAX pattern candidates below as hypotheses; do not rewrite only from generated query text.",
        "verification": "Capture DAX Studio or modeling-MCP Server Timings and Query Plan: FE/SE duration, SE query count, callbacks, materialized rows, peak memory, and result rows. Retest semantic equivalence and duration under controlled cache conditions.",
        "source": SOURCE_URLS["dax_decision_guide"],
    },
    "RENDER": {
        "cause": "Visual rendering",
        "recommendation": "Reduce rows/data points with restrictive filters; compare with a native visual; move detail to drillthrough/tooltips; review custom visuals, web images, geocoding, and conditional formatting.",
        "verification": "Clone the page, replace or simplify this visual, refresh it individually, and compare Visual display duration.",
        "source": SOURCE_URLS["optimization"],
    },
    "PARAM": {
        "cause": "Evaluated parameters",
        "recommendation": "Simplify field-parameter choices and dependent measures; remove unused parameter combinations and reduce downstream visual work.",
        "verification": "Test fixed fields versus the parameterized visual and compare Evaluated parameters plus DAX duration.",
        "source": SOURCE_URLS["performance_analyzer"],
    },
    "MIXED": {
        "cause": "Mixed bottleneck",
        "recommendation": "Address the largest actionable component first, retest, then continue one variable at a time; avoid changing model and visual design simultaneously.",
        "verification": "Use individual-visual refreshes and compare component durations after each isolated change.",
        "source": SOURCE_URLS["performance_analyzer"],
    },
    "OTHER": {
        "cause": "Synchronization/page contention (confirmation needed)",
        "recommendation": "Repeat a warm Refresh visuals capture, optimize slow sibling visuals, and reduce visible visuals or unnecessary interactions on the page.",
        "verification": "Compare repeated warm captures and refresh this visual alone; do not tune its DAX solely because Other is high.",
        "source": SOURCE_URLS["sqlbi"],
    },
    "OK": {
        "cause": "No material bottleneck",
        "recommendation": "No immediate visual-level remediation; retain as a baseline and focus on higher-ranked visuals.",
        "verification": "Retest under the same filters, device, and cache conditions after report changes.",
        "source": SOURCE_URLS["monitoring"],
    },
}

DAX_PATTERNS = {
    "Iterator functions": r"\b(SUMX|AVERAGEX|MINX|MAXX|RANKX|PRODUCTX)\s*\(",
    "Broad table FILTER": r"\bFILTER\s*\(\s*'?[^\[\],]+?'?\s*,",
    "High-cardinality candidate": r"\b(DISTINCTCOUNT|VALUES|CROSSJOIN|GENERATE)\s*\(",
    "Large result shaping": r"\b(TOPN|SUMMARIZE|SUMMARIZECOLUMNS|ADDCOLUMNS)\s*\(",
}

DAX_PERF_PATTERN_CATALOG = [
    {
        "pattern_id": "DAX001",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\bFILTER\s*\(\s*'?[^\[\],]+?'?\s*,"],
        "issue": "Table-scoped FILTER can force iterator work and broad materialization.",
        "action": "Test simple column predicates as separate CALCULATE arguments; use KEEPFILTERS when intersection semantics are required.",
        "evidence": "Confirm reduced FE time/materialized rows and equivalent values in Server Timings and Query Plan.",
        "match_reason": "FILTER(Table, predicate) text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX002",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\b(?:ADDCOLUMNS|SUMMARIZE)\s*\("],
        "issue": "Grouped extension patterns can create Formula Engine work or callbacks.",
        "action": "Compare with an equivalent SUMMARIZECOLUMNS grouping and calculation shape.",
        "evidence": "Check callback events, FE duration, SE rows, and semantic equivalence.",
        "match_reason": "ADDCOLUMNS or SUMMARIZE text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX006/DAX008",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\b(?:SUMX|AVERAGEX|MINX|MAXX)\s*\(", r"\bVALUES\s*\(", r"\bCALCULATE\s*\("],
        "issue": "An iterator over VALUES with per-row CALCULATE can repeat context transitions.",
        "action": "Test precomputing the iterator grain and measure value with SUMMARIZECOLUMNS, then iterate the materialized result.",
        "evidence": "Look for high FE time and many short or repeated SE events; compare result rows and values.",
        "match_reason": "Iterator + VALUES + CALCULATE text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX007",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\b(?:SUMX|AVERAGEX|MINX|MAXX)\s*\(", r"\b(?:IF|SWITCH)\s*\("],
        "issue": "Row-by-row boolean branching inside an iterator can trigger Formula Engine callbacks.",
        "action": "Test a native predicate/COUNTROWS or INT(boolean) form when semantics permit.",
        "evidence": "Confirm CallbackDataID/EncodeCallback on the slow scan and compare values after rewriting.",
        "match_reason": "Iterator + IF/SWITCH text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX011/DAX014",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\bDISTINCTCOUNT\s*\("],
        "issue": "DISTINCTCOUNT performance depends on filter shape and whether the column is a recognized key.",
        "action": "Benchmark alternatives only after checking key metadata: COUNTROWS for a recognized key, or SUMX(DISTINCT(), 1) as a measured alternative.",
        "evidence": "Inspect DCOUNT/xmSQL, key and relationship metadata, FE/SE split, and semantic equivalence.",
        "match_reason": "DISTINCTCOUNT text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX012",
        "tier": "Tier 1 - measure/UDF",
        "any": [r"\bALLEXCEPT\s*\(", r"\bREMOVEFILTERS\s*\([\s\S]{0,300}\bVALUES\s*\("],
        "issue": "Filter-preservation forms can behave and perform differently under direct versus cross-filtering.",
        "action": "Choose ALLEXCEPT or REMOVEFILTERS/ALL + VALUES deliberately for the required filter semantics.",
        "evidence": "Validate across representative grouping/filter contexts and compare query plans.",
        "match_reason": "ALLEXCEPT or REMOVEFILTERS + VALUES text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX013",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\bDEFINE\b[\s\S]*\bMEASURE\b", r"\bSWITCH\s*\("],
        "issue": "Branch measures can block Storage Engine fusion or evaluate expensive branches.",
        "action": "Keep selector filtering direct, branch types consistent, and branch calculations Storage Engine friendly.",
        "evidence": "Check FE time, unused branch work, repeated fact scans, and fusion in trace/query plan.",
        "match_reason": "SWITCH in a captured DEFINE MEASURE block",
        "confidence": "Medium",
    },
    {
        "pattern_id": "DAX016",
        "tier": "Tier 1 - measure/UDF",
        "any": [r"\bCROSSFILTER\s*\(", r"\bUSERELATIONSHIP\s*\("],
        "issue": "Relationship overrides may expose expensive or ambiguous filter paths.",
        "action": "Test alternate propagation locally in the measure before proposing a model relationship change.",
        "evidence": "Inspect joins in xmSQL plus active/inactive, bidirectional, and many-to-many relationship metadata.",
        "match_reason": "CROSSFILTER or USERELATIONSHIP text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX018",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\b(?:SUMX|AVERAGEX|MINX|MAXX)\s*\(", r"\bDIVIDE\s*\("],
        "issue": "DIVIDE inside an iterator can cause row-by-row Formula Engine callbacks.",
        "action": "When the denominator is proven non-zero, test `/`; otherwise pre-filter zero denominators before division.",
        "evidence": "Confirm callback text on the slow scan and validate zero/blank semantics before keeping the change.",
        "match_reason": "Iterator + DIVIDE text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "DAX019/DAX020",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\bDEFINE\b[\s\S]*\bMEASURE\b", r"\b(?:DATESYTD|DATESMTD|DATESQTD|DATEADD|SAMEPERIODLASTYEAR|PARALLELPERIOD)\s*\("],
        "issue": "Time-window logic repeated across sibling measures can block Storage Engine fusion.",
        "action": "Keep base or slice measures filter-free and test applying the time window once in the combining measure.",
        "evidence": "Look for repeated fact scans with similar joins and verify fusion plus semantic equivalence.",
        "match_reason": "Time-intelligence function in a captured DEFINE MEASURE block",
        "confidence": "Medium",
    },
    {
        "pattern_id": "DAX021",
        "tier": "Tier 1 - measure/UDF",
        "required": [r"\bTREATAS\s*\(", r"\b(?:SELECTCOLUMNS|FILTER)\s*\("],
        "issue": "A computed key set pushed back through TREATAS can create large semi-join predicates.",
        "action": "Test precomputing both aggregations at a shared key grain and joining in the Formula Engine.",
        "evidence": "Confirm large IN/INB/ININDEX or compound tuple predicates and compare materialized rows.",
        "match_reason": "TREATAS + computed table text shape",
        "confidence": "Low",
    },
    {
        "pattern_id": "QRY002",
        "tier": "Tier 2 - query structure",
        "required": [r"__ValueFilterDM"],
        "issue": "A visual-level measure filter can evaluate a measure once for filtering and again for display.",
        "action": "With report-author approval, test moving the threshold into a measure that returns BLANK below the cutoff.",
        "evidence": "Compare result shape, values, and duration; this changes report/query behavior and requires approval.",
        "match_reason": "__ValueFilterDM generated-query marker",
        "confidence": "Medium",
    },
    {
        "pattern_id": "QRY003",
        "tier": "Tier 2 - query structure",
        "required": [r"\bSUMMARIZECOLUMNS\s*\(", r"'[^']*(?:Date|Calendar)[^']*'\s*\[\s*(?:Date|DateTime|Timestamp)\s*\]"],
        "issue": "A daily or timestamp grouping may request more rows than the visual needs.",
        "action": "With report-author approval, test a coarser axis/grouping such as YearMonth while preserving the intended business result.",
        "evidence": "Confirm result-row and SE-row reduction, then validate changed grain and values with the report owner.",
        "match_reason": "SUMMARIZECOLUMNS with a date/timestamp grouping candidate",
        "confidence": "Low",
    },
    {
        "pattern_id": "QRY004",
        "tier": "Tier 2 - query structure",
        "required": [r"\bDEFINE\b[\s\S]*\bMEASURE\b"],
        "any": [r"\bCOALESCE\s*\([^\)]*,\s*0\s*\)", r"\bIF\s*\(\s*ISBLANK\s*\([^\)]*\)\s*,\s*0", r"\+\s*0(?:\s|$)"],
        "issue": "BLANK suppression can force evaluation of otherwise empty group combinations.",
        "action": "With report-author approval, test preserving BLANK or adding zero only where the output contract requires it.",
        "evidence": "Compare row count, output shape, values, and duration because removing zero-fill changes results.",
        "match_reason": "Zero/BLANK suppression in a captured DEFINE MEASURE block",
        "confidence": "Medium",
    },
]


def inspect_dax(query: str) -> str:
    if not query.strip():
        return "No captured DAX query"
    matches = [label for label, pattern in DAX_PATTERNS.items() if re.search(pattern, query, flags=re.IGNORECASE)]
    return "; ".join(matches) if matches else "No selected text heuristic matched"


def detect_dax_performance_patterns(query: str) -> list[dict[str, Any]]:
    if not query.strip():
        return []
    matches: list[dict[str, Any]] = []
    for candidate in DAX_PERF_PATTERN_CATALOG:
        required = candidate.get("required", [])
        alternatives = candidate.get("any", [])
        required_match = all(re.search(pattern, query, flags=re.IGNORECASE) for pattern in required)
        alternative_match = not alternatives or any(re.search(pattern, query, flags=re.IGNORECASE) for pattern in alternatives)
        if required_match and alternative_match:
            matches.append({
                "pattern_id": candidate["pattern_id"],
                "tier": candidate["tier"],
                "issue": candidate["issue"],
                "action": candidate["action"],
                "evidence_needed": candidate["evidence"],
                "match_reason": candidate["match_reason"],
                "text_confidence": candidate["confidence"],
                "approval_required": not candidate["tier"].startswith("Tier 1"),
                "source_url": SOURCE_URLS["dax_pattern_catalog"],
            })
    return matches


def dax_measure_scope(query: str) -> str:
    if not query.strip():
        return "No captured query"
    if re.search(r"\bDEFINE\b[\s\S]*\bMEASURE\b", query, flags=re.IGNORECASE):
        return "Measure definitions visible in captured query"
    return "Generated query only - resolve referenced measure/UDF definitions before tuning"


def dax_pattern_next_step(query: str, pattern_ids: str) -> str:
    scope = dax_measure_scope(query)
    if pattern_ids:
        return f"{scope}. Capture Server Timings and Query Plan to validate candidate(s) {pattern_ids}; keep only changes that improve duration beyond run noise and preserve identical results."
    return f"{scope}. Isolate slow measures, then capture FE/SE split, callbacks, SE query count, materialized rows, peak memory, and result rows before choosing a rewrite."


def query_signature(query: str) -> str:
    normalized = re.sub(r"\s+", " ", query.strip()).lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:12] if normalized else ""


def choose_rule(row: pd.Series) -> str:
    actionable = float(row["actionable_ms"])
    if row["direct_query_ms"] >= SLOW_DIRECT_QUERY_MS and row["direct_query_ms"] >= row["render_ms"] and row["direct_query_ms"] >= 0.50 * max(actionable, 1):
        return "DQ"
    if row["dax_ms"] >= SLOW_DAX_MS and row["dax_ms"] >= row["render_ms"] and row["dax_ms"] >= 0.50 * max(actionable, 1):
        return "DAX"
    if row["render_ms"] >= SLOW_RENDER_MS and row["render_ms"] >= row["query_ms"] and row["render_ms"] >= 0.50 * max(actionable, 1):
        return "RENDER"
    if row["parameter_ms"] >= SLOW_RENDER_MS and row["parameter_ms"] >= 0.50 * max(actionable, 1):
        return "PARAM"
    if actionable >= REVIEW_VISUAL_MS:
        return "MIXED"
    if row["other_ms"] > actionable and row["total_ms"] >= REVIEW_VISUAL_MS:
        return "OTHER"
    return "OK"


def severity_for(row: pd.Series) -> str:
    actionable = float(row["actionable_ms"])
    if actionable >= CRITICAL_VISUAL_MS:
        return "Critical"
    if actionable >= SLOW_VISUAL_MS or row["dax_ms"] >= SLOW_DAX_MS or row["direct_query_ms"] >= SLOW_DIRECT_QUERY_MS:
        return "Slow"
    if actionable >= REVIEW_VISUAL_MS or row["render_ms"] >= SLOW_RENDER_MS or row["other_ms"] > actionable:
        return "Review"
    return "Good"


def confidence_for(row: pd.Series) -> str:
    actionable = float(row["actionable_ms"])
    if bool(row.get("is_truncated", False)):
        return "Low"
    if row["rule_id"] == "OTHER":
        return "Low"
    if actionable <= 0:
        return "Low"
    dominant_share = max(row["query_ms"], row["render_ms"], row["parameter_ms"]) / actionable
    return "High" if dominant_share >= 0.65 else "Medium"


def analyze_visuals(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        raise ValueError("No visual timing records were parsed. Confirm that the export contains recorded visual interactions.")
    results = apply_visual_defaults(frame)
    numeric_columns = ["total_ms", "dax_ms", "direct_query_ms", "render_ms", "parameter_ms", "other_ms", "query_ms", "actionable_ms"]
    results[numeric_columns] = results[numeric_columns].apply(pd.to_numeric, errors="coerce").fillna(0).clip(lower=0)
    results["query_pct"] = np.where(results["actionable_ms"] > 0, results["query_ms"] / results["actionable_ms"], 0)
    results["render_pct"] = np.where(results["actionable_ms"] > 0, results["render_ms"] / results["actionable_ms"], 0)
    results["other_pct_of_total"] = np.where(results["total_ms"] > 0, results["other_ms"] / results["total_ms"], 0)
    actionable_total = float(results["actionable_ms"].sum())
    results["contribution_pct"] = np.where(actionable_total > 0, results["actionable_ms"] / actionable_total, 0)
    outlier_threshold = float(results["actionable_ms"].quantile(0.90)) if len(results) >= 5 else float(results["actionable_ms"].max())
    results["is_p90_outlier"] = results["actionable_ms"] >= outlier_threshold
    results["dax_evidence"] = results["dax_query"].map(inspect_dax)
    pattern_matches = results["dax_query"].map(detect_dax_performance_patterns)
    results["dax_pattern_count"] = pattern_matches.map(len)
    results["dax_pattern_ids"] = pattern_matches.map(lambda matches: "; ".join(match["pattern_id"] for match in matches))
    results["dax_pattern_tiers"] = pattern_matches.map(lambda matches: "; ".join(dict.fromkeys(match["tier"] for match in matches)))
    results["dax_measure_scope"] = results["dax_query"].map(dax_measure_scope)
    results["dax_pattern_next_step"] = [
        dax_pattern_next_step(query, pattern_ids)
        for query, pattern_ids in zip(results["dax_query"], results["dax_pattern_ids"])
    ]
    results["query_signature"] = results["dax_query"].map(query_signature)
    signature_counts = results.loc[results["query_signature"] != "", "query_signature"].value_counts()
    results["repeated_query_count"] = results["query_signature"].map(signature_counts).fillna(0).astype(int)
    page_counts = results["page"].value_counts()
    results["page_visual_count"] = results["page"].map(page_counts).astype(int)
    results["page_over_limit"] = results["page_visual_count"] > MAX_VISUALS_PER_PAGE
    results["rule_id"] = results.apply(choose_rule, axis=1)
    results["severity"] = results.apply(severity_for, axis=1)
    results["root_cause"] = results["rule_id"].map(lambda key: RULES_CATALOG[key]["cause"])
    results["confidence"] = results.apply(confidence_for, axis=1)
    results["recommendation"] = results["rule_id"].map(lambda key: RULES_CATALOG[key]["recommendation"])
    has_candidates = results["dax_pattern_ids"].ne("")
    results.loc[has_candidates, "recommendation"] += " Candidate patterns: " + results.loc[has_candidates, "dax_pattern_ids"] + ". Treat text matches as hypotheses until trace evidence confirms them."
    results.loc[results["page_over_limit"], "recommendation"] += " This page exceeds the configured visible-visual limit; remove, defer, or move secondary visuals."
    results.loc[results["repeated_query_count"] > 1, "recommendation"] += " The same query signature repeats; check duplicate visuals/interactions and shared-query opportunities."
    results.loc[results["is_truncated"], "recommendation"] += " Trace detail is truncated for this visual (Metrics Truncated); treat the component split as incomplete and re-capture before tuning."
    results.loc[results["dax_error_count"] > 0, "recommendation"] += " A DAX query for this visual reported Error=true; resolve the failure before interpreting its timing."
    results["next_diagnostic"] = results["rule_id"].map(lambda key: RULES_CATALOG[key]["verification"])
    results.loc[results["dax_query"].str.strip().ne(""), "next_diagnostic"] += " " + results.loc[results["dax_query"].str.strip().ne(""), "dax_pattern_next_step"]
    results["source_url"] = results["rule_id"].map(lambda key: RULES_CATALOG[key]["source"])
    severity_order = pd.CategoricalDtype(["Critical", "Slow", "Review", "Good"], ordered=True)
    results["severity"] = results["severity"].astype(severity_order)
    results = results.sort_values(["severity", "actionable_ms"], ascending=[True, False]).reset_index(drop=True)
    results.insert(0, "rank", np.arange(1, len(results) + 1))

    page_summary = results.groupby("page", as_index=False).agg(
        visual_count=("visual_id", "count"),
        total_actionable_ms=("actionable_ms", "sum"),
        max_actionable_ms=("actionable_ms", "max"),
        median_actionable_ms=("actionable_ms", "median"),
        slow_visual_count=("severity", lambda values: int(values.isin(["Critical", "Slow"]).sum())),
        total_other_ms=("other_ms", "sum"),
        dominant_root_cause=("root_cause", lambda values: values.mode().iloc[0] if not values.mode().empty else "Mixed"),
    )
    page_summary["over_visual_limit"] = page_summary["visual_count"] > MAX_VISUALS_PER_PAGE
    page_summary = page_summary.sort_values("total_actionable_ms", ascending=False).reset_index(drop=True)
    return results, page_summary

In [ ]:
def sanitize_dax_for_pattern_matching(query: str) -> str:
    sanitized = list(query)
    index = 0
    length = len(query)
    while index < length:
        if query[index] == "'":
            index += 1
            while index < length:
                if query[index] == "'" and index + 1 < length and query[index + 1] == "'":
                    index += 2
                elif query[index] == "'":
                    index += 1
                    break
                else:
                    index += 1
        elif query[index] == '"':
            index += 1
            while index < length:
                if query[index] in "\r\n":
                    index += 1
                elif query[index] == '"' and index + 1 < length and query[index + 1] == '"':
                    sanitized[index] = sanitized[index + 1] = " "
                    index += 2
                elif query[index] == '"':
                    index += 1
                    break
                else:
                    sanitized[index] = " "
                    index += 1
        elif query.startswith("/*", index):
            sanitized[index] = sanitized[index + 1] = " "
            index += 2
            while index < length and not query.startswith("*/", index):
                if query[index] not in "\r\n":
                    sanitized[index] = " "
                index += 1
            if index < length:
                sanitized[index] = sanitized[index + 1] = " "
                index += 2
        elif query.startswith("//", index) or query.startswith("--", index):
            sanitized[index] = sanitized[index + 1] = " "
            index += 2
            while index < length and query[index] not in "\r\n":
                sanitized[index] = " "
                index += 1
        else:
            index += 1
    return "".join(sanitized)


def extract_visible_measure_bodies(query: str) -> list[str]:
    sanitized = sanitize_dax_for_pattern_matching(query)
    declaration_pattern = re.compile(r"^([ \t]*)(MEASURE|VAR|TABLE|COLUMN|EVALUATE)\b", re.IGNORECASE)
    measure_pattern = re.compile(r"^([ \t]*)MEASURE\s+'(?:[^']|'')+'\s*\[[^\]\r\n]+\]\s*=\s*(.*)$", re.IGNORECASE)
    bodies: list[str] = []
    active_body: list[str] | None = None
    active_indent = 0
    in_define = False
    for line in sanitized.splitlines(keepends=True):
        logical_line = line.rstrip("\r\n")
        if not in_define:
            if re.match(r"^[ \t]*DEFINE\b", logical_line, flags=re.IGNORECASE):
                in_define = True
            continue
        declaration = declaration_pattern.match(logical_line)
        declaration_indent = len(declaration.group(1).expandtabs(4)) if declaration else -1
        if active_body is not None and declaration and declaration_indent <= active_indent:
            body = "".join(active_body).strip()
            if body:
                bodies.append(body)
            active_body = None
        if declaration and declaration.group(2).upper() == "EVALUATE" and active_body is None:
            break
        if active_body is None and declaration and declaration.group(2).upper() == "MEASURE":
            measure = measure_pattern.match(logical_line)
            if measure:
                active_indent = len(measure.group(1).expandtabs(4))
                active_body = [measure.group(2), line[len(logical_line):]]
            continue
        if active_body is not None:
            active_body.append(line)
    if active_body is not None:
        body = "".join(active_body).strip()
        if body:
            bodies.append(body)
    return bodies


def extract_call_bodies(expression: str, function_names: tuple[str, ...]) -> list[str]:
    if not function_names:
        return []
    names = "|".join(re.escape(name) for name in sorted(function_names, key=len, reverse=True))
    call_pattern = re.compile(rf"(?<![A-Z0-9_])(?:{names})\s*\(", re.IGNORECASE)
    bodies: list[str] = []
    for match in call_pattern.finditer(expression):
        opening = match.end() - 1
        depth = 1
        index = opening + 1
        while index < len(expression):
            if expression[index] in "'\"":
                quote = expression[index]
                index += 1
                while index < len(expression):
                    if expression[index] == quote and index + 1 < len(expression) and expression[index + 1] == quote:
                        index += 2
                    elif expression[index] == quote:
                        index += 1
                        break
                    else:
                        index += 1
            elif expression[index] == "(":
                depth += 1
                index += 1
            elif expression[index] == ")":
                depth -= 1
                if depth == 0:
                    bodies.append(expression[opening + 1:index])
                    break
                index += 1
            else:
                index += 1
    return bodies


def _catalog_candidate_matches(candidate: dict[str, Any], text: str) -> bool:
    pattern_id = candidate["pattern_id"]
    if pattern_id == "DAX006/DAX008":
        iterator_bodies = extract_call_bodies(text, ("SUMX", "AVERAGEX", "MINX", "MAXX"))
        return any(re.search(r"\bVALUES\s*\(", body, flags=re.IGNORECASE) and re.search(r"\bCALCULATE\s*\(", body, flags=re.IGNORECASE) for body in iterator_bodies)
    if pattern_id == "DAX007":
        iterator_bodies = extract_call_bodies(text, ("SUMX", "AVERAGEX", "MINX", "MAXX"))
        return any(re.search(r"\b(?:IF|SWITCH)\s*\(", body, flags=re.IGNORECASE) for body in iterator_bodies)
    if pattern_id == "DAX018":
        iterator_bodies = extract_call_bodies(text, ("SUMX", "AVERAGEX", "MINX", "MAXX"))
        return any(re.search(r"\bDIVIDE\s*\(", body, flags=re.IGNORECASE) for body in iterator_bodies)
    if pattern_id == "DAX021":
        treatas_bodies = extract_call_bodies(text, ("TREATAS",))
        return any(re.search(r"\b(?:SELECTCOLUMNS|FILTER)\s*\(", body, flags=re.IGNORECASE) for body in treatas_bodies)
    required = candidate.get("required", [])
    if pattern_id in {"DAX013", "DAX019/DAX020", "QRY004"}:
        required = [pattern for pattern in required if not ("DEFINE" in pattern.upper() and "MEASURE" in pattern.upper())]
    alternatives = candidate.get("any", [])
    return all(re.search(pattern, text, flags=re.IGNORECASE) for pattern in required) and (not alternatives or any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in alternatives))


def detect_dax_performance_patterns(query: str) -> list[dict[str, Any]]:
    if not query.strip():
        return []
    sanitized = sanitize_dax_for_pattern_matching(query)
    measure_bodies = extract_visible_measure_bodies(sanitized)
    matches: list[dict[str, Any]] = []
    seen: set[str] = set()
    for candidate in DAX_PERF_PATTERN_CATALOG:
        pattern_id = candidate["pattern_id"]
        tier_one = candidate["tier"].startswith("Tier 1")
        if tier_one or pattern_id == "QRY004":
            matched = any(_catalog_candidate_matches(candidate, body) for body in measure_bodies)
        else:
            matched = _catalog_candidate_matches(candidate, sanitized)
        if matched and pattern_id not in seen:
            seen.add(pattern_id)
            matches.append({
                "pattern_id": pattern_id,
                "tier": candidate["tier"],
                "issue": candidate["issue"],
                "action": candidate["action"],
                "evidence_needed": candidate["evidence"],
                "match_reason": candidate["match_reason"],
                "text_confidence": candidate["confidence"],
                "approval_required": not tier_one,
                "source_url": SOURCE_URLS["dax_pattern_catalog"],
            })
    return matches


def dax_measure_scope(query: str) -> str:
    if not query.strip():
        return "No captured query"
    visible_count = len(extract_visible_measure_bodies(query))
    if visible_count:
        return f"{visible_count} visible measure definition(s); referenced dependencies may still require resolution"
    return "Generated query only - resolve referenced measure/UDF definitions before tuning"


def dax_pattern_next_step(query, pattern_ids) -> str:
    scope = dax_measure_scope(query)
    candidate_text = str(pattern_ids or "")
    if candidate_text:
        approval = " Any QRY* candidate requires report-author approval before changing query/report behavior." if re.search(r"\bQRY\d+\b", candidate_text, flags=re.IGNORECASE) else ""
        return f"{scope}. Capture controlled Server Timings and Query Plan to validate candidate(s) {candidate_text}; keep only changes that improve duration beyond run noise and preserve semantic equivalence.{approval}"
    return f"{scope}. Isolate slow measures, then capture controlled Server Timings and Query Plan with FE/SE split, callbacks, SE query count, materialized rows, peak memory, and result rows before choosing a rewrite; require semantic-equivalence validation for every change."

In [ ]:
# QA policy: keep relative-outlier, synchronization, and page-identity semantics aligned.
RULES_CATALOG["OUTLIER"] = {
    "cause": "Relative actionable-duration outlier",
    "recommendation": "Compare this visual with peers under the same interaction and cache state; investigate the dominant actionable component before changing the model or report.",
    "verification": "Repeat the same interaction under controlled conditions and confirm that the visual remains above the capture's 90th percentile.",
    "source": SOURCE_URLS["monitoring"],
}

# Direct Lake awareness: driven by CAPTURE_METADATA["storage_mode"] because the export does not record storage mode.
SOURCE_URLS["direct_lake_analyze"] = "https://learn.microsoft.com/fabric/fundamentals/direct-lake-analyze-query-processing"
SOURCE_URLS["direct_lake_how_it_works"] = "https://learn.microsoft.com/fabric/fundamentals/direct-lake-how-it-works"
RULES_CATALOG["DL_FALLBACK"] = {
    "cause": "Direct Lake fallback to DirectQuery",
    "recommendation": "Find why the query fell back to DirectQuery: SQL views or row-/object-level security defined on the SQL analytics endpoint, tables above Direct Lake guardrails (rows, Parquet files, row groups; run OPTIMIZE with V-Order), or capacity memory pressure. Remove the cause, or set DirectLakeBehavior to DirectLakeOnly in a test copy to confirm (queries then fail instead of falling back).",
    "verification": "Confirm fallback with DirectQuery_Begin/End events in a SQL Server Profiler or workspace monitoring trace (ignore EngineEdition and object-level security checks), then retest after removing the cause and compare DAX and Direct query time.",
    "source": SOURCE_URLS["direct_lake_analyze"],
}
DIRECT_LAKE_ONELAKE_NOTE = (
    "Direct Lake on OneLake does not fall back to DirectQuery, so these DirectQuery events suggest the table is not served by Direct Lake on OneLake "
    "(for example, a composite model with DirectQuery tables); verify the storage mode of the tables involved. "
)
DIRECT_LAKE_COLD_CACHE_NOTE = (
    " Direct Lake loads (transcodes) columns into memory on first use after a refresh, framing, or eviction; unless this capture was warm, DAX time can be "
    "inflated by column loading. Repeat a warm Refresh visuals capture before tuning DAX."
)


def storage_mode_profile() -> dict[str, Any]:
    metadata = globals().get("capture_metadata") or CAPTURE_METADATA
    raw = str(metadata.get("storage_mode", "") or "").strip()
    key = normalize_key(raw)
    is_direct_lake = "directlake" in key
    variant = "onelake" if is_direct_lake and "onelake" in key else "sql" if is_direct_lake and "sql" in key else "unknown" if is_direct_lake else ""
    cache_state = normalize_key(metadata.get("cache_state", "") or "")
    label = raw or "not recorded"
    return {"storage_mode": label, "is_direct_lake": is_direct_lake, "direct_lake_variant": variant, "cache_is_warm": cache_state == "warm"}


def apply_storage_mode_guidance(results: pd.DataFrame) -> pd.DataFrame:
    """Rewrite DirectQuery guidance as Direct Lake fallback guidance and add cold-cache notes when storage mode is Direct Lake."""
    profile = storage_mode_profile()
    results = results.copy()
    results["storage_mode"] = profile["storage_mode"]
    results["direct_lake_fallback_ms"] = np.where(profile["is_direct_lake"], results["direct_query_ms"], 0.0)
    if not profile["is_direct_lake"]:
        return results
    fallback = results["direct_query_ms"] > 0
    onelake_prefix = DIRECT_LAKE_ONELAKE_NOTE if profile["direct_lake_variant"] == "onelake" else ""
    rule_is_fallback = results["rule_id"].eq("DL_FALLBACK")
    if onelake_prefix and rule_is_fallback.any():
        results.loc[rule_is_fallback, "recommendation"] = onelake_prefix + results.loc[rule_is_fallback, "recommendation"].astype(str)
    minor_fallback = fallback & ~rule_is_fallback
    if minor_fallback.any():
        results.loc[minor_fallback, "recommendation"] = [
            f"{recommendation} Direct Lake fallback to DirectQuery was observed for this visual ({milliseconds:,.0f} ms); "
            + ("verify the table storage mode." if onelake_prefix else "investigate the fallback reason (SQL views, SQL-endpoint security, guardrails, memory).")
            for recommendation, milliseconds in zip(results.loc[minor_fallback, "recommendation"], results.loc[minor_fallback, "direct_query_ms"])
        ]
    if not profile["cache_is_warm"]:
        dax_led = results["rule_id"].isin(["DAX", "MIXED", "OUTLIER"]) & (results["dax_ms"] > 0)
        if dax_led.any():
            results.loc[dax_led, "next_diagnostic"] = results.loc[dax_led, "next_diagnostic"].astype(str) + DIRECT_LAKE_COLD_CACHE_NOTE
    return results


def is_relative_outlier(row: pd.Series) -> bool:
    value: Any = row.get("is_p90_outlier", False)
    return bool(value) if isinstance(value, (bool, np.bool_)) else False


def choose_rule(row: pd.Series) -> str:
    actionable = float(row["actionable_ms"])
    if row["direct_query_ms"] >= SLOW_DIRECT_QUERY_MS and row["direct_query_ms"] >= row["render_ms"] and row["direct_query_ms"] >= 0.50 * max(actionable, 1):
        return "DL_FALLBACK" if storage_mode_profile()["is_direct_lake"] else "DQ"
    if row["dax_ms"] >= SLOW_DAX_MS and row["dax_ms"] >= row["render_ms"] and row["dax_ms"] >= 0.50 * max(actionable, 1):
        return "DAX"
    if row["render_ms"] >= SLOW_RENDER_MS and row["render_ms"] >= row["query_ms"] and row["render_ms"] >= 0.50 * max(actionable, 1):
        return "RENDER"
    if row["parameter_ms"] >= SLOW_RENDER_MS and row["parameter_ms"] >= 0.50 * max(actionable, 1):
        return "PARAM"
    if actionable >= REVIEW_VISUAL_MS:
        return "MIXED"
    if row["other_ms"] > actionable and row["total_ms"] >= REVIEW_VISUAL_MS:
        return "OTHER"
    if actionable > 0 and is_relative_outlier(row):
        return "OUTLIER"
    return "OK"


def severity_for(row: pd.Series) -> str:
    actionable = float(row["actionable_ms"])
    if actionable >= CRITICAL_VISUAL_MS:
        return "Critical"
    if actionable >= SLOW_VISUAL_MS or row["dax_ms"] >= SLOW_DAX_MS or row["direct_query_ms"] >= SLOW_DIRECT_QUERY_MS:
        return "Slow"
    if (
        actionable >= REVIEW_VISUAL_MS
        or row["render_ms"] >= SLOW_RENDER_MS
        or (row["other_ms"] > actionable and row["total_ms"] >= REVIEW_VISUAL_MS)
        or (actionable > 0 and is_relative_outlier(row))
    ):
        return "Review"
    return "Good"


def explicit_page_identity(page_names: pd.Series) -> pd.Series:
    normalized = page_names.fillna("").astype(str).str.strip()
    return (
        normalized.ne("")
        & normalized.ne("Unknown page")
        & normalized.ne("Page unavailable in export")
        & ~normalized.str.endswith("(name unavailable)")
    )


def reportable_page_group(page_names: pd.Series) -> pd.Series:
    normalized = page_names.fillna("").astype(str).str.strip()
    inferred_transition = normalized.str.match(
        r"^Page transition \d+ \(name unavailable\)$", na=False
    )
    return explicit_page_identity(page_names) | inferred_transition


def apply_page_identity_guard(
    results: pd.DataFrame, page_summary: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    results = results.copy()
    page_summary = page_summary.copy()
    known_page = reportable_page_group(results["page"])
    page_counts = results.groupby("page")["visual_id"].nunique()
    results["page_visual_count"] = results["page"].map(page_counts).fillna(0).astype(int)
    results["page_over_limit"] = known_page & (results["page_visual_count"] > MAX_VISUALS_PER_PAGE)
    overload_sentence = " This page exceeds the configured visible-visual limit; remove, defer, or move secondary visuals."
    results.loc[~results["page_over_limit"], "recommendation"] = (
        results.loc[~results["page_over_limit"], "recommendation"]
        .str.replace(overload_sentence, "", regex=False)
    )
    page_summary["visual_count"] = page_summary["page"].map(page_counts).fillna(0).astype(int)
    page_summary["over_visual_limit"] = (
        reportable_page_group(page_summary["page"])
        & (page_summary["visual_count"] > MAX_VISUALS_PER_PAGE)
    )
    return results, page_summary

## 5. Load the capture and identify report-level causes

Set `INPUT_PATH` to a specific Performance Analyzer JSON file or to a folder containing captures. A folder selects its newest JSON; if the setting is blank, the notebook folder is searched. When no real capture exists, execution stops unless `USE_SAMPLE_IF_NO_FILE=True`, and synthetic output is always marked as demo data.

The loader rejects non-`.json` inputs and files larger than `MAX_INPUT_MB`, computes the source SHA-256 (the run identity), and records optional `CAPTURE_METADATA`. Reports show only the file name unless `INCLUDE_FULL_SOURCE_PATH=True`.

In [ ]:
EXCLUDED_INPUT_NAMES = {
    "analysis_report.json", "visual_diagnostics.json", "recommendations.json",
    "run-summary.json", "findings.json", "quality-issues.json", "manifest.json",
}


def newest_json(search_directory: Path) -> Path | None:
    output_path = (Path.cwd() / OUTPUT_DIR).resolve()
    candidates = [
        path for path in search_directory.glob("*.json")
        if path.is_file()
        and path.name.casefold() not in EXCLUDED_INPUT_NAMES
        and output_path not in path.resolve().parents
    ]
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None


def resolve_input_path(configured_path: str) -> Path | None:
    if configured_path.strip():
        candidate = Path(configured_path).expanduser()
        candidate = candidate if candidate.is_absolute() else Path.cwd() / candidate
        if not candidate.exists():
            raise FileNotFoundError(f"Configured INPUT_PATH does not exist: {candidate}")
        if candidate.is_dir():
            discovered = newest_json(candidate)
            if discovered is None:
                raise FileNotFoundError(f"No JSON files found in configured INPUT_PATH folder: {candidate}")
            return discovered
        if not candidate.is_file() or candidate.suffix.casefold() != ".json":
            raise ValueError(f"Configured INPUT_PATH must be a JSON file or folder: {candidate}")
        return candidate
    return newest_json(Path.cwd())


def load_capture_metadata() -> dict[str, Any]:
    metadata = {key: value for key, value in CAPTURE_METADATA.items() if str(value).strip()}
    if str(CAPTURE_METADATA_PATH).strip():
        metadata_path = Path(CAPTURE_METADATA_PATH).expanduser()
        with metadata_path.open("r", encoding="utf-8-sig") as metadata_file:
            loaded = json.load(metadata_file)
        if not isinstance(loaded, dict):
            raise ValueError(f"CAPTURE_METADATA_PATH must contain a JSON object: {metadata_path}")
        metadata.update(loaded)
    return metadata


capture_metadata = load_capture_metadata()
source_path = resolve_input_path(INPUT_PATH)
demo_mode = source_path is None
if source_path is not None:
    source_size = source_path.stat().st_size
    if source_size > float(MAX_INPUT_MB) * 1024 * 1024:
        raise ValueError(f"{source_path.name} is {source_size / 1024 / 1024:,.1f} MB, above MAX_INPUT_MB={MAX_INPUT_MB}.")
    source_bytes = source_path.read_bytes()
    try:
        performance_payload = json.loads(source_bytes.decode("utf-8-sig"))
    except UnicodeDecodeError as exc:
        raise ValueError(f"{source_path.name} is not UTF-8 JSON text: {exc}") from exc
    except json.JSONDecodeError as exc:
        raise ValueError(f"Malformed JSON in {source_path}: line {exc.lineno}, column {exc.colno}: {exc.msg}") from exc
    source_info = {
        "source_file_name": source_path.name,
        "source_sha256": hashlib.sha256(source_bytes).hexdigest(),
        "source_size_bytes": source_size,
        "capture_metadata": capture_metadata,
    }
    source_label = str(source_path.resolve()) if INCLUDE_FULL_SOURCE_PATH else source_path.name
elif USE_SAMPLE_IF_NO_FILE:
    performance_payload = build_official_sample_payload()
    source_info = {"source_file_name": "EMBEDDED SYNTHETIC SAMPLE", "capture_metadata": {**capture_metadata, "scenario": "synthetic demo"}}
    source_label = "EMBEDDED SYNTHETIC SAMPLE - NOT A REAL REPORT"
    display(HTML("<div style='padding:12px;border-left:5px solid #b45309;background:#fff7ed'><b>DEMO MODE:</b> No input JSON was found. Results below use synthetic data.</div>"))
else:
    raise FileNotFoundError("No JSON export found. Set INPUT_PATH to a Performance Analyzer JSON file or a folder containing one.")

visuals, parse_metadata, event_model = parse_performance_analyzer_with_model(
    performance_payload, validation_mode=VALIDATION_MODE, source_info=source_info
)
run_summary = event_model["run_summary"]
run_summary.update(storage_mode_profile())
print(f"Source: {source_label} | SHA-256: {run_summary['source_sha256']} | Storage mode: {run_summary['storage_mode']}")
print(f"Format: {parse_metadata['format']} | Version: {parse_metadata['version']} | Parsed visuals: {len(visuals)} | Source events/records: {parse_metadata['event_count']}")
print(f"Data quality: {run_summary['quality_status']} ({run_summary['input_assessment']}) | Issues by rule: {run_summary['issue_counts_by_rule']}")

In [ ]:
diagnostics, page_summary = apply_page_identity_guard(*analyze_visuals(visuals))
diagnostics = apply_storage_mode_guidance(diagnostics)

severity_weight = diagnostics["severity"].astype(str).map({"Critical": 100, "Slow": 60, "Review": 25, "Good": 0})
confidence_weight = diagnostics["confidence"].map({"High": 10, "Medium": 5, "Low": 0})
diagnostics["priority_score"] = (
    severity_weight
    + confidence_weight
    + diagnostics["contribution_pct"] * 50
    + diagnostics["repeated_query_count"].clip(upper=5) * 2
    + diagnostics["page_over_limit"].astype(int) * 5
).round(1)
diagnostics = diagnostics.sort_values(["priority_score", "actionable_ms"], ascending=False).reset_index(drop=True)
diagnostics["rank"] = np.arange(1, len(diagnostics) + 1)


def annotate_card_usage(diagnostics_frame: pd.DataFrame, page_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    annotated_diagnostics = diagnostics_frame.copy()
    annotated_pages = page_frame.copy()
    # The legacy single-value Card is "card"; the new multi-value Card is "cardVisual".
    annotated_diagnostics["is_single_value_card"] = (
        annotated_diagnostics["visual_type"].map(normalize_key).eq("card")
    )
    card_counts = annotated_diagnostics.groupby("page")["is_single_value_card"].sum().astype(int)
    annotated_pages["single_value_card_count"] = (
        annotated_pages["page"].map(card_counts).fillna(0).astype(int)
    )
    annotated_pages["multiple_single_value_cards"] = annotated_pages["single_value_card_count"] > 1
    return annotated_diagnostics, annotated_pages


def annotate_page_other_time(page_frame: pd.DataFrame) -> pd.DataFrame:
    annotated_pages = page_frame.copy()
    summed_visual_ms = annotated_pages["total_actionable_ms"] + annotated_pages["total_other_ms"]
    annotated_pages["other_share_pct"] = np.where(
        summed_visual_ms > 0,
        annotated_pages["total_other_ms"] / summed_visual_ms * 100,
        0.0,
    ).round(1)
    return annotated_pages


diagnostics, page_summary = annotate_card_usage(diagnostics, page_summary)
page_summary = annotate_page_other_time(page_summary)

pattern_rows: list[dict[str, Any]] = []
for _, visual_row in diagnostics.iterrows():
    for pattern_match in detect_dax_performance_patterns(str(visual_row["dax_query"])):
        pattern_rows.append({
            "rank": int(visual_row["rank"]),
            "page": str(visual_row["page"]),
            "visual": str(visual_row["visual"]),
            "visual_id": str(visual_row["visual_id"]),
            "severity": str(visual_row["severity"]),
            "dax_ms": float(visual_row["dax_ms"]),
            "root_cause": str(visual_row["root_cause"]),
            "measure_scope": str(visual_row["dax_measure_scope"]),
            **pattern_match,
        })
dax_pattern_columns = [
    "rank", "page", "visual", "visual_id", "severity", "dax_ms", "root_cause", "measure_scope",
    "pattern_id", "tier", "text_confidence", "approval_required", "match_reason", "issue",
    "action", "evidence_needed", "source_url",
]
dax_pattern_findings = pd.DataFrame(pattern_rows, columns=dax_pattern_columns)
if not dax_pattern_findings.empty:
    dax_pattern_findings = dax_pattern_findings.sort_values(
        ["rank", "pattern_id", "page", "visual"], kind="mergesort"
    ).reset_index(drop=True)

report_findings: list[dict[str, Any]] = []
for _, page_row in page_summary.iterrows():
    if page_row["over_visual_limit"]:
        report_findings.append({
            "Scope": page_row["page"],
            "Finding": "Overloaded page",
            "Evidence": (
                f"{int(page_row['visual_count'])} visuals exceed limit {MAX_VISUALS_PER_PAGE}; "
                f"visuals spent {to_number(page_row['total_other_ms']):,.0f} ms "
                f"({to_number(page_row['other_share_pct']):.1f}%) in Other"
            ),
            "Action": "Remove, defer, or move secondary visuals; review interactions and repeat the page-load capture.",
        })
    if page_row["multiple_single_value_cards"]:
        report_findings.append({
            "Scope": page_row["page"],
            "Finding": "Multiple legacy single-value cards",
            "Evidence": f"{int(page_row['single_value_card_count'])} legacy Card visuals are loaded in this page group",
            "Action": "Where the KPIs share compatible formatting and interactions, consolidate them into one new Card visual using its multi-value layout, then repeat the page-load capture.",
        })
repeated = diagnostics.loc[diagnostics["repeated_query_count"] > 1, ["query_signature", "repeated_query_count"]].drop_duplicates()
for _, query_row in repeated.iterrows():
    report_findings.append({"Scope": "Report", "Finding": "Repeated query signature", "Evidence": f"{query_row['query_signature']} appears {int(query_row['repeated_query_count'])} times", "Action": "Check duplicate visuals, synchronized interactions, and opportunities to simplify the page."})
custom_slow = diagnostics[diagnostics["visual_type"].str.contains("custom", case=False, na=False) & diagnostics["severity"].isin(["Critical", "Slow"])]
if not custom_slow.empty:
    report_findings.append({"Scope": "Report", "Finding": "Slow custom visual type", "Evidence": f"{len(custom_slow)} slow custom visual(s)", "Action": "Compare each with a native visual using the same fields and filters."})
other_share = diagnostics["other_ms"].sum() / max(diagnostics["total_ms"].sum(), 1)
if other_share >= 0.50:
    report_findings.append({"Scope": "Report", "Finding": "Synchronization-heavy capture", "Evidence": f"Other is {other_share:.0%} of summed visual duration", "Action": "Repeat a warm capture and optimize actionable sibling visuals before blaming Other-only visuals."})
if not dax_pattern_findings.empty:
    pattern_ids = ", ".join(sorted(dax_pattern_findings["pattern_id"].unique()))
    report_findings.append({"Scope": "Report", "Finding": "DAX performance candidates", "Evidence": f"Text-shape candidates: {pattern_ids}", "Action": "Resolve dependent measure/UDF definitions and validate candidates with Server Timings and Query Plan before rewriting."})
report_findings_frame = pd.DataFrame(report_findings, columns=["Scope", "Finding", "Evidence", "Action"])

print(f"Analyzed {len(diagnostics)} visuals across {diagnostics['page'].nunique()} page label(s).")
print("Root causes:", diagnostics["root_cause"].astype(str).value_counts().to_dict())
print(f"DAX pattern candidate rows: {len(dax_pattern_findings)}")
print("Page groups over 7 visuals:", int(page_summary["over_visual_limit"].sum()))
print("Page groups with multiple legacy cards:", int(page_summary["multiple_single_value_cards"].sum()))

## 5b. Structured findings, trace data quality, and timeline

The next cells turn the event model into evidence-backed findings (blueprint section 13), apply `QUERY_TEXT_MODE` to every downstream output, and show the run summary, data-quality rules, and an event timeline. Findings are labeled **exact** (directly measured) or **heuristic** (threshold or relative-to-run). A `Metrics Truncated` event lowers visual confidence to Low and qualifies detailed conclusions.

In [ ]:
# Structured findings (blueprint 8.7, 13) and query-text privacy policy (blueprint 15)
FINDING_COLUMNS = [
    "finding_id", "rule_id", "rule_version", "priority", "scope_type", "scope_id", "visual_event_ids", "query_event_ids",
    "observation", "evidence_json", "impact", "next_step", "classification",
]
structured_findings: list[dict[str, Any]] = []


def add_finding(rule_id: str, priority: str, scope_type: str, scope_id: Any, observation: str, evidence: dict[str, Any],
                impact: str, next_step: str, classification: str, visual_ids: Iterable[Any] = (), query_ids: Iterable[Any] = ()) -> None:
    structured_findings.append({
        "finding_id": f"{rule_id}-{len(structured_findings) + 1:04d}", "rule_id": rule_id, "rule_version": FINDING_RULES_VERSION,
        "priority": priority, "scope_type": scope_type, "scope_id": "" if scope_id is None else str(scope_id),
        "visual_event_ids": "; ".join(str(value) for value in visual_ids if value not in (None, "")),
        "query_event_ids": "; ".join(str(value) for value in query_ids if value not in (None, "")),
        "observation": observation, "evidence_json": json.dumps(json_safe_evidence(evidence), sort_keys=True, ensure_ascii=False),
        "impact": impact, "next_step": next_step, "classification": classification,
    })


def json_safe_evidence(evidence: dict[str, Any]) -> dict[str, Any]:
    safe = {}
    for key, value in evidence.items():
        if isinstance(value, (np.integer,)):
            value = int(value)
        elif isinstance(value, (np.floating, float)):
            value = None if not np.isfinite(value) else round(float(value), 3)
        elif isinstance(value, np.bool_):
            value = bool(value)
        elif value is not None and not isinstance(value, (str, int, bool, list, dict)):
            value = str(value)
        safe[key] = value
    return safe


def optional_float(value: Any) -> float | None:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if np.isfinite(number) else None


dax_event_frame = event_model["dax_queries"]
direct_event_frame = event_model["direct_queries"]
visual_update_frame = event_model["visual_updates"]
truncation_present = bool(run_summary["truncation_count"])
truncation_note = " Trace truncation was observed; detailed decomposition is incomplete." if truncation_present else ""

for _, row in diagnostics.loc[diagnostics["severity"].astype(str) != "Good"].iterrows():
    add_finding(
        f"VISUAL_{row['rule_id']}", str(row["severity"]), "Visual", row["visual_event_id"] or row["visual_id"],
        f"Visual '{row['visual']}' ranks {int(row['rank'])} with {row['actionable_ms']:,.0f} ms actionable time ({row['total_ms']:,.0f} ms total).",
        {"actionable_ms": row["actionable_ms"], "total_ms": row["total_ms"], "total_ms_source": row["total_ms_source"], "dax_ms": row["dax_ms"],
         "direct_query_ms": row["direct_query_ms"], "render_ms": row["render_ms"], "other_ms": row["other_ms"], "confidence": row["confidence"],
         "is_p90_outlier": bool(row["is_p90_outlier"]), "data_quality_status": row["data_quality_status"]},
        str(row["root_cause"]), str(row["next_diagnostic"]) + (truncation_note if row["is_truncated"] else ""),
        "Heuristic: configured thresholds plus relative-to-run ranking", [row["visual_event_id"]],
    )

dax_durations = pd.to_numeric(dax_event_frame["duration_ms"], errors="coerce")
for _, row in dax_event_frame.loc[dax_durations >= SLOW_DAX_MS].assign(_d=dax_durations).sort_values("_d", ascending=False).iterrows():
    add_finding(
        "SLOW_DAX_QUERY", "Slow", "Query", row["dax_event_id"],
        f"DAX query {row['query_fingerprint'] or '(no text)'} took {row['duration_ms']:,.0f} ms for visual '{row['visual_title'] or row['visual_event_id']}'.",
        {"duration_ms": row["duration_ms"], "row_count": row["row_count"], "error": row["error"], "canceled": row["canceled"],
         "query_fingerprint": row["query_fingerprint"], "direct_query_count": row["direct_query_count"]},
        "Semantic-model evaluation time for this query (ends at the first result row).",
        "Open the captured DAX in DAX Studio or DAX query view and collect Server Timings and Query Plan." + truncation_note,
        "Exact event elapsed; threshold heuristic", [row["visual_event_id"]], [row["dax_event_id"]],
    )

top_level_direct = direct_event_frame.loc[~direct_event_frame["nested_in_direct_query"].astype(bool)] if not direct_event_frame.empty else direct_event_frame
storage_profile = storage_mode_profile()
if storage_profile["is_direct_lake"]:
    source_impact = "Direct Lake fell back to DirectQuery against the SQL analytics endpoint; fallback queries are usually slower than Direct Lake reads."
    source_next_step = ((DIRECT_LAKE_ONELAKE_NOTE if storage_profile["direct_lake_variant"] == "onelake" else "")
                        + "Identify the fallback reason (SQL views, SQL-endpoint security, guardrails, memory) and confirm with DirectQuery_Begin/End trace events.")
else:
    source_impact = "Source execution, gateway/network, or source-capacity latency. Elapsed and ActualQueryDuration are reported separately."
    source_next_step = "Review the source query plan, folding, indexes/partition pruning, gateway latency, and source concurrency."
for _, row in top_level_direct.iterrows():
    elapsed, actual = optional_float(row["duration_ms"]), optional_float(row["actual_query_duration_ms"])
    if max(elapsed or 0, actual or 0) < SLOW_DIRECT_QUERY_MS:
        continue
    add_finding(
        "SLOW_SOURCE_QUERY", "Critical" if (elapsed or 0) > DIRECT_QUERY_REVIEW_MS else "Slow", "Query", row["direct_query_event_id"],
        f"DirectQuery source query elapsed {elapsed if elapsed is not None else 'unknown'} ms (ActualQueryDuration {actual if actual is not None else 'not supplied'} ms) for visual '{row['visual_title'] or row['visual_event_id']}'.",
        {"duration_ms": elapsed, "actual_query_duration_ms": actual, "data_read_duration_ms": row["data_read_duration_ms"], "rows_read": row["rows_read"],
         "is_capabilities_query": row["is_capabilities_query"], "query_text_available": bool(row["query_text_available"])},
        source_impact,
        source_next_step,
        "Exact event elapsed and source metrics; threshold heuristic", [row["visual_event_id"]], [row["direct_query_event_id"]],
    )

if storage_profile["is_direct_lake"]:
    for _, row in visual_update_frame.loc[visual_update_frame["has_direct_query"].astype(bool)].iterrows():
        add_finding(
            "DIRECT_LAKE_FALLBACK", "Slow" if storage_profile["direct_lake_variant"] == "onelake" else "Review", "Visual", row["visual_event_id"],
            f"Visual '{row['visual_title']}' ran {int(row['direct_query_count'])} DirectQuery event(s) covering {optional_float(row['direct_query_covered_ms']) or 0:,.0f} ms on a {storage_profile['storage_mode']} model.",
            {"direct_query_count": row["direct_query_count"], "direct_query_covered_ms": row["direct_query_covered_ms"], "total_elapsed_ms": row["total_elapsed_ms"],
             "direct_lake_variant": storage_profile["direct_lake_variant"]},
            "Fallback bypasses Direct Lake's in-memory reads for this query." if storage_profile["direct_lake_variant"] != "onelake"
            else "Unexpected: Direct Lake on OneLake does not fall back, so these tables are likely not Direct Lake on OneLake.",
            (DIRECT_LAKE_ONELAKE_NOTE if storage_profile["direct_lake_variant"] == "onelake" else "") + RULES_CATALOG["DL_FALLBACK"]["recommendation"],
            "Exact event presence; storage mode from CAPTURE_METADATA", [row["visual_event_id"]],
        )

for _, frame, column, label, id_column in [
    ("DAX", dax_event_frame, "row_count", "DAX RowCount", "dax_event_id"),
    ("DirectQuery", direct_event_frame, "rows_read", "DirectQuery RowsRead", "direct_query_event_id"),
]:
    values = pd.to_numeric(frame[column], errors="coerce") if not frame.empty else pd.Series(dtype=float)
    for _, row in frame.loc[values >= HIGH_RESULT_ROWS].iterrows():
        add_finding(
            "HIGH_RESULT_VOLUME", "Review", "Query", row[id_column],
            f"{label} {float(row[column]):,.0f} is at or above HIGH_RESULT_ROWS={HIGH_RESULT_ROWS:,}.",
            {column: row[column], "duration_ms": row["duration_ms"]},
            "Large result sets can increase transfer, parsing, and rendering work; this is a diagnostic lead, not proof of causality.",
            "Check visual granularity, Top N filters, and whether the visual needs this many rows.",
            "Exact metric; threshold heuristic", [row["visual_event_id"]], [row[id_column]],
        )

for _, row in visual_update_frame.iterrows():
    total, display_ms, waiting = optional_float(row["total_elapsed_ms"]), optional_float(row["display_covered_ms"]), optional_float(row["derived_other_ms"])
    if total is None or total < REVIEW_VISUAL_MS:
        continue
    if display_ms is not None and display_ms >= 0.5 * total:
        add_finding("RENDER_DOMINANT", "Review", "Visual", row["visual_event_id"],
                    f"Render/transform/geocoding covers {display_ms:,.0f} of {total:,.0f} ms for '{row['visual_title']}'.",
                    {"display_covered_ms": display_ms, "total_elapsed_ms": total, "render_ms": row["render_ms"], "transform_ms": row["transform_ms"], "geocoding_ms": row["geocoding_ms"]},
                    "Client-side display work dominates this visual update.", "Reduce data points, conditional formatting, images, or geocoding; compare with a native visual.",
                    "Derived covered time (interval union); heuristic", [row["visual_event_id"]])
    if waiting is not None and waiting >= 0.5 * total:
        add_finding("WAITING_DOMINANT", "Review", "Visual", row["visual_event_id"],
                    f"Unexplained canvas time is {waiting:,.0f} of {total:,.0f} ms for '{row['visual_title']}'.",
                    {"derived_other_ms": waiting, "total_elapsed_ms": total},
                    "UI-thread queueing, synchronization with sibling visuals, or undocumented work; not a clean compute measure.",
                    "Repeat a warm Refresh visuals capture and optimize slow sibling visuals before tuning this one.",
                    "Derived approximation (lifecycle minus union of explained canvas intervals)", [row["visual_event_id"]])

if not dax_event_frame.empty:
    fingerprinted = dax_event_frame.loc[dax_event_frame["query_fingerprint"].astype(str).ne("")]
    for fingerprint, group in fingerprinted.groupby("query_fingerprint"):
        if len(group) > 1:
            add_finding("REPEATED_QUERY_FINGERPRINT", "Review", "Run", fingerprint,
                        f"DAX fingerprint {fingerprint} executed {len(group)} times across {group['visual_event_id'].nunique()} visual update(s) and {group['derived_interaction_id'].replace('', np.nan).nunique()} interaction(s).",
                        {"executions": len(group), "visual_updates": int(group["visual_event_id"].nunique()), "summed_duration_ms": float(pd.to_numeric(group["duration_ms"], errors="coerce").sum())},
                        "Repeated identical queries may indicate duplicate visuals or re-evaluation across interactions.",
                        "Check duplicate visuals, synchronized slicers, and whether repeated interactions are expected; this is an investigation lead.",
                        "Exact fingerprint match; heuristic interpretation", group["visual_event_id"], group["dax_event_id"])

for _, row in visual_update_frame.loc[visual_update_frame["status"].eq("abandoned")].iterrows():
    add_finding("ABANDONED_UPDATE", "Review", "Visual", row["visual_event_id"],
                f"Visual update '{row['visual_title']}' was abandoned" + (f"; superseded by {row['next_update_event_id']}." if row["next_update_event_id"] else "; no later update of the same visual was captured."),
                {"status_raw": row["status_raw"], "total_elapsed_ms": row["total_elapsed_ms"], "next_update_event_id": row["next_update_event_id"]},
                "Abandoned updates usually follow rapid interactions and inflate perceived latency.", "Confirm whether users trigger overlapping interactions; retest with a single deliberate interaction.",
                "Exact status metric", [row["visual_event_id"], row["next_update_event_id"]])

if truncation_present:
    add_finding("TRACE_INCOMPLETE", "Review", "Run", run_summary["run_id"],
                f"{run_summary['truncation_count']} Metrics Truncated event(s) observed" + (f"; {run_summary['truncated_omitted_events']:,.0f} event(s) omitted." if run_summary["truncated_omitted_events"] is not None else "."),
                {"truncation_count": run_summary["truncation_count"], "omitted_events": run_summary["truncated_omitted_events"]},
                "Event counts and duration decompositions are incomplete; lifecycle totals remain valid.",
                "Capture a narrower interaction (single visual refresh) to stay under the per-query event limits.", "Exact trace marker")

for _, row in dax_event_frame.loc[dax_event_frame["error"].eq(True) | dax_event_frame["canceled"].eq(True)].iterrows():
    state = "failed" if row["error"] is True else "was canceled"
    add_finding("DAX_QUERY_FAILED" if row["error"] is True else "DAX_QUERY_CANCELED", "Slow" if row["error"] is True else "Review", "Query", row["dax_event_id"],
                f"DAX query {row['query_fingerprint'] or row['dax_event_id']} {state} for visual '{row['visual_title'] or row['visual_event_id']}'.",
                {"error": row["error"], "canceled": row["canceled"], "duration_ms": row["duration_ms"]},
                "Timing for failed or canceled queries does not represent a successful evaluation.", "Reproduce the interaction and inspect the query error in Power BI Desktop or DAX Studio.",
                "Exact metric", [row["visual_event_id"]], [row["dax_event_id"]])

findings_frame = pd.DataFrame(structured_findings, columns=FINDING_COLUMNS)
new_report_rows = []
for rule_id, label, action in [
    ("SLOW_DAX_QUERY", "Slow DAX query events", "Collect Server Timings and Query Plan for the listed DAX fingerprints."),
    ("SLOW_SOURCE_QUERY", "Slow DirectQuery source queries", "Review source plans, folding, indexes, gateway, and source capacity."),
    ("HIGH_RESULT_VOLUME", "High result volume", "Review visual granularity and Top N filters (diagnostic lead)."),
    ("REPEATED_QUERY_FINGERPRINT", "Repeated DAX fingerprints (event level)", "Check duplicate visuals and repeated interactions."),
    ("ABANDONED_UPDATE", "Abandoned visual updates", "Retest with deliberate, non-overlapping interactions."),
    ("TRACE_INCOMPLETE", "Truncated trace", "Re-capture a narrower interaction before trusting detailed splits."),
    ("DAX_QUERY_FAILED", "Failed DAX queries", "Resolve query errors before interpreting timings."),
    ("DAX_QUERY_CANCELED", "Canceled DAX queries", "Confirm whether cancellations came from overlapping interactions."),
    ("DIRECT_LAKE_FALLBACK", "Direct Lake fallback to DirectQuery", "Identify fallback reasons (SQL views, SQL-endpoint security, guardrails, memory) and confirm with a trace."),
]:
    matches = findings_frame.loc[findings_frame["rule_id"].eq(rule_id)]
    if not matches.empty:
        new_report_rows.append({"Scope": "Report", "Finding": label, "Evidence": f"{len(matches)} finding(s); see findings.json ({rule_id})", "Action": action})
if not storage_profile["storage_mode"] or storage_profile["storage_mode"] == "not recorded":
    if not direct_event_frame.empty:
        new_report_rows.append({"Scope": "Report", "Finding": "Storage mode not recorded",
                                "Evidence": f"{len(direct_event_frame)} DirectQuery event(s) were captured, but CAPTURE_METADATA['storage_mode'] is blank.",
                                "Action": "If the model is Direct Lake, set storage_mode to 'Direct Lake on SQL' or 'Direct Lake on OneLake' and rerun: these events are then fallbacks with different remediation."})
elif storage_profile["is_direct_lake"] and not storage_profile["cache_is_warm"] and diagnostics["rule_id"].isin(["DAX", "MIXED", "OUTLIER"]).any():
    new_report_rows.append({"Scope": "Report", "Finding": "Direct Lake cache state not confirmed warm",
                            "Evidence": f"cache_state is '{capture_metadata.get('cache_state', '') or 'not recorded'}'; DAX-led visuals may include first-use column loading.",
                            "Action": "Repeat a warm Refresh visuals capture and set cache_state='warm' before tuning DAX."})
if run_summary["quality_status"] != "Pass":
    new_report_rows.append({"Scope": "Report", "Finding": f"Trace data quality: {run_summary['quality_status']}",
                            "Evidence": f"{run_summary['input_assessment']}; issues by rule: {run_summary['issue_counts_by_rule']}",
                            "Action": "Review the Data quality and methodology section before relying on detailed conclusions."})
if new_report_rows:
    report_findings_frame = pd.concat([report_findings_frame, pd.DataFrame(new_report_rows)], ignore_index=True)

# Apply QUERY_TEXT_MODE after all text analysis so fingerprints and DAX candidates use the original text.
if QUERY_TEXT_MODE != "full":
    diagnostics["dax_query"] = diagnostics["dax_query"].map(lambda text: apply_query_text_policy(text, "dax"))
    diagnostics["native_query_text"] = diagnostics["native_query_text"].map(lambda text: apply_query_text_policy(text, "sql"))
    diagnostics["direct_query_executions"] = diagnostics["direct_query_executions"].map(
        lambda executions: [{**execution, "native_query_text": apply_query_text_policy(execution.get("native_query_text"), "sql")} for execution in executions]
    )
    diagnostics["raw_properties"] = diagnostics["raw_properties"].map(
        lambda text: json.dumps(apply_query_policy_to_json(json.loads(text)), ensure_ascii=True, default=str) if str(text).strip() else text
    )
    event_model["dax_queries"]["query_text"] = event_model["dax_queries"]["query_text"].map(lambda text: apply_query_text_policy(text, "dax"))
    event_model["direct_queries"]["query_text"] = event_model["direct_queries"]["query_text"].map(lambda text: apply_query_text_policy(text, "sql"))
    events_frame = event_model["events"]
    dialects = np.where(events_frame["event_type_key"].eq(DIRECT_QUERY_KEY), "sql", "dax")
    for column in ("metrics_json", "raw_event_json"):
        events_frame[column] = [
            text if text is None or (isinstance(text, float) and not np.isfinite(text))
            else json.dumps(apply_query_policy_to_json(json.loads(text), dialect), ensure_ascii=False, default=str)
            for text, dialect in zip(events_frame[column], dialects)
        ]
    dax_event_frame, direct_event_frame = event_model["dax_queries"], event_model["direct_queries"]

print(f"Structured findings: {len(findings_frame)} ({findings_frame['rule_id'].value_counts().to_dict()}) | Query text mode: {QUERY_TEXT_MODE}")

In [ ]:
# Report formatting helpers shared by notebook displays and exported reports
CSV_DANGEROUS_PREFIXES = ("=", "+", "-", "@", "\t", "\r")


def sanitize_csv_frame(frame: pd.DataFrame) -> pd.DataFrame:
    sanitized = frame.copy()
    for column_name in sanitized.columns:
        column = sanitized[column_name]
        if (
            pd.api.types.is_object_dtype(column.dtype)
            or pd.api.types.is_string_dtype(column.dtype)
            or isinstance(column.dtype, pd.CategoricalDtype)
        ):
            sanitized[column_name] = column.astype(object).map(
                lambda value: "'" + value
                if isinstance(value, str) and value.startswith(CSV_DANGEROUS_PREFIXES)
                else value
            )
    return sanitized


def json_safe_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(key): json_safe_value(value[key]) for key in sorted(value, key=str)}
    if isinstance(value, (list, tuple)):
        return [json_safe_value(item) for item in value]
    if isinstance(value, set):
        return [json_safe_value(item) for item in sorted(value, key=str)]
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    if isinstance(value, (str, bool, int)):
        return value
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)


def dataframe_to_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    return [
        {str(column): json_safe_value(value) for column, value in row.items()}
        for row in frame.to_dict(orient="records")
    ]


def report_value(value: Any) -> str:
    safe_value = json_safe_value(value)
    if safe_value is None:
        return ""
    if isinstance(safe_value, float):
        return f"{safe_value:,.3f}".rstrip("0").rstrip(".")
    return str(safe_value)


def markdown_escape(value: Any) -> str:
    text = report_value(value)
    return (
        html.escape(text, quote=False)
        .replace("\\", "\\\\")
        .replace("|", "\\|")
        .replace("\r\n", "<br>")
        .replace("\r", "<br>")
        .replace("\n", "<br>")
    )


def markdown_table(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "_No findings._"
    headers = [markdown_escape(column) for column in frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend(
        "| " + " | ".join(markdown_escape(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    )
    return "\n".join(lines)


def markdown_fenced_code(code: str, language: str = "dax") -> str:
    longest_run = max((len(match.group(0)) for match in re.finditer(r"`+", code)), default=0)
    fence = "`" * max(3, longest_run + 1)
    return f"{fence}{language}\n{code}\n{fence}"


def html_table(frame: pd.DataFrame) -> str:
    if frame.empty:
        return '<p class="empty">No findings.</p>'
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in frame.columns)
    rows = []
    for row in frame.itertuples(index=False, name=None):
        cells = "".join(f"<td>{html.escape(report_value(value))}</td>" for value in row)
        rows.append(f"<tr>{cells}</tr>")
    return f'<div class="table-wrap"><table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>'


def html_disclosure_section(
    title: str,
    content: str,
    *,
    open_by_default: bool = False,
    summary_meta: str = "",
    section_id: str = "",
) -> str:
    open_attribute = " open" if open_by_default else ""
    id_attribute = f' id="{html.escape(section_id, quote=True)}"' if section_id else ""
    escaped_title = html.escape(str(title))
    meta_text = report_value(summary_meta)
    meta_html = (
        f'<span class="summary-meta">{html.escape(meta_text)}</span>'
        if meta_text
        else ""
    )
    return (
        f'<details class="report-section"{id_attribute}{open_attribute}>'
        f"<summary>{escaped_title}{meta_html}</summary>"
        f'<div class="report-section-body">{content}</div>'
        "</details>"
    )

In [ ]:
# HTML renderers for the blueprint report pages: quality banner, visuals table, event trees, and timeline (blueprint 14)
TIMELINE_COLORS = {
    "Visual total": "#cbd5e1", "Canvas query": "#93c5fd", "Query generation": "#a5b4fc", "Result parsing": "#c4b5fd",
    "Visual display": "#fbbf24", "Data transform": "#fcd34d", "Geocoding": "#fdba74", "Semantic query": "#60a5fa",
    "DAX query": "#2563eb", "Model query": "#1d4ed8", "Serialize rowset": "#7dd3fc", "Source connection": "#86efac",
    "Direct source query": "#059669", "Evaluated parameters": "#0f766e", "Change detection": "#f472b6", "Open connection": "#bae6fd",
}
QUALITY_BANNER_CLASS = {"Pass": "pass", "Warning": "warn", "Fail": "fail"}


def format_ms(value: Any, missing: str = "not observed") -> str:
    number = optional_float(value)
    return missing if number is None else f"{number:,.1f}"


def present_text(value: Any) -> str:
    """Text for display; None, NaN, and blank values become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def visual_display_title(row: Any) -> str:
    return present_text(row["visual_title"]) or f"Visual {present_text(row['visual_id']) or present_text(row['visual_event_id'])}"


def html_anchor(prefix: str, value: Any) -> str:
    return f"{prefix}-{hashlib.sha1(str(value).encode('utf-8')).hexdigest()[:12]}"


def render_quality_banner(summary: dict[str, Any]) -> str:
    status = str(summary.get("quality_status", "Pass"))
    return (
        f'<div class="quality-banner {QUALITY_BANNER_CLASS.get(status, "warn")}" role="status">'
        f"<strong>Trace data quality: {html.escape(status)}</strong> &mdash; {html.escape(str(summary.get('input_assessment', '')))}. "
        f"Validation mode: {html.escape(str(summary.get('validation_mode', '')))}; strict schema valid: {html.escape(str(summary.get('strict_schema_valid')))}. "
        "Invalid input, incomplete trace, and schema drift are listed separately in the Data quality section.</div>"
    )


def render_visuals_table_html(visual_frame: pd.DataFrame) -> str:
    if visual_frame.empty:
        return '<p class="empty">No Visual Container Lifecycle events were captured.</p>'
    measure_columns = [
        ("total_elapsed_ms", "Total ms", "unknown"), ("canvas_query_ms", "Canvas query ms", "not observed"),
        ("dax_covered_ms", "DAX covered ms", "not observed"), ("direct_query_covered_ms", "DirectQuery covered ms", "not observed"),
        ("render_ms", "Render ms", "not observed"), ("transform_ms", "Transform ms", "not observed"), ("derived_other_ms", "Derived other ms", "unknown"),
    ]
    statuses = sorted(visual_frame["status"].astype(str).unique())
    flag_names = sorted({re.sub(r"\s*\(.*$", "", flag).strip() for flags in visual_frame["flags"].astype(str) for flag in flags.split(";") if flag.strip()})
    controls = (
        '<div class="table-controls">'
        '<label>Search <input type="search" class="js-search" data-table="visuals-table" placeholder="Title or type"></label> '
        '<label>Status <select class="js-filter" data-table="visuals-table" data-attr="status"><option value="">All</option>'
        + "".join(f'<option value="{html.escape(value, quote=True)}">{html.escape(value)}</option>' for value in statuses)
        + '</select></label> <label>Flag <select class="js-filter" data-table="visuals-table" data-attr="flags" data-mode="contains"><option value="">All</option>'
        + "".join(f'<option value="{html.escape(value, quote=True)}">{html.escape(value)}</option>' for value in flag_names)
        + "</select></label></div>"
    )
    headers = ["Visual", "Type", "Status"] + [label for _, label, _ in measure_columns] + ["Flags", "Quality", "Event tree"]
    header_html = "".join(f'<th class="sortable" data-col="{index}" tabindex="0">{html.escape(label)}</th>' for index, label in enumerate(headers))
    rows = []
    ordered = visual_frame.assign(_sort=pd.to_numeric(visual_frame["total_elapsed_ms"], errors="coerce")).sort_values("_sort", ascending=False, na_position="last")
    for _, row in ordered.iterrows():
        title = visual_display_title(row)
        cells = [
            f"<td>{html.escape(str(title))}</td>", f"<td>{html.escape(present_text(row['visual_type']) or 'Unknown')}</td>",
            f"<td>{html.escape(str(row['status']))}</td>",
        ]
        for column, _, missing in measure_columns:
            number = optional_float(row[column])
            cells.append(f'<td class="num" data-sort="{"" if number is None else number}">{html.escape(format_ms(row[column], missing))}</td>')
        cells.append(f"<td>{html.escape(str(row['flags']))}</td>")
        cells.append(f"<td>{html.escape(str(row['data_quality_status']))}</td>")
        cells.append(f'<td><a href="#{html_anchor("tree", row["visual_event_id"])}">tree</a></td>')
        rows.append(
            f'<tr data-status="{html.escape(str(row["status"]), quote=True)}" data-flags="{html.escape(str(row["flags"]), quote=True)}" '
            f'data-search="{html.escape((str(title) + " " + present_text(row["visual_type"])).casefold(), quote=True)}">' + "".join(cells) + "</tr>"
        )
    return (
        controls + '<div class="table-wrap"><table id="visuals-table" class="interactive"><thead><tr>' + header_html
        + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
        + '<p class="note">"not observed" means no event of that type was captured for the visual; "unknown" means the lifecycle duration is unavailable. Neither is zero.</p>'
    )


def _event_children(model: dict[str, Any]) -> dict[str | None, list[dict[str, Any]]]:
    children: dict[str | None, list[dict[str, Any]]] = defaultdict(list)
    for record in model["records"]:
        if record["id"] in model["by_id"] and model["by_id"][record["id"]] is record:
            children[record["parent_id"]].append(record)
    for siblings in children.values():
        siblings.sort(key=lambda record: (record["start"], record["event_index"]))
    return children


def _policy_metrics_preview(model: dict[str, Any], record: dict[str, Any], limit: int = 300) -> str:
    events_frame = model["events"]
    text = events_frame.at[record["event_index"], "metrics_json"] if record["event_index"] in events_frame.index else None
    if text is None or (isinstance(text, float) and not np.isfinite(text)) or text in ("{}", ""):
        return ""
    text = str(text)
    return text if len(text) <= limit else text[: limit - 3] + "..."


def render_event_tree_html(model: dict[str, Any], root_event_id: str, children: dict | None = None, max_depth: int = 40) -> str:
    children = children or _event_children(model)
    root = model["by_id"].get(root_event_id)
    if root is None:
        return '<p class="empty">Event not found.</p>'

    def node_html(record: dict[str, Any], depth: int) -> str:
        badges = []
        if record["is_instantaneous"]:
            badges.append("instantaneous")
        if record["duration_ms"] is not None and record["duration_ms"] < 0:
            badges.append("negative duration")
        if record["outside_parent"]:
            badges.append("outside parent")
        if record.get("component_inferred"):
            badges.append("component inferred")
        if not record["is_known_type"]:
            badges.append("unknown type")
        if normalize_key(record["name"]) == "metricstruncated":
            badges.append("TRUNCATED")
        duration = "instantaneous" if record["is_instantaneous"] else f"{record['duration_ms']:,.1f} ms" if record["duration_ms"] is not None else "unknown"
        metrics_preview = _policy_metrics_preview(model, record)
        body = (
            f"<span class=\"evt-key\">{html.escape(str(record['event_type_key']))}</span> "
            f"<span class=\"evt-ms\">{html.escape(duration)}</span>"
            + "".join(f' <span class="badge">{html.escape(badge)}</span>' for badge in badges)
            + f" <span class=\"evt-id\">{html.escape(str(record['id']))}</span>"
            + (f"<div class=\"evt-metrics\">{html.escape(metrics_preview)}</div>" if metrics_preview else "")
        )
        kids = children.get(record["id"], []) if depth < max_depth else []
        return "<li>" + body + ("<ul>" + "".join(node_html(child, depth + 1) for child in kids) + "</ul>" if kids else "") + "</li>"

    return f'<ul class="event-tree">{node_html(root, 0)}</ul>'


def render_timeline_svg(model: dict[str, Any], visual_event_id: str | None = None, width: int = 1100, max_events: int | None = None) -> str:
    """Gantt/flame-style SVG: lanes per visual update, sub-rows by depth, color by friendly category, hover titles."""
    max_events = int(max_events or TIMELINE_MAX_EVENTS)
    analyzable = [record for record in model["records"] if record["id"] in model["by_id"] and model["by_id"][record["id"]] is record and record["start"]]
    if visual_event_id is not None:
        analyzable = [record for record in analyzable if record["visual_event_id"] == visual_event_id]
    if not analyzable:
        return '<p class="empty">No timed events to plot.</p>'
    omitted = 0
    if len(analyzable) > max_events:
        keep = sorted(analyzable, key=lambda record: -(record["duration_ms"] or 0))[:max_events]
        omitted = len(analyzable) - len(keep)
        analyzable = keep
    origin = min(record["start"] for record in analyzable)
    horizon = max((record["end"] or record["start"]) for record in analyzable)
    span_ms = max(milliseconds_between(origin, horizon), 1.0)
    label_width, bar_height, lane_gap = 230, 6 if visual_event_id is None else 12, 8
    plot_width = width - label_width - 20
    issue_scope = set(model["quality_issues"].loc[model["quality_issues"]["rule_id"].isin(["CHILD_OUTSIDE_PARENT", "NEGATIVE_DURATION"]), "scope_id"])

    lanes: dict[str, list[dict[str, Any]]] = defaultdict(list)
    lane_titles: dict[str, str] = {}
    for record in analyzable:
        if record["visual_event_id"]:
            lane = record["visual_event_id"]
            lane_titles[lane] = str(metric_lookup(model["by_id"][lane]["metrics"], "visualTitle") or f"Visual {lane}")
        elif record["event_type_key"] == USER_ACTION_KEY:
            lane, lane_titles["__user_actions__"] = "__user_actions__", "User actions"
        else:
            lane, lane_titles["__other__"] = "__other__", "Unattributed / other roots"
        lanes[lane].append(record)
    ordered_lanes = sorted(lanes, key=lambda lane: (lane != "__user_actions__", min(record["start"] for record in lanes[lane])))

    def x_of(moment: datetime) -> float:
        return label_width + milliseconds_between(origin, moment) / span_ms * plot_width

    parts, y = [], 28
    for lane in ordered_lanes:
        records = lanes[lane]
        base_depth = min(record["depth"] or 0 for record in records)
        rows = max((record["depth"] or 0) - base_depth for record in records) + 1
        lane_height = rows * (bar_height + 2) + lane_gap
        parts.append(f'<text x="4" y="{y + bar_height + 2}" font-size="11" fill="#172033">{html.escape(lane_titles[lane][:34])}</text>')
        parts.append(f'<line x1="{label_width}" x2="{width - 20}" y1="{y + lane_height - lane_gap / 2:.1f}" y2="{y + lane_height - lane_gap / 2:.1f}" stroke="#e5e7eb"/>')
        for record in sorted(records, key=lambda record: (record["depth"] or 0, record["start"])):
            row_y = y + ((record["depth"] or 0) - base_depth) * (bar_height + 2)
            start_x = x_of(record["start"])
            duration = "instantaneous" if record["is_instantaneous"] else (f"{record['duration_ms']:,.3f} ms" if record["duration_ms"] is not None else "unknown")
            title = (
                f"{record['event_type_key']}\nid: {record['id']}\nparent: {record['parent_id']}\nstart: {iso_utc(record['start'])}\n"
                f"end: {iso_utc(record['end'])}\nduration: {duration}\nmetrics: {_policy_metrics_preview(model, record, 400) or '{}'}"
            )
            flagged = record["id"] in issue_scope
            if normalize_key(record["name"]) == "metricstruncated":
                parts.append(f'<polygon points="{start_x:.1f},{row_y} {start_x - 5:.1f},{row_y + bar_height + 3} {start_x + 5:.1f},{row_y + bar_height + 3}" fill="#dc2626"><title>{html.escape("TRUNCATED - " + title)}</title></polygon>')
            elif record["end"] is None or record["duration_ms"] is None or record["duration_ms"] < 0:
                half = bar_height / 2 + 1
                parts.append(f'<polygon points="{start_x:.1f},{row_y - 1} {start_x + half:.1f},{row_y + half - 1} {start_x:.1f},{row_y + 2 * half - 1} {start_x - half:.1f},{row_y + half - 1}" '
                             f'fill="{"#dc2626" if flagged else "#475569"}"><title>{html.escape(title)}</title></polygon>')
            else:
                bar_width = max(x_of(record["end"]) - start_x, 1.0)
                color = TIMELINE_COLORS.get(record["friendly_category"], "#9ca3af")
                stroke = ' stroke="#dc2626" stroke-width="1.5"' if flagged else ""
                parts.append(f'<rect x="{start_x:.1f}" y="{row_y}" width="{bar_width:.1f}" height="{bar_height}" fill="{color}"{stroke}><title>{html.escape(title)}</title></rect>')
        y += lane_height
    ticks = []
    for index in range(6):
        tick_ms = span_ms * index / 5
        tick_x = label_width + plot_width * index / 5
        ticks.append(f'<line x1="{tick_x:.1f}" x2="{tick_x:.1f}" y1="18" y2="{y}" stroke="#f1f5f9"/>'
                     f'<text x="{tick_x:.1f}" y="12" font-size="10" text-anchor="middle" fill="#566075">{tick_ms:,.0f} ms</text>')
    legend = "".join(
        f'<rect x="{label_width + index * 118}" y="{y + 8}" width="10" height="10" fill="{color}"/><text x="{label_width + index * 118 + 14}" y="{y + 17}" font-size="10" fill="#172033">{html.escape(name)}</text>'
        for index, (name, color) in enumerate(list(TIMELINE_COLORS.items())[:7])
    )
    note = f'<text x="4" y="{y + 40}" font-size="10" fill="#b91c1c">{omitted:,} shortest event(s) omitted (TIMELINE_MAX_EVENTS={max_events:,}).</text>' if omitted else ""
    height = y + (52 if omitted else 30)
    return (
        f'<svg class="timeline" xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img" '
        f'aria-label="Event timeline in milliseconds from capture start">{"".join(ticks)}{"".join(parts)}{legend}{note}</svg>'
    )


SEVERITY_CLASS = {"Critical": "sev-critical", "Slow": "sev-slow", "Review": "sev-review", "Good": "sev-good"}
PRIORITY_TIMING_FIELDS = [
    ("DAX ms", "#2563eb"), ("DirectQuery ms", "#059669"), ("Render ms", "#d97706"), ("Other ms", "#94a3b8"),
]


def _priority_timing_bar(record: dict[str, Any], width: int = 520) -> str:
    """Horizontal bar of component timings relative to total (components can overlap, so shares are indicative)."""
    total = max(optional_float(record.get("Total ms")) or 0.0, 1.0)
    parts, x = [], 0.0
    for label, color in PRIORITY_TIMING_FIELDS:
        value = optional_float(record.get(label)) or 0.0
        if value <= 0:
            continue
        segment = min(value / total * width, width - x)
        if segment <= 0:
            continue
        parts.append(f'<rect x="{x:.1f}" y="0" width="{segment:.1f}" height="14" fill="{color}"><title>{html.escape(label)}: {value:,.0f}</title></rect>')
        x += segment
    legend = "".join(
        f'<rect x="{index * 130}" y="22" width="10" height="10" fill="{color}"/><text x="{index * 130 + 14}" y="31" font-size="11" fill="#566075">{html.escape(label)}</text>'
        for index, (label, color) in enumerate(PRIORITY_TIMING_FIELDS)
    )
    return (f'<svg class="priority-bar" width="{width}" height="36" viewBox="0 0 {width} 36" role="img" aria-label="Timing breakdown">'
            f'<rect x="0" y="0" width="{width}" height="14" fill="#eef2f7"/>{"".join(parts)}{legend}</svg>')


def render_priorities_html(priority_frame: pd.DataFrame) -> str:
    """Readable remediation priorities: a compact sortable summary plus one expandable card per visual."""
    if priority_frame.empty:
        return '<p class="empty">No visuals were ranked.</p>'
    records = dataframe_to_records(priority_frame)
    summary_columns = [("Rank", True), ("Severity", False), ("Visual", False), ("Likely root cause", False),
                       ("Confidence", False), ("Actionable ms", True), ("Total ms", True), ("Priority score", True)]
    header = "".join(f'<th class="sortable" data-col="{index}" tabindex="0">{html.escape(label)}</th>' for index, (label, _) in enumerate(summary_columns))
    rows, cards = [], []
    for record in records:
        anchor = html_anchor("priority", f"{record['Rank']}-{record['Visual']}")
        severity = present_text(record.get("Severity")) or "Good"
        badge = f'<span class="sev {SEVERITY_CLASS.get(severity, "sev-good")}">{html.escape(severity)}</span>'
        visual_cell = (f'<a href="#{anchor}">{html.escape(present_text(record.get("Visual")))}</a>'
                       f'<div class="muted">{html.escape(present_text(record.get("Page")))} &middot; {html.escape(present_text(record.get("Visual type")))}</div>')
        cells = []
        for label, numeric in summary_columns:
            if label == "Severity":
                cells.append(f"<td>{badge}</td>")
            elif label == "Visual":
                cells.append(f"<td>{visual_cell}</td>")
            elif numeric:
                number = optional_float(record.get(label))
                shown = "" if number is None else (f"{number:,.0f}" if label != "Priority score" else f"{number:,.1f}")
                cells.append(f'<td class="num" data-sort="{"" if number is None else number}">{html.escape(shown)}</td>')
            else:
                cells.append(f"<td>{html.escape(present_text(record.get(label)))}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")

        timing_chips = "".join(
            f'<div class="chip"><span>{html.escape(label)}</span><strong>{html.escape(format_ms(record.get(label), "n/a").replace(".0", ""))}</strong></div>'
            for label in ("Actionable ms", "Total ms", "DAX ms", "DirectQuery ms", "Render ms", "Other ms")
        )
        details = [("Recommendation", record.get("Recommendation")), ("Next diagnostic", record.get("Next diagnostic"))]
        dax_details = [
            ("DAX text evidence", record.get("DAX text evidence")), ("DAX visibility", record.get("DAX visibility")),
            ("DAX pattern candidates", record.get("DAX pattern candidates")), ("DAX pattern tiers", record.get("DAX pattern tiers")),
            ("DAX pattern next step", record.get("DAX pattern next step")),
        ]
        dax_html = "".join(
            f"<dt>{html.escape(label)}</dt><dd>{html.escape(present_text(value))}</dd>"
            for label, value in dax_details if present_text(value) and present_text(value) not in {"0", "No captured DAX query", "No captured query"}
        )
        source = present_text(record.get("Source"))
        source_html = f'<p class="muted">Guidance: <a href="{html.escape(source, quote=True)}">{html.escape(source)}</a></p>' if source.startswith("https://") else ""
        cards.append(
            f'<details class="priority-card {SEVERITY_CLASS.get(severity, "sev-good")}" id="{anchor}">'
            f'<summary><span class="rank">#{html.escape(present_text(record.get("Rank")))}</span> {badge} '
            f'<strong>{html.escape(present_text(record.get("Visual")))}</strong> '
            f'<span class="muted">{html.escape(present_text(record.get("Page")))} &middot; {html.escape(present_text(record.get("Likely root cause")))} &middot; '
            f'{html.escape(format_ms(record.get("Actionable ms"), "n/a").replace(".0", ""))} ms actionable &middot; confidence {html.escape(present_text(record.get("Confidence")))}</span></summary>'
            f'<div class="chip-row">{timing_chips}</div>{_priority_timing_bar(record)}'
            + "".join(f"<h4>{html.escape(label)}</h4><p>{html.escape(present_text(value))}</p>" for label, value in details if present_text(value))
            + (f'<h4>DAX evidence</h4><dl class="kv">{dax_html}</dl>' if dax_html else "")
            + source_html + "</details>"
        )
    return (
        '<div class="table-wrap"><table id="priority-table" class="compact"><thead><tr>' + header + "</tr></thead><tbody>"
        + "".join(rows) + "</tbody></table></div>"
        + '<p class="note">Select a visual to open its card. Component timings can overlap, so the bar shows indicative shares of total time.</p>'
        + "".join(cards)
    )


def render_priorities_markdown(priority_frame: pd.DataFrame) -> str:
    if priority_frame.empty:
        return "_No visuals were ranked._"
    summary = priority_frame[["Rank", "Severity", "Page", "Visual", "Likely root cause", "Confidence", "Actionable ms", "Total ms"]]
    sections = [markdown_table(summary)]
    for record in dataframe_to_records(priority_frame):
        lines = [
            f"### #{markdown_escape(record['Rank'])} {markdown_escape(record['Visual'])} ({markdown_escape(record['Severity'])})",
            f"- **Page / type:** {markdown_escape(record['Page'])} / {markdown_escape(record['Visual type'])}",
            f"- **Likely root cause:** {markdown_escape(record['Likely root cause'])} (confidence {markdown_escape(record['Confidence'])})",
            "- **Timing (ms):** " + " | ".join(
                f"{label.replace(' ms', '')} {markdown_escape(format_ms(record.get(label), 'n/a'))}"
                for label in ("Actionable ms", "Total ms", "DAX ms", "DirectQuery ms", "Render ms", "Other ms")
            ),
            f"- **Recommendation:** {markdown_escape(record['Recommendation'])}",
            f"- **Next diagnostic:** {markdown_escape(record['Next diagnostic'])}",
        ]
        if present_text(record.get("DAX pattern candidates")):
            lines.append(f"- **DAX pattern candidates:** {markdown_escape(record['DAX pattern candidates'])} ({markdown_escape(record.get('DAX pattern tiers'))})")
        if present_text(record.get("Source")):
            lines.append(f"- **Guidance:** {markdown_escape(record['Source'])}")
        sections.append("\n".join(lines))
    return "\n\n".join(sections)


print("Report renderers ready.")

In [ ]:
NOTEBOOK_REPORT_STYLE = """<style>
.quality-banner{padding:10px 14px;margin:8px 0;border-left:6px solid #64748b;background:#f8fafc}
.quality-banner.pass{border-color:#15803d;background:#f0fdf4}.quality-banner.warn{border-color:#b45309;background:#fffbeb}.quality-banner.fail{border-color:#b91c1c;background:#fef2f2}
</style>"""
display(HTML(NOTEBOOK_REPORT_STYLE + render_quality_banner(run_summary)))

provenance_keys = [
    "source_file_name", "source_sha256", "export_version", "session_id", "storage_mode", "direct_lake_variant", "min_start_utc", "max_end_utc", "capture_elapsed_ms",
    "event_count", "analyzable_event_count", "quarantined_event_count", "unattributed_event_count", "visual_update_count",
    "visual_status_counts", "dax_query_count", "direct_query_count", "dax_error_count", "dax_canceled_count", "orphan_count",
    "cycle_count", "duplicate_id_count", "truncation_count", "truncated_omitted_events", "user_action_count", "validation_mode",
    "strict_schema_valid", "quality_status", "input_assessment", "query_text_mode", "parser_version", "mapping_version", "finding_rules_version",
]
run_summary_frame = pd.DataFrame([{"Field": key, "Value": report_value(run_summary.get(key))} for key in provenance_keys])
display(Markdown("### Run summary and provenance"))
display(run_summary_frame)

display(Markdown("### Data-quality issues by rule"))
quality_issue_frame = event_model["quality_issues"]
issue_rule_summary = pd.DataFrame([
    {"Rule": rule_id, "Severity": event_model["issues"].severity_by_rule.get(rule_id, QUALITY_RULES[rule_id]["severity"]),
     "Class": QUALITY_RULES[rule_id]["class"], "Occurrences": count, "Condition": QUALITY_RULES[rule_id]["condition"]}
    for rule_id, count in sorted(run_summary["issue_counts_by_rule"].items())
], columns=["Rule", "Severity", "Class", "Occurrences", "Condition"])
display(issue_rule_summary if not issue_rule_summary.empty else Markdown("No data-quality issues were detected."))

display(Markdown(f"### Visual updates (top {int(SUMMARY_TOP_N)} by lifecycle elapsed)"))
top_visual_updates = visual_update_frame.assign(_sort=pd.to_numeric(visual_update_frame["total_elapsed_ms"], errors="coerce")).sort_values("_sort", ascending=False).head(int(SUMMARY_TOP_N))
display(top_visual_updates[["visual_title", "visual_type", "status", "total_elapsed_ms", "dax_covered_ms", "direct_query_covered_ms", "render_ms", "derived_other_ms", "backend_activity", "flags"]])

display(Markdown(f"### DAX query events (top {int(SUMMARY_TOP_N)} by elapsed)"))
top_dax_events = dax_event_frame.assign(_sort=pd.to_numeric(dax_event_frame["duration_ms"], errors="coerce")).sort_values("_sort", ascending=False).head(int(SUMMARY_TOP_N))
display(top_dax_events[["visual_title", "duration_ms", "row_count", "error", "canceled", "query_fingerprint", "direct_query_count", "direct_query_covered_ms"]])

display(Markdown("### Structured findings"))
display(findings_frame[["finding_id", "priority", "scope_type", "observation", "classification"]].head(int(TOP_N)) if not findings_frame.empty else Markdown("No structured findings crossed a configured rule."))

display(Markdown("### Timeline (ms from capture start; hover a bar for event details)"))
display(HTML(render_timeline_svg(event_model)))

## 6. Performance summary and hotspots

The main chart stacks only query, render, and parameter work. `Other ms` remains in the evidence table because it helps identify page synchronization, but it is intentionally excluded from actionable ranking.

In [ ]:
slow_count = int(diagnostics["severity"].isin(["Critical", "Slow"]).sum())
slowest = diagnostics.loc[diagnostics["actionable_ms"].idxmax()]
overloaded_pages = int(page_summary["over_visual_limit"].sum())
kpis = [
    ("Visuals analyzed", f"{len(diagnostics):,}"),
    ("Critical / slow", f"{slow_count:,}"),
    ("Slowest actionable visual", f"{slowest['visual']} ({slowest['actionable_ms']:,.0f} ms)"),
    ("Pages over visual limit", f"{overloaded_pages:,}"),
]
kpi_html = "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:10px;margin:10px 0'>" + "".join(
    f"<div style='border:1px solid #d1d5db;border-top:4px solid #0f766e;padding:12px;background:#fff'><div style='font-size:12px;color:#475569'>{html.escape(label)}</div><div style='font-size:20px;font-weight:700;color:#111827'>{html.escape(value)}</div></div>"
    for label, value in kpis
) + "</div>"
display(HTML(kpi_html))

chart_data = diagnostics.nlargest(TOP_N, "actionable_ms").sort_values("actionable_ms")
chart_data = chart_data.assign(label=chart_data["page"].astype(str) + " | " + chart_data["visual"].astype(str))
if PLOTLY_AVAILABLE:
    hotspot_figure = go.Figure()
    for column, label, color in [
        ("query_ms", "Query (max of DAX/DirectQuery)", "#2563eb"),
        ("render_ms", "Visual display", "#d97706"),
        ("parameter_ms", "Evaluated parameters", "#0f766e"),
    ]:
        hotspot_figure.add_bar(y=chart_data["label"], x=chart_data[column], name=label, orientation="h", marker_color=color)
    hotspot_figure.update_layout(
        title="Top actionable visual bottlenecks", barmode="stack", template="plotly_white",
        xaxis_title="Actionable duration (ms)", yaxis_title="", height=max(380, 42 * len(chart_data)),
        legend_title="Component", margin=dict(l=20, r=20, t=60, b=40),
    )
    hotspot_figure.show()

    pareto_data = diagnostics.sort_values("actionable_ms", ascending=False).reset_index(drop=True)
    pareto_data["cumulative_pct"] = pareto_data["actionable_ms"].cumsum() / max(pareto_data["actionable_ms"].sum(), 1) * 100
    pareto_figure = go.Figure()
    pareto_figure.add_bar(x=pareto_data["visual"], y=pareto_data["actionable_ms"], name="Actionable ms", marker_color="#2563eb")
    pareto_figure.add_scatter(x=pareto_data["visual"], y=pareto_data["cumulative_pct"], name="Cumulative %", mode="lines+markers", yaxis="y2", line_color="#b91c1c")
    pareto_figure.update_layout(title="Actionable-duration Pareto", template="plotly_white", yaxis_title="Actionable ms", yaxis2=dict(title="Cumulative %", overlaying="y", side="right", range=[0, 105]), xaxis_tickangle=-30)
    pareto_figure.show()
else:
    axis = chart_data.set_index("label")[["query_ms", "render_ms", "parameter_ms"]].plot.barh(stacked=True, color=["#2563eb", "#d97706", "#0f766e"], figsize=(11, max(4, len(chart_data) * 0.45)))
    axis.set_title("Top actionable visual bottlenecks")
    axis.set_xlabel("Actionable duration (ms)")
    axis.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 7. Prioritized remediation plan

Start at rank 1. Apply one change, repeat the same interaction under comparable filters/cache/device conditions, and compare the same timing component.

In [ ]:
priority_columns = [
    "rank", "severity", "page", "visual", "visual_type", "root_cause", "actionable_ms", "total_ms",
    "dax_ms", "direct_query_ms", "render_ms", "other_ms", "confidence", "priority_score",
    "dax_pattern_ids", "dax_measure_scope", "dax_evidence", "recommendation", "next_diagnostic", "source_url",
]
priority_view = diagnostics[priority_columns].rename(columns={
    "rank": "Rank", "severity": "Severity", "page": "Page", "visual": "Visual", "visual_type": "Visual type",
    "root_cause": "Likely root cause", "actionable_ms": "Actionable ms", "total_ms": "Total ms",
    "dax_ms": "DAX ms", "direct_query_ms": "DirectQuery ms", "render_ms": "Render ms", "other_ms": "Other ms",
    "confidence": "Confidence", "priority_score": "Priority score", "dax_pattern_ids": "DAX pattern candidates",
    "dax_measure_scope": "DAX visibility", "dax_evidence": "DAX text evidence",
    "recommendation": "Recommendation", "next_diagnostic": "Next diagnostic", "source_url": "Source",
})

severity_colors = {"Critical": "#fee2e2", "Slow": "#ffedd5", "Review": "#fef9c3", "Good": "#dcfce7"}

def highlight_severity(value: Any) -> str:
    return f"background-color: {severity_colors.get(str(value), '#ffffff')}; font-weight: 700"

compact_priority_columns = [
    "Rank", "Severity", "Page", "Visual", "Visual type", "Likely root cause", "Confidence",
    "Actionable ms", "Total ms", "DAX ms", "DirectQuery ms", "Render ms", "Other ms", "Priority score",
]
compact_priority_view = priority_view[compact_priority_columns].head(int(TOP_N))
if importlib.util.find_spec("jinja2") is not None:
    styled_priority = (
        compact_priority_view.style
        .map(highlight_severity, subset=["Severity"])
        .format({column: "{:,.0f}" for column in ["Actionable ms", "Total ms", "DAX ms", "DirectQuery ms", "Render ms", "Other ms"]})
        .format({"Priority score": "{:.1f}"})
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
        .hide(axis="index")
    )
    display(styled_priority)
else:
    display(compact_priority_view)

display(Markdown(f"### Recommendations for the top {min(int(SUMMARY_TOP_N), len(priority_view))} visuals\n\n"
                 + render_priorities_markdown(priority_view.head(int(SUMMARY_TOP_N))).split("\n\n", 1)[-1]))

if report_findings_frame.empty:
    display(Markdown("**Report-level findings:** No repeated-query, page-overload, custom-visual, synchronization-heavy, or DAX text pattern crossed its configured rule."))
else:
    display(Markdown("### Report-level findings"))
    display(report_findings_frame)

## 8. Page, severity, and DAX evidence

Page totals are sums of captured visual work, not report wall-clock duration, because visual operations can queue or overlap. Captured DAX is retained in full for export; the notebook preview is shortened for readability.

In [ ]:
display(Markdown("### Page summary"))
display(page_summary.rename(columns={
    "page": "Page", "visual_count": "Visuals", "total_actionable_ms": "Summed actionable ms",
    "max_actionable_ms": "Max visual ms", "median_actionable_ms": "Median visual ms",
    "slow_visual_count": "Critical/slow visuals", "total_other_ms": "Summed Other ms",
    "other_share_pct": "Other share %",
    "dominant_root_cause": "Most common root cause", "over_visual_limit": "Over visual limit",
    "single_value_card_count": "Legacy single-value cards",
    "multiple_single_value_cards": "Card consolidation candidate",
}))

severity_counts = diagnostics["severity"].astype(str).value_counts().reindex(["Critical", "Slow", "Review", "Good"], fill_value=0)
if PLOTLY_AVAILABLE:
    severity_figure = go.Figure(go.Bar(
        x=severity_counts.index, y=severity_counts.values,
        marker_color=["#b91c1c", "#ea580c", "#ca8a04", "#15803d"],
    ))
    severity_figure.update_layout(title="Visual severity counts", template="plotly_white", xaxis_title="Severity", yaxis_title="Visual count", height=320)
    severity_figure.show()
else:
    severity_counts.plot.bar(color=["#b91c1c", "#ea580c", "#ca8a04", "#15803d"], title="Visual severity counts")
    plt.show()

dax_evidence = diagnostics.loc[diagnostics["dax_query"].str.strip() != "", [
    "rank", "page", "visual", "severity", "dax_ms", "query_signature", "repeated_query_count",
    "dax_pattern_ids", "dax_measure_scope", "dax_evidence", "dax_query",
]].copy()
dax_evidence["query_preview"] = dax_evidence["dax_query"].map(lambda query: query if len(query) <= 500 else query[:497] + "...")
display(Markdown("### Captured DAX evidence"))
if dax_evidence.empty:
    display(Markdown("No DAX query text was present in the parsed export."))
else:
    display(dax_evidence.drop(columns="dax_query").rename(columns={
        "rank": "Rank", "page": "Page", "visual": "Visual", "severity": "Severity", "dax_ms": "DAX ms",
        "query_signature": "Query signature", "repeated_query_count": "Repeat count",
        "dax_pattern_ids": "Pattern candidates", "dax_measure_scope": "Measure visibility",
        "dax_evidence": "General text heuristic", "query_preview": "Query preview",
    }))

display(Markdown("### Routed DAX performance candidates"))
if dax_pattern_findings.empty:
    display(Markdown(
        "No catalog pattern was inferred from the captured query text. This does **not** clear the measures: "
        "resolve referenced measure/UDF definitions and use Server Timings and Query Plan to inspect FE/SE work."
    ))
else:
    display(dax_pattern_findings.rename(columns={
        "rank": "Rank", "page": "Page", "visual": "Visual", "visual_id": "Visual ID",
        "severity": "Severity", "dax_ms": "DAX ms", "root_cause": "Visual root cause",
        "measure_scope": "Measure visibility", "pattern_id": "Pattern", "tier": "Tier",
        "text_confidence": "Text confidence", "approval_required": "Approval required",
        "match_reason": "Why matched", "issue": "Possible issue", "action": "Candidate action",
        "evidence_needed": "Evidence needed", "source_url": "Pattern source",
    }))

## 9. Export the report

This export writes sanitized CSV, JSON, Markdown, and standalone HTML report files under `OUTPUT_DIR`. CSV text cells that could be interpreted as formulas are neutralized before writing.

| File | Purpose |
| --- | --- |
| `analysis_report.html` / `index.html` | Self-contained report: quality banner, Run summary, Visuals (search, filter, sort), Visual event trees, Queries (expandable, copyable text), Timeline, Data quality and methodology, plus the remediation sections. Carries a hash-based Content-Security-Policy. |
| `analysis_report.md` | Portable narrative report with the same sections (timeline in HTML only). |
| `analysis_report.json` | Findings, provenance, run summary, quality issues, visual updates, and query events. |
| `visual_diagnostics.csv`, `page_summary.csv`, `dax_queries.csv`, `dax_pattern_findings.csv`, `direct_query_queries.csv` | Visual-level diagnostics (unchanged contract). |
| `events.csv` (+ `events.parquet` with pyarrow) | One row per raw event, including quarantined records, with hierarchy fields, `metrics_json`, and `raw_event_json`. |
| `visual-updates.csv`, `dax-queries.csv`, `direct-queries.csv` | Event-level tables from blueprint section 8. |
| `quality-issues.json`, `findings.json`, `run-summary.json` | Data-quality issues, structured findings, and run provenance. |
| `manifest.json` | Artifact list with SHA-256 hashes, source hash, parser/mapping/rule versions, and analysis timestamp. |

In [ ]:
output_dir = Path(OUTPUT_DIR).expanduser()
if not output_dir.is_absolute():
    output_dir = (Path.cwd() / output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

visual_export = diagnostics.sort_values(
    ["rank", "page", "visual", "visual_id"], kind="mergesort"
).reset_index(drop=True)
page_export = page_summary.sort_values(
    ["total_actionable_ms", "page"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
findings_export = report_findings_frame.sort_values(
    ["Scope", "Finding", "Evidence", "Action"], kind="mergesort"
).reset_index(drop=True)
dax_pattern_export = dax_pattern_findings.reindex(columns=dax_pattern_columns).sort_values(
    ["rank", "pattern_id", "page", "visual"], kind="mergesort"
).reset_index(drop=True)

dax_columns = [
    "rank", "page", "visual", "visual_id", "severity", "dax_ms", "query_signature",
    "repeated_query_count", "dax_pattern_count", "dax_pattern_ids", "dax_pattern_tiers",
    "dax_measure_scope", "dax_pattern_next_step", "dax_evidence", "dax_query",
]
dax_mask = diagnostics["dax_query"].fillna("").astype(str).str.strip().ne("")
dax_export = diagnostics.loc[dax_mask, dax_columns].sort_values(
    ["rank", "page", "visual", "visual_id"], kind="mergesort"
).reset_index(drop=True)

direct_query_columns = [
    "page_group", "visual", "visual_id", "visual_type", "evidence_type",
    "direct_query_ms", "visual_dax_ms", "visual_query_ms",
    "visual_actionable_ms", "visual_total_ms", "capture_status",
    "native_query_text",
]
direct_query_evidence: list[dict[str, Any]] = []
for _, visual_row in visual_export.iterrows():
    executions = visual_row.get("direct_query_executions", [])
    has_per_execution_data = isinstance(executions, list) and len(executions) > 0
    execution_candidates = executions if has_per_execution_data else []
    if not has_per_execution_data and float(visual_row["direct_query_ms"]) > DIRECT_QUERY_REVIEW_MS:
        native_query_text = str(visual_row.get("native_query_text", "") or "")
        execution_candidates = [{
            "duration_ms": float(visual_row["direct_query_ms"]),
            "native_query_text": native_query_text,
            "capture_status": (
                "Captured" if native_query_text.strip() else "DirectQuery text was not captured"
            ),
            "evidence_type": "Aggregate fallback",
        }]
    for execution in execution_candidates:
        execution_ms = to_number(execution.get("duration_ms"))
        if execution_ms <= DIRECT_QUERY_REVIEW_MS:
            continue
        native_query_text = str(execution.get("native_query_text", "") or "")
        capture_status = (
            "Captured" if native_query_text.strip() else "DirectQuery text was not captured"
        )
        direct_query_evidence.append({
            "page_group": str(visual_row["page"]),
            "visual": str(visual_row["visual"]),
            "visual_id": str(visual_row["visual_id"]),
            "visual_type": str(visual_row["visual_type"]),
            "evidence_type": str(execution.get("evidence_type", "Individual execution")),
            "direct_query_ms": round(execution_ms, 3),
            "visual_dax_ms": float(visual_row["dax_ms"]),
            "visual_query_ms": float(visual_row["query_ms"]),
            "visual_actionable_ms": float(visual_row["actionable_ms"]),
            "visual_total_ms": float(visual_row["total_ms"]),
            "capture_status": capture_status,
            "native_query_text": native_query_text,
        })
direct_query_export = pd.DataFrame(
    direct_query_evidence, columns=direct_query_columns
).sort_values(
    ["page_group", "visual", "visual_id", "direct_query_ms"],
    ascending=[True, True, True, False],
    kind="mergesort",
).reset_index(drop=True)

priority_columns_export = [
    "rank", "severity", "page", "visual", "visual_type", "root_cause", "confidence",
    "priority_score", "dax_pattern_count", "dax_pattern_ids", "dax_pattern_tiers", "dax_measure_scope",
    "dax_pattern_next_step", "actionable_ms", "total_ms", "dax_ms", "direct_query_ms", "render_ms",
    "other_ms", "dax_evidence", "recommendation", "next_diagnostic", "source_url",
]
priority_export = visual_export[priority_columns_export].head(int(TOP_N)).rename(columns={
    "rank": "Rank", "severity": "Severity", "page": "Page", "visual": "Visual",
    "visual_type": "Visual type", "root_cause": "Likely root cause", "confidence": "Confidence",
    "priority_score": "Priority score", "dax_pattern_count": "DAX pattern count",
    "dax_pattern_ids": "DAX pattern candidates", "dax_pattern_tiers": "DAX pattern tiers",
    "dax_measure_scope": "DAX visibility", "dax_pattern_next_step": "DAX pattern next step",
    "actionable_ms": "Actionable ms", "total_ms": "Total ms", "dax_ms": "DAX ms",
    "direct_query_ms": "DirectQuery ms", "render_ms": "Render ms", "other_ms": "Other ms",
    "dax_evidence": "DAX text evidence", "recommendation": "Recommendation",
    "next_diagnostic": "Next diagnostic", "source_url": "Source",
})
page_report = page_export.rename(columns={
    "page": "Page", "visual_count": "Visuals", "total_actionable_ms": "Summed actionable ms",
    "max_actionable_ms": "Max visual ms", "median_actionable_ms": "Median visual ms",
    "slow_visual_count": "Critical/slow visuals", "total_other_ms": "Summed Other ms",
    "dominant_root_cause": "Most common root cause", "over_visual_limit": "Over visual limit",
})

thresholds = {
    "critical_visual_ms": int(CRITICAL_VISUAL_MS),
    "max_visuals_per_page": int(MAX_VISUALS_PER_PAGE),
    "review_visual_ms": int(REVIEW_VISUAL_MS),
    "slow_dax_ms": int(SLOW_DAX_MS),
    "slow_direct_query_ms": int(SLOW_DIRECT_QUERY_MS),
    "direct_query_review_ms": int(DIRECT_QUERY_REVIEW_MS),
    "slow_render_ms": int(SLOW_RENDER_MS),
    "slow_visual_ms": int(SLOW_VISUAL_MS),
    "top_n": int(TOP_N),
}
threshold_frame = pd.DataFrame(
    [{"Setting": key, "Value": value} for key, value in sorted(thresholds.items())]
)
severity_text = diagnostics["severity"].astype(str)
slow_visuals = int(severity_text.isin(["Critical", "Slow"]).sum())
slowest_row = visual_export.sort_values(
    ["actionable_ms", "rank"], ascending=[False, True], kind="mergesort"
).iloc[0]
kpi_records = [
    {"Metric": "Visuals analyzed", "Value": int(len(visual_export))},
    {"Metric": "Pages analyzed", "Value": int(visual_export["page"].nunique())},
    {"Metric": "Critical / slow visuals", "Value": slow_visuals},
    {"Metric": "Slowest actionable visual", "Value": str(slowest_row["visual"])},
    {"Metric": "Slowest actionable ms", "Value": float(slowest_row["actionable_ms"])},
    {"Metric": "Pages over visual limit", "Value": int(page_export["over_visual_limit"].sum())},
    {"Metric": "Captured DAX queries", "Value": int(len(dax_export))},
    {"Metric": "DirectQuery queries over threshold", "Value": int(len(direct_query_export))},
    {"Metric": "Trace data quality", "Value": str(run_summary["quality_status"])},
    {"Metric": "Structured findings", "Value": int(len(findings_frame))},
]
kpi_frame = pd.DataFrame(kpi_records)
root_cause_frame = (
    visual_export["root_cause"].astype(str).value_counts().rename_axis("Root cause").reset_index(name="Visuals")
    .sort_values(["Visuals", "Root cause"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

visual_csv_path = output_dir / "visual_diagnostics.csv"
page_csv_path = output_dir / "page_summary.csv"
dax_csv_path = output_dir / "dax_queries.csv"
dax_pattern_csv_path = output_dir / "dax_pattern_findings.csv"
direct_query_csv_path = output_dir / "direct_query_queries.csv"
json_path = output_dir / "analysis_report.json"
markdown_path = output_dir / "analysis_report.md"
html_path = output_dir / "analysis_report.html"

for export_frame, export_path in [
    (visual_export, visual_csv_path),
    (page_export, page_csv_path),
    (dax_export, dax_csv_path),
    (dax_pattern_export, dax_pattern_csv_path),
    (direct_query_export, direct_query_csv_path),
]:
    sanitize_csv_frame(export_frame).to_csv(
        export_path, index=False, encoding="utf-8", lineterminator="\n"
    )

# Blueprint normalized outputs (sections 8, 18, 21): lossless events plus event-level tables.
events_csv_path = output_dir / "events.csv"
events_parquet_path = output_dir / "events.parquet"
visual_updates_csv_path = output_dir / "visual-updates.csv"
dax_events_csv_path = output_dir / "dax-queries.csv"
direct_events_csv_path = output_dir / "direct-queries.csv"
quality_issues_json_path = output_dir / "quality-issues.json"
findings_json_path = output_dir / "findings.json"
run_summary_json_path = output_dir / "run-summary.json"
index_html_path = output_dir / "index.html"
manifest_json_path = output_dir / "manifest.json"
for export_frame, export_path in [
    (event_model["events"], events_csv_path),
    (visual_update_frame, visual_updates_csv_path),
    (dax_event_frame, dax_events_csv_path),
    (direct_event_frame, direct_events_csv_path),
]:
    sanitize_csv_frame(export_frame).to_csv(export_path, index=False, encoding="utf-8", lineterminator="\n")
if PYARROW_AVAILABLE:
    event_model["events"].to_parquet(events_parquet_path, index=False)


def write_json_artifact(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(json_safe_value(payload), ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


provenance = {**RULESET_VERSIONS, "run_id": run_summary["run_id"], "source_file_name": run_summary["source_file_name"],
              "source_sha256": run_summary["source_sha256"], "validation_mode": VALIDATION_MODE, "query_text_mode": QUERY_TEXT_MODE}
write_json_artifact(run_summary_json_path, run_summary)
write_json_artifact(quality_issues_json_path, {
    **provenance, "quality_status": run_summary["quality_status"], "input_assessment": run_summary["input_assessment"],
    "issue_counts_by_rule": run_summary["issue_counts_by_rule"], "rules": QUALITY_RULES,
    "issues": dataframe_to_records(event_model["quality_issues"]),
})
write_json_artifact(findings_json_path, {**provenance, "findings": dataframe_to_records(findings_frame)})

report_payload = {
    "dax_evidence": dataframe_to_records(dax_export),
    "dax_pattern_findings": dataframe_to_records(dax_pattern_export),
    "direct_query_queries": dataframe_to_records(direct_query_export),
    "direct_query_review_metadata": {
        "threshold_ms": int(DIRECT_QUERY_REVIEW_MS),
        "comparison": "strictly greater than",
    },
    "kpis": dataframe_to_records(kpi_frame),
    "page_summary": dataframe_to_records(page_export),
    "parse_metadata": json_safe_value(parse_metadata),
    "provenance": json_safe_value(provenance),
    "run_summary": json_safe_value(run_summary),
    "quality_issues": dataframe_to_records(event_model["quality_issues"]),
    "findings": dataframe_to_records(findings_frame),
    "visual_updates": dataframe_to_records(visual_update_frame),
    "dax_query_events": dataframe_to_records(dax_event_frame),
    "direct_query_events": dataframe_to_records(direct_event_frame),
    "remediation_priorities": dataframe_to_records(priority_export),
    "report_findings": dataframe_to_records(findings_export),
    "root_cause_summary": dataframe_to_records(root_cause_frame),
    "source": str(source_label),
    "thresholds": thresholds,
    "title": "Power BI Performance Analyzer Diagnostics",
    "visual_diagnostics": dataframe_to_records(visual_export),
}
json_path.write_text(
    json.dumps(
        report_payload,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

if dax_pattern_export.empty:
    markdown_dax_pattern_section = (
        "_No text-shape candidate was inferred; model definitions and execution traces remain required._"
    )
    html_dax_pattern_section = (
        '<p class="empty">No text-shape candidate was inferred; model definitions and execution traces remain required.</p>'
    )
else:
    markdown_dax_pattern_section = markdown_table(dax_pattern_export)
    html_dax_pattern_section = html_table(dax_pattern_export)

markdown_dax_sections = []
for record in dataframe_to_records(dax_export):
    heading = f"### Rank {record['rank']}: {markdown_escape(record['page'])} | {markdown_escape(record['visual'])}"
    metadata_line = (
        f"Severity: **{markdown_escape(record['severity'])}** | "
        f"DAX ms: **{markdown_escape(record['dax_ms'])}** | "
        f"Signature: `{markdown_escape(record['query_signature'])}` | "
        f"Repeat count: **{markdown_escape(record['repeated_query_count'])}**"
    )
    evidence_line = f"Text heuristic: {markdown_escape(record['dax_evidence'])}"
    markdown_dax_sections.append(
        "\n\n".join([heading, metadata_line, evidence_line, markdown_fenced_code(str(record["dax_query"]))])
    )
if not markdown_dax_sections:
    markdown_dax_sections.append("_No DAX query text was present in the parsed export._")

markdown_direct_query_sections = []
for record in dataframe_to_records(direct_query_export):
    query_text = str(record["native_query_text"] or "DirectQuery text was not captured")
    heading = (
        f"### {markdown_escape(record['page_group'])} | {markdown_escape(record['visual'])}"
    )
    metadata_line = (
        f"Type: **{markdown_escape(record['evidence_type'])}** | "
        f"Visual type: **{markdown_escape(record['visual_type'])}** | "
        f"DirectQuery ms: **{markdown_escape(record['direct_query_ms'])}** | "
        f"Visual DAX/query/actionable/total ms: "
        f"**{markdown_escape(record['visual_dax_ms'])} / "
        f"{markdown_escape(record['visual_query_ms'])} / "
        f"{markdown_escape(record['visual_actionable_ms'])} / "
        f"{markdown_escape(record['visual_total_ms'])}** | "
        f"Capture status: **{markdown_escape(record['capture_status'])}**"
    )
    markdown_direct_query_sections.append(
        "\n\n".join([heading, metadata_line, markdown_fenced_code(query_text, "text")])
    )
if not markdown_direct_query_sections:
    markdown_direct_query_sections.append(
        "_No individual DirectQuery execution exceeded the strict review threshold._"
    )

# Blueprint report pages shared by Markdown and HTML (section 14)
METHODOLOGY_ITEMS = [
    ("Event type", "Events are classified by the exact `(component, name)` pair. Power BI 1.1.0 exports that omit `component` get an inferred component (documented name, then parent component), reported as EVENT_REQUIRED_FIELD_MISSING warnings. Names outside the documented catalog are retained and reported as UNKNOWN_EVENT_TYPE; pane-style names (for example `DAX Query`, `Visual display`) fall back to name matching."),
    ("Duration", "`duration_ms = end - start` (wall-clock elapsed, including queueing). Events without `end` are instantaneous or incomplete and have unknown duration, never zero. Negative durations are flagged and excluded from covered time; raw timestamps are never modified."),
    ("Covered time", "Category time within a visual is the union of that category's intervals, intersected with the visual lifecycle interval (`covered_ms`). Overlapping DAX or DirectQuery events are never summed; `dax_descendant_sum_ms` is shown only as a work-volume indicator and can exceed elapsed time."),
    ("Visual timing", "`total_ms` is the Visual Container Lifecycle elapsed time when available (otherwise derived from components, labeled in `total_ms_source`). For event traces, `query_ms` is the union of DAX and DirectQuery intervals (equal to max(DAX, DirectQuery) when DirectQuery runs inside DAX) and `actionable_ms` is the union of DAX, DirectQuery, Render, and Evaluated-parameter intervals (equal to query + render + parameters when they do not overlap). Flattened records use `max(DAX, DirectQuery) + render + parameters`. `other_ms` is exported Other time or `total_ms - actionable_ms`."),
    ("Derived other", "`derived_other_ms = lifecycle elapsed - union(Canvas query, Query generation, Result parsing, Render, Data View Transform, Geocoding, Evaluated parameters)`, clamped at zero. It approximates UI-thread waiting and is not a clean compute measure (DAX Studio excludes Other for this reason)."),
    ("Interactions", "`derived_interaction_id` assigns each root visual update to the nearest preceding User Action within INTERACTION_WINDOW_MS, skipping ambiguous ties. It is a temporal reporting heuristic, not a parentId relationship."),
    ("Cache and visibility", "No DSE/AS events under a visual means backend activity was not observed: a possible canvas cache hit or a non-query visual, not zero backend time. AS events are absent for models hosted in SQL Server Analysis Services, Power BI, or Azure Analysis Services. `Execute Direct Query` appears only for DirectQuery/composite paths, and its source text only when the user owns the model and the source is SQL."),
    ("Truncation", "Any `Metrics Truncated` event means event detail is incomplete (documented limits: 300 events per semantic query, 200 per DAX query). Lifecycle totals remain valid; affected visuals get Low confidence and a qualification."),
    ("Direct Lake", f"Storage mode for this run: {run_summary.get('storage_mode', 'not recorded')}. The export does not record storage mode, so it comes from CAPTURE_METADATA['storage_mode']. For Direct Lake on SQL analytics endpoints, DirectQuery events mean the query fell back to DirectQuery; they get the DL_FALLBACK root cause, fallback-specific remediation, and DIRECT_LAKE_FALLBACK findings. Direct Lake on OneLake never falls back, so DirectQuery events there point to non-Direct-Lake tables. Direct Lake loads (transcodes) columns into memory on first use, so DAX-led visuals get a cold-cache note unless cache_state is 'warm'."),
    ("Clock alignment", f"Children outside their parent interval beyond CLOCK_TOLERANCE_MS={CLOCK_TOLERANCE_MS} ms are flagged CHILD_OUTSIDE_PARENT; intervals are clipped only for covered-time presentation."),
    ("Findings", "Findings in findings.json carry a rule ID and version, evidence, impact, next diagnostic step, and an exact-versus-heuristic classification. Thresholds are configurable triage defaults, not Microsoft SLAs. Cache comparison requires multiple intentionally labeled runs; record `cache_state` in CAPTURE_METADATA and compare runs externally."),
    ("Privacy", f"QUERY_TEXT_MODE={QUERY_TEXT_MODE}. `redact` replaces literal values while keeping object names; `omit` removes query text from every output. Fingerprints are computed from the original text. Query text is escaped in HTML and never executed; the HTML report carries a restrictive Content-Security-Policy."),
    ("Versions", f"parser {PARSER_VERSION}; mapping {MAPPING_VERSION}; finding rules {FINDING_RULES_VERSION}; validation mode {VALIDATION_MODE}."),
]
mapping_frame = pd.DataFrame(
    [{"Component": component, "Event name": name, "Friendly category": spec["category"], "Has end": spec["has_end"],
      "Documented metrics": ", ".join(sorted(spec["metrics"]))} for (component, name), spec in KNOWN_EVENT_CATALOG.items()]
)
rule_catalog_frame = pd.DataFrame(
    [{"Rule": rule_id, "Default severity": spec["severity"], "Class": spec["class"], "Condition": spec["condition"]} for rule_id, spec in QUALITY_RULES.items()]
)
visual_update_display_columns = [
    ("visual_title", "Visual"), ("visual_type", "Type"), ("status", "Status"), ("total_elapsed_ms", "Total ms"),
    ("canvas_query_ms", "Canvas query ms"), ("dax_covered_ms", "DAX covered ms"), ("direct_query_covered_ms", "DirectQuery covered ms"),
    ("render_ms", "Render ms"), ("transform_ms", "Transform ms"), ("derived_other_ms", "Derived other ms"),
    ("derived_interaction_id", "Interaction"), ("flags", "Flags"), ("data_quality_status", "Quality"),
]


def visual_updates_display(frame: pd.DataFrame) -> pd.DataFrame:
    display_frame = pd.DataFrame()
    for column, label in visual_update_display_columns:
        if column.endswith("_ms"):
            display_frame[label] = frame[column].map(lambda value, col=column: format_ms(value, "unknown" if col in {"total_elapsed_ms", "derived_other_ms"} else "not observed"))
        else:
            display_frame[label] = frame[column].map(lambda value: "" if value is None or (isinstance(value, float) and not np.isfinite(value)) else str(value))
    return display_frame


ordered_visual_updates = visual_update_frame.assign(_sort=pd.to_numeric(visual_update_frame["total_elapsed_ms"], errors="coerce")).sort_values(
    ["_sort", "visual_event_id"], ascending=[False, True], na_position="last", kind="mergesort").drop(columns="_sort")
top_visual_updates_display = visual_updates_display(ordered_visual_updates.head(int(SUMMARY_TOP_N)))
dax_event_display_columns = ["dax_event_id", "visual_title", "duration_ms", "row_count", "error", "canceled", "query_fingerprint",
                             "query_text_available", "as_event_count", "direct_query_count", "direct_query_covered_ms", "is_truncated", "derived_interaction_id"]
direct_event_display_columns = ["direct_query_event_id", "visual_title", "duration_ms", "actual_query_duration_ms", "data_read_duration_ms",
                                "rows_read", "is_capabilities_query", "nested_in_direct_query", "query_fingerprint", "query_text_available"]
ordered_dax_events = dax_event_frame.assign(_sort=pd.to_numeric(dax_event_frame["duration_ms"], errors="coerce")).sort_values(
    ["_sort", "dax_event_id"], ascending=[False, True], na_position="last", kind="mergesort").drop(columns="_sort")
ordered_direct_events = direct_event_frame.assign(_sort=pd.to_numeric(direct_event_frame["duration_ms"], errors="coerce")).sort_values(
    ["_sort", "direct_query_event_id"], ascending=[False, True], na_position="last", kind="mergesort").drop(columns="_sort")
findings_display = findings_frame[["finding_id", "priority", "scope_type", "observation", "next_step", "classification"]]
schema_drift_issues = event_model["quality_issues"].loc[event_model["quality_issues"]["rule_class"].eq("Schema drift")]
quality_banner_text = (
    f"**Trace data quality: {run_summary['quality_status']}** - {run_summary['input_assessment']}. "
    f"Validation mode: {VALIDATION_MODE}; strict schema valid: {run_summary['strict_schema_valid']}."
)
markdown_methodology = "\n".join(f"- **{markdown_escape(title)}:** {markdown_escape(text)}" for title, text in METHODOLOGY_ITEMS)
markdown_blueprint_sections = [
    "## Run summary\n\n" + quality_banner_text + "\n\n" + markdown_table(run_summary_frame)
    + f"\n\n### Top {int(SUMMARY_TOP_N)} visuals by lifecycle elapsed\n\n" + markdown_table(top_visual_updates_display)
    + f"\n\n### Top {int(SUMMARY_TOP_N)} DAX queries by elapsed\n\n" + markdown_table(ordered_dax_events[dax_event_display_columns].head(int(SUMMARY_TOP_N))),
    "## Visuals\n\n_\"not observed\" means no event of that type was captured; \"unknown\" means lifecycle duration is unavailable. Neither is zero._\n\n"
    + markdown_table(visual_updates_display(ordered_visual_updates)),
    "## Queries\n\n### DAX query events\n\n" + markdown_table(ordered_dax_events[dax_event_display_columns])
    + "\n\n### DirectQuery source query events\n\n" + markdown_table(ordered_direct_events[direct_event_display_columns])
    + "\n\n_Full query text (subject to QUERY_TEXT_MODE) is in dax-queries.csv, direct-queries.csv, and the HTML report._",
    "## Timeline\n\n_The interactive timeline is in analysis_report.html (Timeline section)._",
    "## Structured findings\n\n" + markdown_table(findings_display),
    "## Data quality and methodology\n\n### Issues by rule\n\n" + markdown_table(issue_rule_summary)
    + "\n\n### Schema drift\n\n" + markdown_table(schema_drift_issues)
    + "\n\n### Methodology\n\n" + markdown_methodology
    + "\n\n### Event mapping\n\n" + markdown_table(mapping_frame)
    + "\n\n### Quality rule catalog\n\n" + markdown_table(rule_catalog_frame),
]

markdown_reference_section = "\n".join([
    "## References",
    f"- DAX performance decision guide: {SOURCE_URLS['dax_decision_guide']}",
    f"- DAX performance pattern catalog: {SOURCE_URLS['dax_pattern_catalog']}",
])
markdown_report = "\n\n".join([
    "# Power BI Performance Analyzer Diagnostics",
    f"**Source:** {markdown_escape(source_label)} | **SHA-256:** `{run_summary['source_sha256']}` | "
    f"**Parser/mapping/rules:** {PARSER_VERSION} / {MAPPING_VERSION} / {FINDING_RULES_VERSION}",
    quality_banner_text,
    markdown_blueprint_sections[0],
    "## KPI summary\n\n" + markdown_table(kpi_frame),
    "## Thresholds\n\n" + markdown_table(threshold_frame),
    "## Root-cause summary\n\n" + markdown_table(root_cause_frame),
    f"## Root-cause and remediation priorities (top {int(TOP_N)})\n\n" + render_priorities_markdown(priority_export),
    "## Page summary\n\n" + markdown_table(page_report),
    "## Report findings\n\n" + markdown_table(findings_export),
    "## DAX performance pattern candidates\n\n" + markdown_dax_pattern_section,
    "## Full DAX evidence\n\n" + "\n\n".join(markdown_dax_sections),
    "## DirectQuery queries over 5 seconds\n\n" + "\n\n".join(markdown_direct_query_sections),
    *markdown_blueprint_sections[1:],
    markdown_reference_section,
]) + "\n"
markdown_path.write_text(markdown_report, encoding="utf-8")

html_dax_sections = []
for record in dataframe_to_records(dax_export):
    html_dax_sections.append(
        '<article class="dax-evidence">'
        f"<h3>Rank {html.escape(report_value(record['rank']))}: "
        f"{html.escape(report_value(record['page']))} | {html.escape(report_value(record['visual']))}</h3>"
        f"<p><strong>Severity:</strong> {html.escape(report_value(record['severity']))} &nbsp; "
        f"<strong>DAX ms:</strong> {html.escape(report_value(record['dax_ms']))} &nbsp; "
        f"<strong>Signature:</strong> {html.escape(report_value(record['query_signature']))} &nbsp; "
        f"<strong>Repeat count:</strong> {html.escape(report_value(record['repeated_query_count']))}</p>"
        f"<p><strong>Text heuristic:</strong> {html.escape(report_value(record['dax_evidence']))}</p>"
        f"<pre><code>{html.escape(str(record['dax_query']))}</code></pre>"
        "</article>"
    )
if not html_dax_sections:
    html_dax_sections.append('<p class="empty">No DAX query text was present in the parsed export.</p>')

html_direct_query_sections = []
for record in dataframe_to_records(direct_query_export):
    query_text = str(record["native_query_text"] or "DirectQuery text was not captured")
    html_direct_query_sections.append(
        '<details class="direct-query-evidence">'
        f"<summary>{html.escape(report_value(record['page_group']))} | "
        f"{html.escape(report_value(record['visual']))} | "
        f"{html.escape(report_value(record['direct_query_ms']))} ms</summary>"
        f"<p><strong>Type:</strong> {html.escape(report_value(record['evidence_type']))} &nbsp; "
        f"<strong>Visual type:</strong> {html.escape(report_value(record['visual_type']))} &nbsp; "
        f"<strong>Capture status:</strong> {html.escape(report_value(record['capture_status']))}</p>"
        f"<p><strong>Visual DAX/query/actionable/total ms:</strong> "
        f"{html.escape(report_value(record['visual_dax_ms']))} / "
        f"{html.escape(report_value(record['visual_query_ms']))} / "
        f"{html.escape(report_value(record['visual_actionable_ms']))} / "
        f"{html.escape(report_value(record['visual_total_ms']))}</p>"
        f"<pre><code>{html.escape(query_text)}</code></pre>"
        "</details>"
    )
if not html_direct_query_sections:
    html_direct_query_sections.append(
        '<p class="empty">No individual DirectQuery execution exceeded the strict review threshold.</p>'
    )

kpi_cards = "".join(
    '<div class="kpi">'
    f"<span>{html.escape(report_value(record['Metric']))}</span>"
    f"<strong>{html.escape(report_value(record['Value']))}</strong>"
    "</div>"
    for record in dataframe_to_records(kpi_frame)
)
html_reference_section = html_disclosure_section(
    "References",
    "<ul>"
    f'<li>DAX performance decision guide: <a href="{html.escape(SOURCE_URLS["dax_decision_guide"], quote=True)}">'
    f'{html.escape(SOURCE_URLS["dax_decision_guide"])}</a></li>'
    f'<li>DAX performance pattern catalog: <a href="{html.escape(SOURCE_URLS["dax_pattern_catalog"], quote=True)}">'
    f'{html.escape(SOURCE_URLS["dax_pattern_catalog"])}</a></li>'
    "</ul>",
    summary_meta="2 links",
)
children_index = _event_children(event_model)
html_tree_sections = []
for _, update in ordered_visual_updates.iterrows():
    title = visual_display_title(update)
    html_tree_sections.append(
        f'<details class="visual-tree" id="{html_anchor("tree", update["visual_event_id"])}">'
        f"<summary>{html.escape(str(title))} | {html.escape(format_ms(update['total_elapsed_ms'], 'unknown'))} ms | "
        f"{html.escape(str(update['status']))} | {html.escape(str(update['backend_activity']))}</summary>"
        f'<div class="timeline-wrap small">{render_timeline_svg(event_model, visual_event_id=update["visual_event_id"], width=1000)}</div>'
        f"{render_event_tree_html(event_model, update['visual_event_id'], children_index)}</details>"
    )
if not html_tree_sections:
    html_tree_sections.append('<p class="empty">No Visual Container Lifecycle events were captured.</p>')

html_query_blocks = []
for kind, frame, id_column, columns in [
    ("DAX", ordered_dax_events, "dax_event_id", dax_event_display_columns),
    ("DirectQuery source", ordered_direct_events, "direct_query_event_id", direct_event_display_columns),
]:
    blocks = []
    for record in dataframe_to_records(frame):
        code_id = html_anchor("q", record[id_column])
        text = str(record.get("query_text") or "")
        body = (
            f'<button type="button" class="copy" data-target="{code_id}">Copy</button><pre><code id="{code_id}">{html.escape(text)}</code></pre>'
            if text.strip() else '<p class="empty">Query text was not captured in the export.</p>'
        )
        blocks.append(
            f'<details class="query-evidence"><summary>{html.escape(kind)} {html.escape(str(record[id_column]))} | '
            f"{html.escape(str(record.get('visual_title') or record.get('visual_event_id') or 'unattributed'))} | "
            f"{html.escape(format_ms(record.get('duration_ms'), 'unknown'))} ms | fingerprint {html.escape(str(record.get('query_fingerprint') or 'n/a'))}</summary>{body}</details>"
        )
    html_query_blocks.append(
        f"<h3>{html.escape(kind)} query events ({len(frame):,})</h3>" + html_table(frame[columns])
        + ("".join(blocks) if blocks else "")
    )

html_methodology = "<dl class=\"methodology\">" + "".join(
    f"<dt>{html.escape(title)}</dt><dd>{html.escape(text)}</dd>" for title, text in METHODOLOGY_ITEMS
) + "</dl>"
html_sections = [
    f'<nav class="toc" aria-label="Report sections"><a href="#run-summary">Run summary</a> <a href="#visuals">Visuals</a> '
    f'<a href="#queries">Queries</a> <a href="#timeline">Timeline</a> <a href="#data-quality">Data quality and methodology</a> '
    f'<a href="#priorities">Priorities</a></nav>',
    render_quality_banner(run_summary),
    html_disclosure_section(
        "Run summary",
        html_table(run_summary_frame)
        + f"<h3>Top {int(SUMMARY_TOP_N)} visuals by lifecycle elapsed</h3>" + html_table(top_visual_updates_display)
        + f"<h3>Top {int(SUMMARY_TOP_N)} DAX queries by elapsed</h3>" + html_table(ordered_dax_events[dax_event_display_columns].head(int(SUMMARY_TOP_N))),
        open_by_default=True, summary_meta=f"{run_summary['event_count']:,} events | {run_summary['visual_update_count']:,} visual updates",
        section_id="run-summary",
    ),
    html_disclosure_section(
        "KPI summary",
        f'<div class="kpi-grid">{kpi_cards}</div>',
        open_by_default=True,
        summary_meta=f"{len(kpi_frame)} metrics",
    ),
    html_disclosure_section(
        "Visuals",
        render_visuals_table_html(visual_update_frame),
        summary_meta=f"{len(visual_update_frame)} visual updates", section_id="visuals",
    ),
    html_disclosure_section(
        "Visual event trees",
        "".join(html_tree_sections),
        summary_meta=f"{len(visual_update_frame)} trees", section_id="event-trees",
    ),
    html_disclosure_section(
        "Queries",
        "".join(html_query_blocks),
        summary_meta=f"{len(dax_event_frame)} DAX | {len(direct_event_frame)} DirectQuery", section_id="queries",
    ),
    html_disclosure_section(
        "Timeline",
        f'<div class="timeline-wrap">{render_timeline_svg(event_model)}</div>'
        '<p class="note">Lanes are visual updates (sub-rows by event depth); colors are friendly categories. Diamonds mark instantaneous or '
        'unknown-duration events, red triangles mark Metrics Truncated, and red outlines mark clock anomalies or negative durations. Hover for details.</p>',
        summary_meta=f"{run_summary['analyzable_event_count']:,} events", section_id="timeline",
    ),
    html_disclosure_section(
        "Structured findings",
        html_table(findings_display),
        summary_meta=f"{len(findings_frame)} findings",
    ),
    html_disclosure_section(
        "Thresholds",
        html_table(threshold_frame),
        summary_meta=f"{len(threshold_frame)} settings",
    ),
    html_disclosure_section(
        "Root-cause summary",
        html_table(root_cause_frame),
        summary_meta=f"{len(root_cause_frame)} categories",
    ),
    html_disclosure_section(
        f"Root-cause and remediation priorities (top {int(TOP_N)})",
        render_priorities_html(priority_export),
        summary_meta=f"{len(priority_export)} items", section_id="priorities",
    ),
    html_disclosure_section(
        "Page summary",
        html_table(page_report),
        summary_meta=f"{len(page_report)} pages",
    ),
    html_disclosure_section(
        "Report findings",
        html_table(findings_export),
        summary_meta=f"{len(findings_export)} items",
    ),
    html_disclosure_section(
        "DAX performance pattern candidates",
        html_dax_pattern_section,
        summary_meta=f"{len(dax_pattern_export)} items",
    ),
    html_disclosure_section(
        "Full DAX evidence",
        "".join(html_dax_sections),
        summary_meta=f"{len(dax_export)} queries",
    ),
    html_disclosure_section(
        "DirectQuery queries over 5 seconds",
        "".join(html_direct_query_sections),
        summary_meta=f"{len(direct_query_export)} queries",
    ),
    html_disclosure_section(
        "Data quality and methodology",
        "<h3>Issues by rule</h3>" + html_table(issue_rule_summary)
        + "<h3>Issue detail</h3>" + html_table(event_model["quality_issues"])
        + "<h3>Schema drift (unknown event types and metrics)</h3>" + html_table(schema_drift_issues)
        + "<h3>Methodology and formulas</h3>" + html_methodology
        + "<h3>Event mapping</h3>" + html_table(mapping_frame)
        + "<h3>Quality rule catalog</h3>" + html_table(rule_catalog_frame),
        summary_meta=f"{run_summary['quality_status']} | {sum(run_summary['issue_counts_by_rule'].values()):,} issue occurrence(s)",
        section_id="data-quality",
    ),
    html_reference_section,
]
html_style = """
:root { color-scheme: light; --ink: #172033; --muted: #566075; --line: #d8dee9; --panel: #f6f8fb; --accent: #087f5b; --accent-2: #2457a6; }
* { box-sizing: border-box; }
body { margin: 0; background: #ffffff; color: var(--ink); font-family: "Segoe UI", Tahoma, sans-serif; line-height: 1.5; }
main { width: min(1500px, 96vw); margin: 0 auto; padding: 32px 0 64px; }
h1, h3 { line-height: 1.2; }
h1 { margin-bottom: 8px; }
.source { color: var(--muted); overflow-wrap: anywhere; }
.toc { display: flex; flex-wrap: wrap; gap: 14px; margin: 12px 0; }
.toc a { color: var(--accent-2); }
.quality-banner { margin: 16px 0; padding: 12px 16px; border-left: 6px solid #64748b; background: var(--panel); }
.quality-banner.pass { border-color: #15803d; background: #f0fdf4; }
.quality-banner.warn { border-color: #b45309; background: #fffbeb; }
.quality-banner.fail { border-color: #b91c1c; background: #fef2f2; }
.kpi-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 12px; }
.kpi { min-height: 98px; padding: 14px; border: 1px solid var(--line); border-top: 4px solid var(--accent); background: var(--panel); }
.kpi span { display: block; color: var(--muted); font-size: 0.82rem; }
.kpi strong { display: block; margin-top: 8px; font-size: 1.25rem; overflow-wrap: anywhere; }
.table-wrap { overflow-x: auto; border: 1px solid var(--line); }
.table-controls { display: flex; flex-wrap: wrap; gap: 12px; margin: 8px 0; }
.table-controls input, .table-controls select { margin-left: 4px; padding: 4px 6px; }
table { width: 100%; border-collapse: collapse; font-size: 0.86rem; }
th, td { padding: 9px 10px; border-bottom: 1px solid var(--line); text-align: left; vertical-align: top; }
th { overflow-wrap: normal; word-break: normal; hyphens: none; min-width: 7ch; }
td { overflow-wrap: break-word; max-width: 60ch; }
table.compact td, table.compact th { white-space: nowrap; }
table.compact td:nth-child(3), table.compact td:nth-child(4) { white-space: normal; min-width: 18ch; }
.muted { color: var(--muted); font-size: 0.82rem; }
.sev { display: inline-block; padding: 1px 8px; border-radius: 10px; font-size: 0.78rem; font-weight: 700; }
.sev-critical .sev, .sev.sev-critical { background: #fee2e2; color: #991b1b; }
.sev-slow .sev, .sev.sev-slow { background: #ffedd5; color: #9a3412; }
.sev-review .sev, .sev.sev-review { background: #fef9c3; color: #854d0e; }
.sev-good .sev, .sev.sev-good { background: #dcfce7; color: #166534; }
.priority-card { margin: 10px 0; padding: 10px 14px; border: 1px solid var(--line); border-left: 6px solid #94a3b8; background: #fff; }
.priority-card.sev-critical { border-left-color: #b91c1c; } .priority-card.sev-slow { border-left-color: #ea580c; }
.priority-card.sev-review { border-left-color: #ca8a04; } .priority-card.sev-good { border-left-color: #15803d; }
.priority-card > summary { cursor: pointer; line-height: 1.6; }
.priority-card .rank { font-weight: 700; color: var(--accent-2); }
.priority-card h4 { margin: 12px 0 4px; }
.priority-card p { margin: 0 0 6px; max-width: 110ch; }
.chip-row { display: flex; flex-wrap: wrap; gap: 8px; margin: 10px 0 6px; }
.chip { min-width: 110px; padding: 6px 10px; border: 1px solid var(--line); background: var(--panel); }
.chip span { display: block; color: var(--muted); font-size: 0.75rem; }
.chip strong { font-size: 1.05rem; }
svg.priority-bar { display: block; max-width: 100%; height: auto; margin: 4px 0 6px; }
dl.kv { display: grid; grid-template-columns: minmax(150px, 220px) 1fr; gap: 4px 12px; margin: 0; }
dl.kv dt { font-weight: 600; } dl.kv dd { margin: 0; }
th { position: sticky; top: 0; background: #e9eef6; color: #1d3557; }
th.sortable { cursor: pointer; }
th[aria-sort="ascending"]::after { content: " \\25B2"; }
th[aria-sort="descending"]::after { content: " \\25BC"; }
td.num { text-align: right; white-space: nowrap; }
tbody tr:nth-child(even) { background: var(--panel); }
.empty, .note { color: var(--muted); font-style: italic; }
.report-section { width: 100%; margin: 24px 0 0; border-top: 2px solid var(--line); }
.report-section > summary { cursor: pointer; padding: 14px 4px 10px; color: var(--ink); font-size: 1.45rem; font-weight: 650; line-height: 1.2; }
.report-section > summary:focus-visible { outline: 3px solid var(--accent-2); outline-offset: 3px; }
.report-section > summary .summary-meta { margin-left: 10px; color: var(--muted); font-size: 0.82rem; font-weight: 400; white-space: nowrap; }
.report-section-body { padding: 8px 0 12px; }
.dax-evidence { margin: 18px 0; padding: 16px; border-left: 4px solid var(--accent-2); background: var(--panel); }
.dax-evidence h3 { margin-top: 0; }
.direct-query-evidence, .query-evidence, .visual-tree { margin: 10px 0; padding: 10px 14px; border-left: 4px solid var(--accent); background: var(--panel); }
.direct-query-evidence summary, .query-evidence summary, .visual-tree summary { cursor: pointer; font-weight: 700; overflow-wrap: anywhere; }
.timeline-wrap { max-height: 720px; overflow: auto; border: 1px solid var(--line); background: #fff; }
.timeline-wrap.small { max-height: 320px; margin: 8px 0; }
svg.timeline { width: 100%; min-width: 900px; height: auto; }
table.interactive { min-width: 1200px; }
table.interactive th { overflow-wrap: normal; }
.event-tree, .event-tree ul { list-style: none; margin: 0; padding-left: 18px; border-left: 1px dashed var(--line); }
.event-tree li { margin: 3px 0; }
.evt-key { font-weight: 600; }
.evt-ms { color: var(--accent-2); }
.evt-id { color: var(--muted); font-size: 0.78rem; }
.evt-metrics { color: var(--muted); font-family: Consolas, "Courier New", monospace; font-size: 0.78rem; overflow-wrap: anywhere; }
.badge { display: inline-block; padding: 0 6px; border-radius: 8px; background: #fee2e2; color: #991b1b; font-size: 0.72rem; }
.methodology dt { font-weight: 700; margin-top: 10px; }
.methodology dd { margin: 2px 0 0 0; }
button.copy { margin: 6px 0; padding: 3px 10px; cursor: pointer; }
pre { max-height: 520px; overflow: auto; padding: 14px; background: #111827; color: #f8fafc; white-space: pre-wrap; overflow-wrap: anywhere; }
code { font-family: Consolas, "Courier New", monospace; }
@media (max-width: 700px) { main { width: 94vw; padding-top: 20px; } .report-section > summary { font-size: 1.2rem; } .report-section > summary .summary-meta { display: block; margin: 5px 0 0 1.2rem; white-space: normal; } th, td { padding: 7px; } }
"""
html_script = """
(function () {
  function tableRows(id) { var table = document.getElementById(id); return table ? Array.prototype.slice.call(table.tBodies[0].rows) : []; }
  function applyFilters(id) {
    var search = document.querySelector('.js-search[data-table="' + id + '"]');
    var term = search ? search.value.trim().toLowerCase() : '';
    var filters = Array.prototype.slice.call(document.querySelectorAll('.js-filter[data-table="' + id + '"]'));
    tableRows(id).forEach(function (row) {
      var visible = !term || (row.getAttribute('data-search') || '').indexOf(term) !== -1;
      filters.forEach(function (filter) {
        if (!filter.value) { return; }
        var actual = row.getAttribute('data-' + filter.getAttribute('data-attr')) || '';
        visible = visible && (filter.getAttribute('data-mode') === 'contains' ? actual.indexOf(filter.value) !== -1 : actual === filter.value);
      });
      row.hidden = !visible;
    });
  }
  function onFilter(event) { var id = event.target.getAttribute && event.target.getAttribute('data-table'); if (id) { applyFilters(id); } }
  document.addEventListener('input', onFilter);
  document.addEventListener('change', onFilter);
  function sortBy(header) {
    var table = header.closest('table'), column = Number(header.getAttribute('data-col'));
    var ascending = header.getAttribute('aria-sort') !== 'ascending';
    Array.prototype.forEach.call(table.querySelectorAll('th'), function (cell) { cell.removeAttribute('aria-sort'); });
    header.setAttribute('aria-sort', ascending ? 'ascending' : 'descending');
    var body = table.tBodies[0], rows = Array.prototype.slice.call(body.rows);
    rows.sort(function (a, b) {
      var left = a.cells[column], right = b.cells[column], result;
      var leftSort = left.getAttribute('data-sort'), rightSort = right.getAttribute('data-sort');
      if (leftSort !== null && rightSort !== null) {
        result = (leftSort === '' ? -Infinity : Number(leftSort)) - (rightSort === '' ? -Infinity : Number(rightSort));
      } else {
        result = left.textContent.localeCompare(right.textContent);
      }
      return ascending ? result : -result;
    });
    rows.forEach(function (row) { body.appendChild(row); });
  }
  document.addEventListener('click', function (event) {
    var header = event.target.closest && event.target.closest('th.sortable');
    if (header) { sortBy(header); return; }
    var button = event.target.closest && event.target.closest('button.copy');
    if (button && navigator.clipboard) {
      var code = document.getElementById(button.getAttribute('data-target'));
      if (code) { navigator.clipboard.writeText(code.textContent).then(function () { button.textContent = 'Copied'; }); }
    }
  });
  document.addEventListener('keydown', function (event) {
    if (event.key === 'Enter' && event.target.matches && event.target.matches('th.sortable')) { sortBy(event.target); }
  });
  function openTarget() {
    if (!location.hash) { return; }
    var element = document.getElementById(decodeURIComponent(location.hash.slice(1)));
    while (element) { if (element.tagName === 'DETAILS') { element.open = true; } element = element.parentElement; }
  }
  window.addEventListener('hashchange', openTarget);
  openTarget();
})();
"""


def csp_source_hash(text: str) -> str:
    return "'sha256-" + base64.b64encode(hashlib.sha256(text.encode("utf-8")).digest()).decode("ascii") + "'"


content_security_policy = (
    f"default-src 'none'; style-src {csp_source_hash(html_style)}; script-src {csp_source_hash(html_script)}; "
    "img-src data:; base-uri 'none'; form-action 'none'"
)
assert not set(content_security_policy) & set('"<>&'), "CSP must be attribute-safe without escaping"
html_report = (
    "<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n"
    f'<meta http-equiv="Content-Security-Policy" content="{content_security_policy}">\n'
    '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
    "<title>Power BI Performance Analyzer Diagnostics</title>\n"
    f"<style>{html_style}</style>\n</head>\n<body>\n<main>\n"
    "<h1>Power BI Performance Analyzer Diagnostics</h1>\n"
    f'<p class="source"><strong>Source:</strong> {html.escape(str(source_label))} &nbsp; <strong>SHA-256:</strong> {html.escape(str(run_summary["source_sha256"]))} '
    f"&nbsp; <strong>Parser / mapping / rules:</strong> {html.escape(PARSER_VERSION)} / {html.escape(MAPPING_VERSION)} / {html.escape(FINDING_RULES_VERSION)} "
    f"&nbsp; <strong>Query text:</strong> {html.escape(QUERY_TEXT_MODE)}</p>\n"
    + "".join(html_sections)
    + f"\n</main>\n<script>{html_script}</script>\n</body>\n</html>\n"
)
html_path.write_text(html_report, encoding="utf-8")
index_html_path.write_text(html_report, encoding="utf-8")

exported_paths = [
    visual_csv_path,
    page_csv_path,
    dax_csv_path,
    dax_pattern_csv_path,
    direct_query_csv_path,
    json_path,
    markdown_path,
    html_path,
    index_html_path,
    events_csv_path,
    *([events_parquet_path] if PYARROW_AVAILABLE else []),
    visual_updates_csv_path,
    dax_events_csv_path,
    direct_events_csv_path,
    quality_issues_json_path,
    findings_json_path,
    run_summary_json_path,
]
manifest = {
    **provenance,
    "analysis_timestamp_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "quality_status": run_summary["quality_status"],
    "artifacts": [
        {"name": path.name, "bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
        for path in exported_paths
    ],
}
write_json_artifact(manifest_json_path, manifest)
exported_paths.append(manifest_json_path)
print("Exported report files:")
for exported_path in exported_paths:
    print(f"- {exported_path} ({exported_path.stat().st_size:,} bytes)")

## 10. Automated contract tests

These contract tests cover official and flattened schema variants, useful validation errors, duration math and severity thresholds, high-Other ranking behavior, legacy-versus-new Card classification, CSV/Markdown/HTML escaping, and deterministic report exports.

The blueprint fixture matrix (section 17) follows: all 20 required fixtures (valid minimal file through malicious strings), plus missing-end handling, 1.1.0 component inference and strict-mode rejection, temporal interaction attribution, interval-union math, opt-in status-code mapping, query-text redaction/omission, a golden overlap fixture, and report provenance/CSP checks. The regression cell then checks parser invariants on the current capture, or on every capture in `REGRESSION_CAPTURES_DIR`.

In [ ]:
def captured_value_error(payload: Any) -> str:
    try:
        parse_performance_analyzer(payload)
    except ValueError as exc:
        return str(exc)
    raise AssertionError("Expected parse_performance_analyzer to raise ValueError")


def test_official_parser_contract():
    frame, metadata = parse_performance_analyzer(build_synthetic_payload())
    assert metadata["format"] == "official-event-tree"
    assert len(frame) == 3
    assert frame.columns.tolist() == NORMALIZED_COLUMNS


def test_flattened_aliases_and_duration_math():
    payload = [{
        "PageName": "Overview",
        "VisualName": "Alias Visual",
        "VisualId": "alias-1",
        "DurationMs": 2600,
        "DAXQueryMs": 1200,
        "DirectQueryMs": 1500,
        "VisualDisplayMs": 200,
        "OtherMs": 900,
        "QueryText": "EVALUATE ROW(\"Value\", 1)",
    }]
    frame, metadata = parse_performance_analyzer(payload)
    row = frame.iloc[0]
    assert metadata["format"] == "flattened-records"
    assert row["query_ms"] == 1500
    assert row["query_ms"] != 1200 + 1500
    assert row["actionable_ms"] == 1500 + 200
    assert row["total_ms"] == 2600
    assert row["dax_query"] == payload[0]["QueryText"]


def test_invalid_payload_errors_are_actionable():
    scalar_error = captured_value_error("not an object or array")
    assert "object or array" in scalar_error

    empty_events_error = captured_value_error({"events": []})
    assert "empty" in empty_events_error.lower()
    assert "events" in empty_events_error.lower()


def test_severity_threshold_boundaries():
    def timing_row(actionable_ms):
        return pd.Series({
            "actionable_ms": actionable_ms,
            "dax_ms": 0,
            "direct_query_ms": 0,
            "render_ms": 0,
            "other_ms": 0,
            "total_ms": actionable_ms,
            "is_p90_outlier": False,
        })

    assert severity_for(timing_row(CRITICAL_VISUAL_MS)) == "Critical"
    assert severity_for(timing_row(SLOW_VISUAL_MS)) == "Slow"
    assert severity_for(timing_row(REVIEW_VISUAL_MS)) == "Review"
    assert severity_for(timing_row(REVIEW_VISUAL_MS - 1)) == "Good"


def test_relative_outlier_and_unknown_page_policy():
    rows = []
    for index in range(10):
        actionable_ms = 100.0 + index * 10
        rows.append({
            "page": "Unknown page",
            "visual": f"Visual {index}",
            "visual_id": f"visual-{index}",
            "visual_type": "card",
            "start_time": "",
            "total_ms": actionable_ms,
            "dax_ms": actionable_ms,
            "direct_query_ms": 0.0,
            "render_ms": 0.0,
            "parameter_ms": 0.0,
            "other_ms": 0.0,
            "query_ms": actionable_ms,
            "actionable_ms": actionable_ms,
            "dax_query": "",
            "source_event_count": 1,
            "raw_properties": "{}",
        })
    frame = pd.DataFrame(rows, columns=NORMALIZED_COLUMNS)
    analyzed, pages = apply_page_identity_guard(*analyze_visuals(frame))
    outlier = analyzed.loc[analyzed["actionable_ms"].idxmax()]
    assert bool(outlier["is_p90_outlier"])
    assert outlier["rule_id"] == "OUTLIER"
    assert str(outlier["severity"]) == "Review"
    assert not analyzed["page_over_limit"].any()
    assert not pages["over_visual_limit"].any()
    assert not analyzed["recommendation"].str.contains("exceeds the configured visible-visual limit", regex=False).any()


def test_synthetic_high_other_ranking():
    frame, _ = parse_performance_analyzer(build_synthetic_payload())
    synthetic_diagnostics, _ = analyze_visuals(frame)
    assert synthetic_diagnostics["visual"].tolist() == ["Sales Matrix", "Customer Map", "Revenue Card"]
    revenue_card = synthetic_diagnostics.loc[synthetic_diagnostics["visual"] == "Revenue Card"].iloc[0]
    assert revenue_card["actionable_ms"] == revenue_card["query_ms"] + revenue_card["render_ms"] + revenue_card["parameter_ms"]
    assert revenue_card["other_ms"] > revenue_card["actionable_ms"]
    assert revenue_card["rule_id"] == "OTHER"
    assert str(revenue_card["severity"]) == "Review"
    assert revenue_card["confidence"] == "Low"


def test_inspect_dax_iterator_and_table_filter():
    evidence = inspect_dax("SUMX(FILTER('Sales', 'Sales'[Amount] > 0), 'Sales'[Amount])")
    assert "Iterator functions" in evidence
    assert "Broad table FILTER" in evidence


def test_visible_measure_dax_candidates():
    table_filter_query = """DEFINE
    MEASURE 'Sales'[Positive Revenue] =
        CALCULATE(
            SUM('Sales'[Amount]),
            FILTER('Sales', 'Sales'[Amount] > 0)
        )
EVALUATE
ROW("Result", [Positive Revenue])"""
    table_filter_candidates = {
        candidate["pattern_id"]: candidate
        for candidate in detect_dax_performance_patterns(table_filter_query)
    }
    assert "DAX001" in table_filter_candidates
    assert table_filter_candidates["DAX001"]["tier"] == "Tier 1 - measure/UDF"
    assert table_filter_candidates["DAX001"]["approval_required"] is False

    context_transition_query = """DEFINE
    MEASURE 'Sales'[Revenue by Customer] =
        SUMX(
            VALUES('Customer'[CustomerKey]),
            CALCULATE(SUM('Sales'[Amount]))
        )
EVALUATE
ROW("Result", [Revenue by Customer])"""
    context_transition_ids = {
        candidate["pattern_id"]
        for candidate in detect_dax_performance_patterns(context_transition_query)
    }
    assert "DAX006/DAX008" in context_transition_ids

    divide_iterator_query = """DEFINE
    MEASURE 'Sales'[Weighted Unit Revenue] =
        SUMX(
            'Sales',
            DIVIDE('Sales'[Amount], 'Sales'[Quantity], 0)
        )
EVALUATE
ROW("Result", [Weighted Unit Revenue])"""
    divide_iterator_ids = {
        candidate["pattern_id"]
        for candidate in detect_dax_performance_patterns(divide_iterator_query)
    }
    assert "DAX018" in divide_iterator_ids
    assert dax_measure_scope(table_filter_query).startswith("1 visible measure definition(s)")


def test_generated_query_scope_and_sanitization():
    generated_query = """DEFINE
    VAR __ValueFilterDM =
        FILTER(VALUES('Product'[Category]), [Sales Amount] > 100)
EVALUATE
ROW("Matching categories", COUNTROWS(__ValueFilterDM))"""
    generated_candidates = detect_dax_performance_patterns(generated_query)
    generated_by_id = {candidate["pattern_id"]: candidate for candidate in generated_candidates}
    assert set(generated_by_id) == {"QRY002"}
    assert generated_by_id["QRY002"]["tier"] == "Tier 2 - query structure"
    assert generated_by_id["QRY002"]["approval_required"] is True
    assert not any(candidate["tier"].startswith("Tier 1") for candidate in generated_candidates)
    assert dax_measure_scope(generated_query).startswith("Generated query only")

    two_measure_query = """DEFINE
    MEASURE 'Sales'[Base Revenue] = SUM('Sales'[Amount])
    MEASURE 'Sales'[Double Revenue] = [Base Revenue] * 2
EVALUATE
ROW("Result", [Double Revenue])"""
    assert dax_measure_scope(two_measure_query).startswith("2 visible measure definition(s)")

    non_executable_patterns = """DEFINE
    VAR __HarmlessText =
        "DEFINE MEASURE 'Sales'[Fake] = SUMX(VALUES('Sales'[Key]), CALCULATE(DIVIDE(1, 1))) __ValueFilterDM"
    // MEASURE 'Sales'[Line Comment] = CALCULATE(1, FILTER('Sales', 'Sales'[Amount] > 0))
    -- SUMX(VALUES('Sales'[Key]), CALCULATE(DIVIDE(1, 1)))
    /* MEASURE 'Sales'[Block Comment] = SUMX('Sales', DIVIDE('Sales'[Amount], 1)) */
EVALUATE
ROW("Text", __HarmlessText)"""
    assert detect_dax_performance_patterns(non_executable_patterns) == []


def test_measure_locality_for_composite_patterns():
    split_patterns_query = """DEFINE
    MEASURE 'Sales'[Iterator Only] = SUMX('Sales', 'Sales'[Amount])
    MEASURE 'Sales'[Values Only] = COUNTROWS(VALUES('Customer'[CustomerKey]))
    MEASURE 'Sales'[Calculate Only] = CALCULATE(SUM('Sales'[Amount]))
    MEASURE 'Sales'[Divide Only] = DIVIDE(SUM('Sales'[Amount]), SUM('Sales'[Quantity]), 0)
EVALUATE
ROW("Result", [Iterator Only] + [Values Only] + [Calculate Only] + [Divide Only])"""
    pattern_ids = {
        candidate["pattern_id"]
        for candidate in detect_dax_performance_patterns(split_patterns_query)
    }
    assert "DAX006/DAX008" not in pattern_ids
    assert "DAX018" not in pattern_ids


def test_dax_candidates_do_not_change_actionable_ranking():
    frame, _ = parse_performance_analyzer(build_synthetic_payload())
    baseline_diagnostics, _ = analyze_visuals(frame)

    candidate_frame = frame.copy()
    target_visual_id = candidate_frame.iloc[0]["visual_id"]
    candidate_frame.loc[candidate_frame["visual_id"] == target_visual_id, "dax_query"] = """DEFINE
    MEASURE 'Sales'[Positive Revenue] =
        CALCULATE(
            SUM('Sales'[Amount]),
            FILTER('Sales', 'Sales'[Amount] > 0)
        )
EVALUATE
ROW("Result", [Positive Revenue])"""
    candidate_diagnostics, _ = analyze_visuals(candidate_frame)

    def actionable_snapshot(snapshot_frame):
        return [
            (int(row.rank), row.visual_id, float(row.actionable_ms), row.rule_id, str(row.severity))
            for row in snapshot_frame.itertuples(index=False)
        ]

    assert actionable_snapshot(candidate_diagnostics) == actionable_snapshot(baseline_diagnostics)
    changed_row = candidate_diagnostics.loc[candidate_diagnostics["visual_id"] == target_visual_id].iloc[0]
    assert "DAX001" in changed_row["dax_pattern_ids"]


def test_csv_formula_prefix_sanitization():
    dangerous = ["=1+1", "+cmd", "-2", "@SUM(A1:A2)", "\tformula", "\rformula"]
    frame = pd.DataFrame({
        "text": dangerous + ["normal"],
        "number": list(range(len(dangerous) + 1)),
    })
    sanitized = sanitize_csv_frame(frame)
    assert sanitized["text"].iloc[:6].tolist() == ["'" + value for value in dangerous]
    assert sanitized.loc[6, "text"] == "normal"
    assert sanitized["number"].tolist() == frame["number"].tolist()


def test_markdown_and_html_escaping():
    assert markdown_escape("A|B\nC") == "A\\|B<br>C"
    rendered = html_table(pd.DataFrame({"Value": ["<script>alert(1)</script> & safe"]}))
    assert "<script>" not in rendered
    assert "&lt;script&gt;alert(1)&lt;/script&gt; &amp; safe" in rendered


def test_deterministic_serializers_and_json_export():
    def serialize_report_json():
        return json.dumps(
            report_payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n"

    def serialize_csv(frame):
        return sanitize_csv_frame(frame).to_csv(index=False, lineterminator="\n")

    serializers = [
        serialize_report_json,
        lambda: markdown_table(priority_export),
        lambda: html_table(priority_export),
        lambda: serialize_csv(visual_export),
        lambda: serialize_csv(dax_pattern_export),
    ]
    for serializer in serializers:
        first_serialization = serializer().encode("utf-8")
        assert first_serialization == serializer().encode("utf-8")

    exported_json = json.loads(json_path.read_text(encoding="utf-8"))
    assert exported_json["remediation_priorities"] == dataframe_to_records(priority_export)
    assert exported_json["visual_diagnostics"] == dataframe_to_records(visual_export)
    assert exported_json["dax_pattern_findings"] == dataframe_to_records(dax_pattern_export)
    assert visual_csv_path.read_bytes() == serialize_csv(visual_export).encode("utf-8")
    assert dax_pattern_csv_path.read_bytes() == serialize_csv(dax_pattern_export).encode("utf-8")
    enriched_columns = {
        "dax_pattern_count",
        "dax_pattern_ids",
        "dax_pattern_tiers",
        "dax_measure_scope",
        "dax_pattern_next_step",
    }
    assert enriched_columns.issubset(visual_export.columns)
    assert dax_pattern_export.columns.tolist() == dax_pattern_columns


def test_dax_pattern_exports_and_references():
    assert dax_pattern_csv_path.exists()
    assert dax_pattern_export.columns.tolist() == dax_pattern_columns
    exported_json = json.loads(json_path.read_text(encoding="utf-8"))
    assert "dax_pattern_findings" in exported_json
    assert exported_json["dax_pattern_findings"] == dataframe_to_records(dax_pattern_export)
    if dax_pattern_export.empty:
        assert dax_pattern_csv_path.read_bytes() == (",".join(dax_pattern_columns) + "\n").encode("utf-8")

    reference_urls = [
        SOURCE_URLS["dax_decision_guide"],
        SOURCE_URLS["dax_pattern_catalog"],
    ]
    for report_text in (markdown_report, html_report):
        assert "DAX performance pattern candidates" in report_text
        for reference_url in reference_urls:
            assert reference_url in report_text


def test_standalone_html_has_no_external_assets():
    normalized_html = html_report.casefold()
    assert "<script src=" not in normalized_html
    assert "<link" not in normalized_html


contract_tests = [
    ("official parser contract", test_official_parser_contract),
    ("flattened aliases and duration math", test_flattened_aliases_and_duration_math),
    ("invalid payload errors", test_invalid_payload_errors_are_actionable),
    ("severity threshold boundaries", test_severity_threshold_boundaries),
    ("relative outlier and unknown-page policy", test_relative_outlier_and_unknown_page_policy),
    ("synthetic high-Other ranking", test_synthetic_high_other_ranking),
    ("DAX iterator and table-filter inspection", test_inspect_dax_iterator_and_table_filter),
    ("visible-measure DAX candidates", test_visible_measure_dax_candidates),
    ("generated-query scope and sanitization", test_generated_query_scope_and_sanitization),
    ("measure-local composite patterns", test_measure_locality_for_composite_patterns),
    ("DAX candidates preserve actionable ranking", test_dax_candidates_do_not_change_actionable_ranking),
    ("CSV formula-prefix sanitization", test_csv_formula_prefix_sanitization),
    ("Markdown and HTML escaping", test_markdown_and_html_escaping),
    ("deterministic serializers and JSON export", test_deterministic_serializers_and_json_export),
    ("DAX pattern exports and references", test_dax_pattern_exports_and_references),
    ("standalone HTML dependencies", test_standalone_html_has_no_external_assets),
]

passed = 0
for test_name, test_function in contract_tests:
    test_function()
    passed += 1
    print(f"PASS: {test_name}")
print(f"Total passed: {passed}")

In [ ]:
# Blueprint section 17 fixture matrix, golden checks, and privacy/provenance contract tests
FIXTURE_BASE = datetime(2026, 1, 1, tzinfo=timezone.utc)


def fixture_time(offset_ms: float) -> str:
    return (FIXTURE_BASE + timedelta(milliseconds=offset_ms)).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def fixture_event(event_id, name, component, start_ms, end_ms=None, parent=None, metrics=None):
    record = {"id": event_id, "name": name, "start": fixture_time(start_ms)}
    if component is not None:
        record["component"] = component
    if end_ms is not None:
        record["end"] = fixture_time(end_ms)
    if parent is not None:
        record["parentId"] = parent
    if metrics is not None:
        record["metrics"] = metrics
    return record


def fixture_visual(event_id, start_ms, end_ms, title="Visual", status="finished", visual_id=None, visual_type="tableEx"):
    return fixture_event(event_id, "Visual Container Lifecycle", REPORT_CANVAS, start_ms, end_ms,
                         metrics={"status": status, "visualTitle": title, "visualId": visual_id or f"id-{event_id}", "visualType": visual_type})


def fixture_model(*events, mode="compatible"):
    return build_event_model({"version": "1.0.0", "events": list(events)}, validation_mode=mode)


def fixture_rules(model):
    return set(model["run_summary"]["issue_counts_by_rule"])


def update_row(model, visual_event_id):
    return model["visual_updates"].set_index("visual_event_id").loc[visual_event_id]


def canvas_visual_with_query(prefix="v", start=0, end=1000):
    return [
        fixture_visual(prefix, start, end, title=f"Visual {prefix}"),
        fixture_event(f"{prefix}-q", "Query", REPORT_CANVAS, start + 10, start + 800, prefix),
        fixture_event(f"{prefix}-sq", "Execute Semantic Query", DSE_COMPONENT, start + 20, start + 780, f"{prefix}-q"),
        fixture_event(f"{prefix}-r", "Render", REPORT_CANVAS, start + 800, start + 980, prefix),
    ]


def test_f01_valid_minimal_file():
    events = [fixture_event("ua", "User Action", REPORT_CANVAS, 0, metrics={"sourceLabel": "UserAction_Refresh"}),
              fixture_visual("v", 10, 353, title="Count of ProductKey"),
              fixture_event("q", "Query", REPORT_CANVAS, 11, 335, "v"), fixture_event("r", "Render", REPORT_CANVAS, 335, 353, "v")]
    model = fixture_model(*events)
    assert fixture_rules(model) == set() and model["quality_status"] == "Pass"
    assert model["strict_schema_valid"] and fixture_model(*events, mode="strict")["quality_status"] == "Pass"
    row = update_row(model, "v")
    assert row["total_elapsed_ms"] == 343.0 and row["render_ms"] == 18.0 and row["canvas_query_ms"] == 324.0


def test_f02_empty_events():
    model = build_event_model({"version": "1.0.0", "events": []})
    assert model["run_summary"]["event_count"] == 0 and "EVENTS_EMPTY" in fixture_rules(model)
    assert "empty" in captured_value_error({"version": "1.0.0", "events": []}).lower()


def test_f03_instantaneous_user_action():
    model = fixture_model(fixture_event("ua", "User Action", REPORT_CANVAS, 0, metrics={"sourceLabel": "UserAction_ChangePage"}), *canvas_visual_with_query())
    event_row = model["events"].set_index("event_id").loc["ua"]
    assert bool(event_row["is_instantaneous"]) and pd.isna(event_row["duration_ms"])
    assert "EVENT_END_MISSING" not in fixture_rules(model)


def test_f04_completed_visual_with_query_and_render():
    row = update_row(fixture_model(*canvas_visual_with_query()), "v")
    assert row["has_canvas_query"] and row["render_ms"] == 180.0 and row["status"] == "finished"


def test_f05_visual_without_query():
    model = fixture_model(fixture_visual("v", 0, 100), fixture_event("r", "Render", REPORT_CANVAS, 10, 90, "v"))
    row = update_row(model, "v")
    assert not row["has_canvas_query"] and row["possible_canvas_cache_hit"] and "non-query" in row["backend_activity"]
    assert len(build_visual_frame(model)) == 1


def test_f06_possible_cache_hit():
    model = fixture_model(fixture_visual("v", 0, 100), fixture_event("q", "Query", REPORT_CANVAS, 5, 20, "v"), fixture_event("r", "Render", REPORT_CANVAS, 20, 90, "v"))
    row = update_row(model, "v")
    assert row["possible_canvas_cache_hit"] and row["backend_activity"] == "Not observed (possible canvas cache hit)"
    assert bool(build_visual_frame(model).iloc[0]["possible_canvas_cache_hit"])


def overlapping_query_fixture():
    return [
        fixture_visual("v", 0, 1000, title="Overlap"), fixture_event("q", "Query", REPORT_CANVAS, 5, 950, "v"),
        fixture_event("sq", "Execute Semantic Query", DSE_COMPONENT, 10, 940, "q"),
        fixture_event("dax-a", "Execute DAX Query", DSE_COMPONENT, 20, 720, "sq", {"QueryText": "EVALUATE 'Sales'", "RowCount": 10}),
        fixture_event("dax-b", "Execute DAX Query", DSE_COMPONENT, 220, 920, "sq", {"QueryText": "EVALUATE 'Geo'", "RowCount": 4}),
        fixture_event("as-a", "Execute Query", AS_COMPONENT, 30, 710, "dax-a"),
        fixture_event("dq-1", "Execute Direct Query", AS_COMPONENT, 100, 600, "as-a", {"QueryText": "SELECT 1", "ActualQueryDuration": 450, "RowsRead": 1, "DataReadDuration": 5}),
        fixture_event("dq-2", "Execute Direct Query", AS_COMPONENT, 400, 700, "as-a", {"QueryText": "SELECT 2", "ActualQueryDuration": 250, "RowsRead": 2, "DataReadDuration": 3}),
        fixture_event("r", "Render", REPORT_CANVAS, 950, 990, "v"),
    ]


def test_f07_semantic_query_with_multiple_dax_queries():
    model = fixture_model(*overlapping_query_fixture())
    row = update_row(model, "v")
    assert row["query_count"] == 2 and row["dax_covered_ms"] == 900.0 and row["dax_descendant_sum_ms"] == 1400.0
    frame_row = build_visual_frame(model).iloc[0]
    assert frame_row["dax_ms"] == 900.0 and frame_row["actionable_ms"] <= frame_row["total_ms"]
    assert len(model["dax_queries"]) == 2


def test_f08_overlapping_direct_queries():
    model = fixture_model(*overlapping_query_fixture())
    dax_row = model["dax_queries"].set_index("dax_event_id").loc["dax-a"]
    assert dax_row["direct_query_count"] == 2 and dax_row["direct_query_covered_ms"] == 600.0
    assert update_row(model, "v")["direct_query_covered_ms"] == 600.0
    frame_row = build_visual_frame(model).iloc[0]
    assert frame_row["native_query_text"] == "SELECT 1" and len(frame_row["direct_query_executions"]) == 2
    assert model["direct_queries"].set_index("direct_query_event_id").loc["dq-1", "actual_query_duration_ms"] == 450.0


def test_f09_abandoned_visual():
    model = fixture_model(fixture_visual("v1", 0, 300, status="abandoned", visual_id="same"), fixture_visual("v2", 310, 900, visual_id="same"))
    assert "VISUAL_ABANDONED" in fixture_rules(model)
    row = update_row(model, "v1")
    assert row["status"] == "abandoned" and row["next_update_event_id"] == "v2" and "Abandoned" in row["flags"]


def test_f10_dax_error_and_cancellation():
    model = fixture_model(fixture_visual("v", 0, 500), fixture_event("q", "Query", REPORT_CANVAS, 1, 400, "v"),
                          fixture_event("sq", "Execute Semantic Query", DSE_COMPONENT, 2, 390, "q"),
                          fixture_event("d1", "Execute DAX Query", DSE_COMPONENT, 3, 100, "sq", {"QueryText": "EVALUATE A", "Error": True}),
                          fixture_event("d2", "Execute DAX Query", DSE_COMPONENT, 110, 200, "sq", {"QueryText": "EVALUATE B", "Canceled": "true"}))
    assert {"DAX_ERROR", "DAX_CANCELED"} <= fixture_rules(model) and model["quality_status"] == "Fail"
    frame_row = build_visual_frame(model).iloc[0]
    assert frame_row["dax_error_count"] == 1 and frame_row["dax_canceled_count"] == 1


def test_f11_dse_and_as_truncation():
    model = fixture_model(*overlapping_query_fixture(),
                          fixture_event("t-dse", "Metrics Truncated", DSE_COMPONENT, 500, None, "sq"),
                          fixture_event("t-as", "Metrics Truncated", AS_COMPONENT, 510, None, "as-a", {"Count": 42}))
    assert model["run_summary"]["truncation_count"] == 2 and model["run_summary"]["truncated_omitted_events"] == 42
    assert update_row(model, "v")["is_truncated"]
    analyzed, _ = analyze_visuals(build_visual_frame(model))
    assert (analyzed["confidence"] == "Low").all() and analyzed["recommendation"].str.contains("truncated").all()


def test_f12_duplicate_id():
    model = fixture_model(fixture_visual("v", 0, 1000), fixture_event("r", "Render", REPORT_CANVAS, 100, 200, "v"),
                          fixture_event("r", "Render", REPORT_CANVAS, 500, 900, "v"))
    assert "EVENT_ID_DUPLICATE" in fixture_rules(model) and len(model["events"]) == 3
    assert update_row(model, "v")["render_ms"] == 100.0
    assert int(model["events"]["is_quarantined"].sum()) == 1


def test_f13_missing_parent():
    model = fixture_model(*canvas_visual_with_query(), fixture_event("orphan", "Render", REPORT_CANVAS, 50, 60, "does-not-exist"))
    assert "PARENT_NOT_FOUND" in fixture_rules(model)
    assert bool(model["events"].set_index("event_id").loc["orphan", "is_unattributed"])
    assert build_visual_frame(model)["visual_event_id"].tolist() == ["v"]


def test_f14_parent_cycle():
    model = fixture_model(*canvas_visual_with_query(), fixture_event("a", "Render", REPORT_CANVAS, 1, 2, "b"),
                          fixture_event("b", "Render", REPORT_CANVAS, 1, 2, "a"), fixture_event("c", "Render", REPORT_CANVAS, 1, 2, "a"))
    assert "PARENT_CYCLE" in fixture_rules(model)
    quarantined = model["events"].set_index("event_id")["quarantine_reason"]
    assert quarantined["a"] == "parent cycle" and quarantined["c"] == "descendant of parent cycle"


def test_f15_child_outside_parent():
    model = fixture_model(fixture_visual("v", 100, 500), fixture_event("r", "Render", REPORT_CANVAS, 50, 520, "v"))
    assert "CHILD_OUTSIDE_PARENT" in fixture_rules(model)
    row = update_row(model, "v")
    assert row["render_ms"] == 400.0 and row["clipped_child_count"] == 1 and row["data_quality_status"] == "Warning"
    assert model["events"].set_index("event_id").loc["r", "start_utc"] == "2026-01-01T00:00:00.050000Z"


def test_f16_negative_duration():
    model = fixture_model(fixture_visual("v", 0, 500), fixture_event("r", "Render", REPORT_CANVAS, 300, 200, "v"))
    assert "NEGATIVE_DURATION" in fixture_rules(model)
    assert model["events"].set_index("event_id").loc["r", "duration_ms"] == -100.0
    row = update_row(model, "v")
    assert pd.isna(row["render_ms"]) and "Child negative duration" in row["flags"] and row["data_quality_status"] == "Incomplete"


def test_f17_unknown_event_type_and_metric():
    model = fixture_model(fixture_visual("v", 0, 500), fixture_event("x", "Brand New Event", REPORT_CANVAS, 10, 20, "v", {"shiny": 1}),
                          fixture_event("r", "Render", REPORT_CANVAS, 20, 40, "v", {"futureMetric": "kept"}))
    assert {"UNKNOWN_EVENT_TYPE", "UNKNOWN_METRIC"} <= fixture_rules(model)
    events = model["events"].set_index("event_id")
    assert json.loads(events.loc["r", "metrics_json"]) == {"futureMetric": "kept"}
    assert json.loads(events.loc["x", "raw_event_json"])["metrics"] == {"shiny": 1}


def test_f18_missing_query_text():
    model = fixture_model(fixture_visual("v", 0, 500), fixture_event("q", "Query", REPORT_CANVAS, 1, 400, "v"),
                          fixture_event("sq", "Execute Semantic Query", DSE_COMPONENT, 2, 390, "q"),
                          fixture_event("d", "Execute DAX Query", DSE_COMPONENT, 3, 300, "sq", {"RowCount": 1}),
                          fixture_event("dq", "Execute Direct Query", AS_COMPONENT, 4, 200, "d", {"RowsRead": 1}))
    assert model["run_summary"]["issue_counts_by_rule"]["QUERY_TEXT_UNAVAILABLE"] == 2
    frame_row = build_visual_frame(model).iloc[0]
    assert frame_row["direct_query_executions"][0]["capture_status"] == "DirectQuery text was not captured"


def test_f19_cross_run_repeated_fingerprint():
    first = fixture_model(*overlapping_query_fixture())
    second_events = overlapping_query_fixture()
    second_events[3]["metrics"]["QueryText"] = "  evaluate   'SALES' "
    second = fixture_model(*second_events)
    fingerprint = lambda model: model["dax_queries"].set_index("dax_event_id").loc["dax-a", "query_fingerprint"]
    assert fingerprint(first) == fingerprint(second) != ""


def test_f20_malicious_strings_are_escaped():
    payload_title = "<script>alert('x')</script>"
    model = fixture_model(fixture_visual("v", 0, 500, title=payload_title),
                          fixture_event("r", "Render", REPORT_CANVAS, 10, 40, "v", {"note": "<img src=x onerror=alert(1)>"}))
    rendered = render_visuals_table_html(model["visual_updates"]) + render_event_tree_html(model, "v") + render_timeline_svg(model)
    assert "<script>" not in rendered and "<img" not in rendered and "&lt;script&gt;" in rendered


def test_missing_end_is_unknown_not_zero():
    model = fixture_model(fixture_event("v", "Visual Container Lifecycle", REPORT_CANVAS, 0, None, metrics={"status": "started", "visualTitle": "Open"}),
                          fixture_event("r", "Render", REPORT_CANVAS, 10, 60, "v"))
    row = update_row(model, "v")
    assert pd.isna(row["total_elapsed_ms"]) and pd.isna(row["derived_other_ms"]) and "EVENT_END_MISSING" in fixture_rules(model)
    assert "Lifecycle duration unknown" in row["flags"] and "Status started" in row["flags"]
    assert build_visual_frame(model).iloc[0]["total_ms_source"].startswith("derived from components")


def test_component_inference_for_110_exports():
    events = canvas_visual_with_query()
    for event in events:
        if event["id"] in {"v", "v-q", "v-r"}:
            event.pop("component")
    payload = {"version": "1.1.0", "sessionId": "abc", "events": events}
    model = build_event_model(payload)
    inferred = model["events"].set_index("event_id")
    assert inferred.loc["v-q", "component"] == REPORT_CANVAS and bool(inferred.loc["v-q", "component_inferred"])
    assert model["run_summary"]["issue_counts_by_rule"]["EVENT_REQUIRED_FIELD_MISSING"] == 3 and not model["strict_schema_valid"]
    assert update_row(model, "v")["render_ms"] == 180.0
    try:
        build_event_model(payload, validation_mode="strict")
    except StrictSchemaError as exc:
        assert "1.1.0" in str(exc) and "sessionId" in str(exc)
    else:
        raise AssertionError("strict mode must reject a 1.1.0 export")


def test_interaction_attribution_is_temporal_and_bounded():
    events = [fixture_event("ua1", "User Action", REPORT_CANVAS, 0, metrics={"sourceLabel": "UserAction_ChangeSlicer"}),
              fixture_visual("v1", 100, 200), fixture_event("ua2", "User Action", REPORT_CANVAS, 1000, metrics={"sourceLabel": "UserAction_ChangePage"}),
              fixture_visual("v2", 1500, 1600), fixture_visual("v3", 1000 + INTERACTION_WINDOW_MS + 1, 1000 + INTERACTION_WINDOW_MS + 50)]
    updates = fixture_model(*events)["visual_updates"].set_index("visual_event_id")
    assert updates.loc["v1", "derived_interaction_id"] == "interaction-001" and updates.loc["v2", "interaction_label"] == "UserAction_ChangePage"
    assert updates.loc["v3", "derived_interaction_id"] == ""
    tie = fixture_model(fixture_event("a", "User Action", REPORT_CANVAS, 0), fixture_event("b", "User Action", REPORT_CANVAS, 0), fixture_visual("v", 10, 20))
    assert update_row(tie, "v")["derived_interaction_id"] == ""


def test_covered_ms_union_and_clip():
    at = lambda ms: FIXTURE_BASE + timedelta(milliseconds=ms)
    assert covered_ms([(at(0), at(700)), (at(200), at(900))]) == (900.0, 0)
    assert covered_ms([(at(0), at(100)), (at(200), at(300))]) == (200.0, 0)
    assert covered_ms([(at(-50), at(100)), (at(90), at(80))], clip=(at(0), at(1000))) == (100.0, 1)


def test_status_code_map_is_opt_in():
    global VISUAL_STATUS_CODE_MAP
    original = VISUAL_STATUS_CODE_MAP
    try:
        VISUAL_STATUS_CODE_MAP = {}
        default_model = fixture_model(fixture_visual("v", 0, 100, status=3))
        assert "VISUAL_STATUS_NONSTANDARD" in fixture_rules(default_model) and update_row(default_model, "v")["status"] == "undocumented (3)"
        VISUAL_STATUS_CODE_MAP = {"3": "abandoned"}
        assert "VISUAL_ABANDONED" in fixture_rules(fixture_model(fixture_visual("v", 0, 100, status=3)))
    finally:
        VISUAL_STATUS_CODE_MAP = original


def test_query_text_redaction_and_omit():
    dax = "EVALUATE CALCULATETABLE('Sales', 'Sales'[Region] = \"West\", 'Date'[Year] = 2026)"
    redacted = apply_query_text_policy(dax, "dax", "redact")
    assert "'Sales'[Region]" in redacted and "West" not in redacted and "2026" not in redacted
    sql = "SELECT [Region] FROM [dbo].[Orders] WHERE [Customer] = N'Contoso' AND [Year] = 2026"
    redacted_sql = apply_query_text_policy(sql, "sql", "redact")
    assert "[dbo].[Orders]" in redacted_sql and "Contoso" not in redacted_sql and "2026" not in redacted_sql
    assert "omitted" in apply_query_text_policy(dax, "dax", "omit") and apply_query_text_policy(dax, "dax", "full") == dax
    nested = apply_query_policy_to_json({"metrics": {"QueryText": dax, "RowCount": 5}}, "dax", "omit")
    assert "West" not in json.dumps(nested) and nested["metrics"]["RowCount"] == 5
    odd = apply_query_policy_to_json({"QueryText": ["SELECT 'secret'"], "queryText": {"x": "secret"}}, "sql", "omit")
    assert "secret" not in json.dumps(odd)
    exotic = apply_query_text_policy("SELECT 1e9, 0xDEADBEEF, $$secret$$, $tag$hidden$tag$, E'esc' FROM [t]", "sql", "redact")
    assert not any(token in exotic for token in ("1e9", "DEADBEEF", "secret", "hidden", "esc")) and "[t]" in exotic
    assert query_fingerprint(dax) == query_signature(dax)


def test_direct_lake_storage_mode_guidance():
    global capture_metadata
    original = capture_metadata
    dq_heavy = pd.Series({"actionable_ms": 3000.0, "direct_query_ms": 2800.0, "dax_ms": 3000.0, "render_ms": 50.0, "query_ms": 3000.0,
                          "parameter_ms": 0.0, "other_ms": 0.0, "total_ms": 3100.0, "is_p90_outlier": False})
    try:
        capture_metadata = {**original, "storage_mode": "Import"}
        assert choose_rule(dq_heavy) == "DQ" and not storage_mode_profile()["is_direct_lake"]
        capture_metadata = {**original, "storage_mode": "Direct Lake on SQL", "cache_state": "cold"}
        assert choose_rule(dq_heavy) == "DL_FALLBACK" and storage_mode_profile()["direct_lake_variant"] == "sql"
        frame = build_visual_frame(fixture_model(*overlapping_query_fixture()))
        guided = apply_storage_mode_guidance(analyze_visuals(frame)[0])
        assert guided["direct_lake_fallback_ms"].iloc[0] == guided["direct_query_ms"].iloc[0] > 0
        assert "fallback" in guided["recommendation"].iloc[0].lower()
        dax_frame = build_visual_frame(fixture_model(*[event for event in overlapping_query_fixture() if not event["id"].startswith("dq")]))
        dax_guided = apply_storage_mode_guidance(analyze_visuals(dax_frame.assign(dax_ms=1500.0, query_ms=1500.0, actionable_ms=1540.0))[0])
        assert "transcodes" in dax_guided["next_diagnostic"].iloc[0]
        capture_metadata = {**original, "storage_mode": "Direct Lake on SQL", "cache_state": "warm"}
        assert "transcodes" not in apply_storage_mode_guidance(analyze_visuals(dax_frame.assign(dax_ms=1500.0, query_ms=1500.0, actionable_ms=1540.0))[0])["next_diagnostic"].iloc[0]
        capture_metadata = {**original, "storage_mode": "Direct Lake on OneLake"}
        onelake = apply_storage_mode_guidance(analyze_visuals(frame.assign(direct_query_ms=2800.0, dax_ms=2900.0, query_ms=2900.0, actionable_ms=2940.0))[0])
        assert onelake["rule_id"].iloc[0] == "DL_FALLBACK" and "does not fall back" in onelake["recommendation"].iloc[0]
    finally:
        capture_metadata = original


def test_name_only_matching_ignores_documented_components():
    model = fixture_model(fixture_event("x", "Visual Container Lifecycle", DSE_COMPONENT, 0, 100), fixture_visual("v", 0, 100))
    assert model["visual_updates"]["visual_event_id"].tolist() == ["v"]


def test_golden_overlap_fixture():
    model = fixture_model(*overlapping_query_fixture())
    row = update_row(model, "v")
    golden = {"total_elapsed_ms": 1000.0, "canvas_query_ms": 945.0, "dax_covered_ms": 900.0, "direct_query_covered_ms": 600.0,
              "render_ms": 40.0, "derived_other_ms": 15.0, "query_count": 2, "direct_query_count": 2, "data_quality_status": "Good"}
    assert {key: row[key] for key in golden} == golden
    assert model["events"][["event_id", "depth", "visual_event_id", "dax_query_event_id"]].to_dict("records")[6] == {
        "event_id": "dq-1", "depth": 5, "visual_event_id": "v", "dax_query_event_id": "dax-a"}
    assert fixture_rules(model) == set()


def test_report_provenance_csp_and_blueprint_outputs():
    assert "Content-Security-Policy" in html_report and "unsafe-inline" not in html_report
    assert priority_export.empty or ('id="priority-table"' in html_report and html_report.count('class="priority-card') == len(priority_export))
    rendered_priorities = render_priorities_html(pd.DataFrame([{**{column: "" for column in priority_export.columns}, "Rank": 1, "Severity": "Slow", "Visual": "<b>x</b>"}]))
    assert "<b>x</b>" not in rendered_priorities and "&lt;b&gt;x&lt;/b&gt;" in rendered_priorities
    assert csp_source_hash(html_script) in html_report and csp_source_hash(html_style) in html_report
    assert run_summary["source_sha256"] in html_report and PARSER_VERSION in html_report and run_summary["source_sha256"] in markdown_report
    for path in (events_csv_path, visual_updates_csv_path, dax_events_csv_path, direct_events_csv_path, quality_issues_json_path,
                 findings_json_path, run_summary_json_path, manifest_json_path, index_html_path):
        assert path.exists(), path
    assert len(pd.read_csv(events_csv_path)) == run_summary["event_count"]
    manifest_names = {artifact["name"] for artifact in json.loads(manifest_json_path.read_text(encoding="utf-8"))["artifacts"]}
    assert {"analysis_report.html", "events.csv", "quality-issues.json"} <= manifest_names
    exported = json.loads(json_path.read_text(encoding="utf-8"))
    assert exported["provenance"]["parser_version"] == PARSER_VERSION and exported["run_summary"]["run_id"] == run_summary["run_id"]


blueprint_tests = [
    ("F01 valid minimal file", test_f01_valid_minimal_file), ("F02 empty events array", test_f02_empty_events),
    ("F03 instantaneous User Action", test_f03_instantaneous_user_action), ("F04 visual with Query and Render", test_f04_completed_visual_with_query_and_render),
    ("F05 visual with no query", test_f05_visual_without_query), ("F06 possible cache hit", test_f06_possible_cache_hit),
    ("F07 multiple DAX queries", test_f07_semantic_query_with_multiple_dax_queries), ("F08 overlapping DirectQuery", test_f08_overlapping_direct_queries),
    ("F09 abandoned visual", test_f09_abandoned_visual), ("F10 DAX error and cancel", test_f10_dax_error_and_cancellation),
    ("F11 DSE and AS truncation", test_f11_dse_and_as_truncation), ("F12 duplicate id", test_f12_duplicate_id),
    ("F13 missing parent", test_f13_missing_parent), ("F14 parent cycle", test_f14_parent_cycle),
    ("F15 child outside parent", test_f15_child_outside_parent), ("F16 negative duration", test_f16_negative_duration),
    ("F17 unknown event type and metric", test_f17_unknown_event_type_and_metric), ("F18 missing query text", test_f18_missing_query_text),
    ("F19 cross-run repeated fingerprint", test_f19_cross_run_repeated_fingerprint), ("F20 malicious strings", test_f20_malicious_strings_are_escaped),
    ("missing end is unknown, not zero", test_missing_end_is_unknown_not_zero), ("1.1.0 component inference", test_component_inference_for_110_exports),
    ("temporal interaction attribution", test_interaction_attribution_is_temporal_and_bounded), ("interval union", test_covered_ms_union_and_clip),
    ("status code map opt-in", test_status_code_map_is_opt_in), ("query-text redaction and omit", test_query_text_redaction_and_omit),
    ("name-only matching limited to undocumented components", test_name_only_matching_ignores_documented_components),
    ("Direct Lake storage-mode guidance", test_direct_lake_storage_mode_guidance),
    ("golden overlap fixture", test_golden_overlap_fixture), ("provenance, CSP, and blueprint outputs", test_report_provenance_csp_and_blueprint_outputs),
]
blueprint_passed = 0
for test_name, test_function in blueprint_tests:
    test_function()
    blueprint_passed += 1
    print(f"PASS: {test_name}")
print(f"Blueprint fixture tests passed: {blueprint_passed}/{len(blueprint_tests)}")

In [ ]:
# Real-version regression matrix (blueprint 17.3): run every capture in REGRESSION_CAPTURES_DIR through the parser invariants.
def regression_invariants(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_bytes().decode("utf-8-sig"))
    frame, metadata, model = parse_performance_analyzer_with_model(payload, validation_mode="compatible",
                                                                   source_info={"source_file_name": path.name, "source_sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    updates = model["visual_updates"]
    tolerance = float(CLOCK_TOLERANCE_MS)
    total = pd.to_numeric(updates["total_elapsed_ms"], errors="coerce")

    def within_lifecycle(column: str) -> bool:
        values = pd.to_numeric(updates[column], errors="coerce")
        return bool((values.isna() | total.isna() | (values <= total + tolerance)).all())

    checks = {
        "every event retained": len(model["events"]) == (len(payload["events"]) if isinstance(payload, dict) and isinstance(payload.get("events"), list) else len(model["events"])),
        "DAX covered <= lifecycle": within_lifecycle("dax_covered_ms"),
        "DirectQuery covered <= lifecycle": within_lifecycle("direct_query_covered_ms"),
        "actionable <= total": bool((frame["actionable_ms"] <= frame["total_ms"] + tolerance).all()),
        "every lifecycle is a visual row": set(updates["visual_event_id"]) <= set(frame["visual_event_id"]),
    }
    return {"capture": path.name, "format": metadata["format"], "version": metadata["version"], "events": len(model["events"]),
            "visual_updates": len(updates), "quality": model["quality_status"], **checks, "all_passed": all(checks.values())}


if str(REGRESSION_CAPTURES_DIR).strip():
    regression_directory = Path(REGRESSION_CAPTURES_DIR).expanduser()
    regression_matrix = pd.DataFrame([regression_invariants(path) for path in sorted(regression_directory.glob("*.json"))])
    display(regression_matrix)
    assert not regression_matrix.empty, f"No JSON captures found in {regression_directory}"
    assert regression_matrix["all_passed"].all(), "One or more real captures violated a parser invariant."
    print(f"PASS: regression matrix over {len(regression_matrix)} capture(s)")
else:
    current_capture = regression_invariants(source_path) if source_path is not None else None
    if current_capture is not None:
        assert current_capture["all_passed"], current_capture
        print(f"PASS: current capture invariants ({current_capture['version']}, {current_capture['events']:,} events)")
    print("Set REGRESSION_CAPTURES_DIR to a folder of controlled captures (import, DirectQuery, composite, remote model, cold/warm, "
          "page vs visual refresh, change detection, geocoding, field parameters) to run the full real-version regression matrix.")

In [ ]:
synthetic_card_diagnostics = pd.DataFrame({
    "page": ["Page A", "Page A", "Page A", "Page B"],
    "visual_type": ["card", " CARD ", "cardVisual", "card"],
})
synthetic_card_pages = pd.DataFrame({"page": ["Page A", "Page B"]})
annotated_cards, annotated_card_pages = annotate_card_usage(
    synthetic_card_diagnostics,
    synthetic_card_pages,
)

assert annotated_cards["is_single_value_card"].tolist() == [True, True, False, True]
assert annotated_card_pages["single_value_card_count"].tolist() == [2, 1]
assert annotated_card_pages["multiple_single_value_cards"].tolist() == [True, False]
print("PASS: legacy Card classification excludes the new multi-value Card")

In [ ]:
boundary_pages = (
    ["Named page"] * 7
    + ["Page transition 99 (name unavailable)"] * 8
    + ["Unknown page"] * 8
)
boundary_results = pd.DataFrame({
    "page": boundary_pages,
    "visual_id": [f"boundary-{index}" for index in range(len(boundary_pages))],
    "recommendation": [""] * len(boundary_pages),
})
boundary_summary = pd.DataFrame({"page": list(dict.fromkeys(boundary_pages))})
_, guarded_boundary_summary = apply_page_identity_guard(boundary_results, boundary_summary)
boundary_flags = guarded_boundary_summary.set_index("page")["over_visual_limit"].to_dict()

assert boundary_flags["Named page"] is False or not bool(boundary_flags["Named page"])
assert bool(boundary_flags["Page transition 99 (name unavailable)"])
assert not bool(boundary_flags["Unknown page"])

other_time_summary = annotate_page_other_time(pd.DataFrame({
    "total_actionable_ms": [600.0],
    "total_other_ms": [400.0],
}))
assert float(other_time_summary.loc[0, "other_share_pct"]) == 40.0
print("PASS: page limit is greater than 7 and Other-time percentage is calculated")

## 11. How to interpret the diagnosis

- `actionable_ms = max(DAX, DirectQuery) + render + parameters`; use it to rank the components most likely to benefit from investigation. For event traces it is computed as the interval union of those events, which equals the formula when they do not overlap.
- Category times are **covered time**: the interval union of that category's events inside the visual lifecycle. Overlapping DAX or DirectQuery events are never summed; `dax_descendant_sum_ms` is only a work-volume indicator.
- **not observed** means no event of that type was captured (for example, a possible canvas cache hit); **unknown** means a duration could not be computed. Neither is zero.
- `derived_interaction_id` is a temporal heuristic (nearest preceding User Action), not a `parentId` relationship.
- **Other** time is retained as evidence but excluded from ranking because it is not directly attributable to a single actionable component.
- **Confidence** reflects the strength and specificity of the available capture evidence, not proof of root cause.
- Component sums are not wall-clock duration because work can overlap or wait in queues.
- DAX heuristics and candidate IDs are conservative text-shape routes to investigate; they are not confirmed diagnoses or proven root causes.
- **Tier 1 - measure/UDF** candidates are emitted only when a `DEFINE MEASURE` body is visible in the capture. Referenced measures, UDFs, or other dependencies may still need definition resolution before review.
- **Tier 2 - query/report structure** candidates require report-author approval because a rewrite can alter grouping, filtering, BLANK/zero behavior, or result shape.

## 12. Remediation and retest workflow

1. Resolve the visible measure/UDF definitions and referenced dependencies; record one specific interaction, including its page, filters, slicers, and device context.
2. Capture a controlled cold- and warm-cache baseline. Use Performance Analyzer for report interaction timing and DAX Query View or DAX Studio for Server Timings and Query Plan evidence.
3. Review candidates as investigation routes, and obtain report-author approval before testing any Tier 2 query/report structure change.
4. Change one variable at a time under the same interaction, filter context, device, and capacity conditions.
5. Compare FE and SE duration, SE query count, callbacks, materialized rows, peak memory, and result rows against the baseline.
6. Validate semantic equivalence, including values, filters, grouping, BLANK/zero behavior, and result shape. Keep a rewrite only when improvement exceeds run noise and values/results remain equivalent.
7. Recapture Performance Analyzer, retain the before-and-after traces and artifacts, and, when relevant, validate in the Power BI Service on the target Fabric capacity.

## 13. Limitations and escalation

- One capture cannot prove whether the root cause is the model, storage mode, capacity, network, or browser.
- Cache state, filter context, device, and capacity conditions can materially affect results.
- Performance Analyzer does not expose enough engine evidence to confirm a DAX pattern; use it to locate interactions that need deeper investigation.
- A trace with few clean SE scans and low parallelism can point toward storage or layout work rather than a DAX rewrite, but still requires controlled validation.
- Performance Analyzer event names and export schemas may evolve; custom visual labels are heuristic.
- Power BI 1.1.0 exports omit `component` on many canvas events, add `sessionId`, and use undocumented numeric visual status codes (for example `1`, `3`). The notebook infers the component, reports the drift, and does not interpret status codes unless `VISUAL_STATUS_CODE_MAP` is set.
- AS events are absent for models hosted in SQL Server Analysis Services, Power BI, or Azure Analysis Services; missing AS detail is not proof of zero engine work. DirectQuery source text appears only when the user owns the model and the source is SQL.
- A `Metrics Truncated` event means event detail is incomplete; lifecycle totals remain valid, but component splits for affected visuals are qualified and marked Low confidence.
- Parity with the Performance Analyzer pane is not claimed; the derived categories are documented in the report's methodology section. Real-version parity needs a regression matrix of controlled captures (`REGRESSION_CAPTURES_DIR`).
- Cache comparison needs multiple intentionally labeled runs; this notebook analyzes one run at a time.
- The export does not record storage mode. For **Direct Lake** models, set `CAPTURE_METADATA["storage_mode"]` to `"Direct Lake on SQL"` or `"Direct Lake on OneLake"`: DirectQuery events are then treated as Direct Lake fallback (or, for OneLake, as a sign that tables are not Direct Lake), and DAX-led visuals get a cold-cache (column loading) note unless `cache_state` is `"warm"`. Without it, DirectQuery events receive source-tuning guidance that does not apply to Direct Lake fallback.
- Performance Analyzer has no built-in baseline, so retain comparable captures and artifacts.
- Escalate with DAX Studio Server Timings and query plans, browser developer tools, Fabric Capacity Metrics, and Log Analytics or other monitoring evidence as appropriate.

## 14. References and evidence policy

**Primary Microsoft guidance**

- [Microsoft Performance Analyzer](https://learn.microsoft.com/power-bi/create-reports/performance-analyzer)
- [Power BI optimization guidance](https://learn.microsoft.com/power-bi/guidance/power-bi-optimization)
- [Monitor report performance](https://learn.microsoft.com/power-bi/guidance/monitor-report-performance)
- [Troubleshoot report performance](https://learn.microsoft.com/power-bi/guidance/report-performance-troubleshoot)

**Configured DAX sources (`SOURCE_URLS`)**

- [`dax_decision_guide`: Microsoft DAX performance decision guide](https://github.com/microsoft/skills-for-fabric/blob/main/skills/semantic-model-authoring/references/dax-perf-decision-guide.md)
- [`dax_pattern_catalog`: Microsoft DAX performance pattern catalog](https://github.com/microsoft/skills-for-fabric/blob/main/skills/semantic-model-authoring/references/dax-perf-patterns.md)

**Secondary implementation context**

- [Fabric Community best-practice rules post](https://community.fabric.microsoft.com/blog/fbc_pbiupdatesblog/best-practice-rules-to-improve-your-models-performance/5175648)
- [SQLBI Performance Analyzer article](https://www.sqlbi.com/articles/introducing-the-power-bi-performance-analyzer/)
- [GitHub: m-kovalsky/ReportAnalyzer](https://github.com/m-kovalsky/ReportAnalyzer)

**Anecdotal only**

- [Reddit discussion: Performance Analyzer in Power BI Service](https://www.reddit.com/r/PowerBI/comments/1jwqep2/performance_analyzer_in_power_bi_service/)

**Direct Lake**

- [Analyze query processing for Direct Lake semantic models](https://learn.microsoft.com/fabric/fundamentals/direct-lake-analyze-query-processing)
- [How Direct Lake works](https://learn.microsoft.com/fabric/fundamentals/direct-lake-how-it-works)

No recommendation was derived from the Reddit discussion. Configured thresholds are triage defaults, not Microsoft SLAs.

## 15. Completion checklist

- [ ] Set `INPUT_PATH` to the intended Performance Analyzer export.
- [ ] Review the configured thresholds for the report and environment.
- [ ] Choose `VALIDATION_MODE` and `QUERY_TEXT_MODE` (use `redact` or `omit` before sharing outputs) and record `CAPTURE_METADATA`.
- [ ] Run All and check the sample-versus-real source banner before interpreting results.
- [ ] Read the data-quality banner; resolve Fail items (quarantined records, DAX errors) and note truncation before trusting component splits.
- [ ] Start with rank 1 and resolve its visible definitions and dependencies.
- [ ] Capture controlled cold/warm baselines, Server Timings, and Query Plan evidence.
- [ ] Review candidate routes; obtain report-author approval for Tier 2 changes.
- [ ] Apply one change under the same interaction and filter context.
- [ ] Validate engine metrics, result rows, and semantic equivalence; reject changes within run noise or with changed values/results.
- [ ] Recapture Performance Analyzer and archive the before-and-after artifacts and traces.